# KASIRNet — revision notebook (Information Sciences, INS-D-26-12219)

One notebook = full training → full inference → every analysis requested by the reviewers → all ablations → manuscript tables.
Each code cell below is **identical** to the module of the same name in the repository (`tools/build_notebook.py` generates this notebook from them).

**Hardware.** Colab *G4* runtime (NVIDIA RTX PRO 6000 Blackwell Server Edition, 96 GB). The exact GPU, driver, CUDA/cuDNN/PyTorch/pyiqa versions are written to every `run.json`.

**How to run.** Runtime → G4 → *Run all*. All settings are in `Config` (no argparse). Stages are idempotent and every training run checkpoints atomically every 2,000 iterations to Google Drive, so when a Colab session ends simply *Run all* again: the current run resumes from its last checkpoint (bit-exact data order and RNG state) and finished work is skipped.

| stage | what it does | main outputs (under `OUTPUT_ROOT`) |
|---|---|---|
| `audit` | restores the datasets from the Google-Drive cache (or downloads them once and caches them), builds the frozen training manifest with the train/test leakage audit, validates every category and benchmark (pairing, alignment, direction, sizes), writes the Step-0 facts (metric definition, SSIM-loss gradient, SRB gate initialisation, submitted checkpoint, data pairing) | `datasets_cache/*.tar`, `data_manifest/{train_manifest,data_validation}.json`, `analysis/step0_facts/` |
| `train_full` | reference KASIRNet (batch 64, Adam 3e-4 with 1,500-iteration warm-up, cosine, clip 1.0, FP16, six loss terms), `EPOCHS` = 150; also the "full, seed 1" reference of every ablation table | `checkpoints/KASIRNet_full/final.pth`, `runs/KASIRNet_full/{run.json, curves.png, history_*.csv}` |
| `train_main` | final model of Tables 4–5: `KASIRNet_full` + `MAIN_EPOCHS` = 150 (warm restart), 300 epochs in total | `checkpoints/KASIRNet_main/final.pth`, `runs/KASIRNet_main/` |
| `eval_full` | inference protocol chosen once on the held-out validation pairs (`full` / train-consistent `tiled256` / `tiled384`), then 8 benchmarks + Rain100H + Snow100K-S/M + Gaussian σ15/25/50 + LIVE1/LOL/DPDD (Köhler registered to its reference) | `analysis/inference_protocol/`, `runs/KASIRNet_main/eval/<bench>/per_image.csv` (PSNR, SSIM, LPIPS, DISTS), `summary.json` |
| `analyses_full` | cost (TODO-13), bootstrap CI + Wilcoxon (TODO-12C), LPIPS/DISTS + divergent % (TODO-14), prompt probe + mixed sets (TODO-16), spectral/phase analysis, feature stability, Figs. S1/S2 (TODO-06), explainability | `tables/`, `analysis/` |
| `ablations` | 45 runs with the same 150 epochs and recipe as `KASIRNet_full` (which is their seed-1 full reference, trained once): TODO-12A (5 variants × 3 seeds), TODO-09B (extractors), TODO-12B (interactions), TODO-15 (loss weights ×0.5/×2), Table 6 rows, Table 7 rows, SCWAM design | `runs/<run_id>/run.json` + GoPro/Rain100L/SIDD per-image CSVs |
| `analyses_ablation` | SIM vs severity (TODO-09A) and all ablation tables | `tables/table6…`, `table7…`, `table8…`, `table9…`, `table12…` |
| `tables` | Tables 4/5, Rain100H (TODO-02), zero-shot (TODO-17), Section 4.1 reproducibility summary | `tables/*.csv`, `tables/*.tex` |

**Parallel sessions.** If more than one G4 session is available, open this notebook in each of them (same Google account / Drive) and *Run all*: a run being trained holds a lock with a heartbeat, so the sessions share the ablation queue without training the same run twice (`Config.LOCK_STALE_MINUTES`). The measured s/iteration and the GPU-hours left are printed at the start of the `ablations` stage.

Competitor outputs that the authors hold (HIDE Restormer, TextOCR CNN-ViT, Köhler Rahman et al., RESIDE-6K ADSM, SIDD Defusion; Rain100L / Snow100K-L methods for Figs. S1/S2) are read from `Config.COMPETITOR_ROOT` and scored with the same metric function.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ---- environment (Colab G4).  pyiqa is pinned to the version that produced the submitted figures.
import importlib, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyiqa==0.1.16', '--no-deps'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lmdb', 'gdown', 'kagglehub'], check=False)
PIP_NAME = {'cv2': 'opencv-python-headless', 'yaml': 'pyyaml', 'skimage': 'scikit-image', 'sklearn': 'scikit-learn',
            'clip': 'openai-clip', 'PIL': 'pillow'}
for _ in range(10):  # install any import-time dependency of pyiqa missing from the runtime
    try:
        import pyiqa
        break
    except ModuleNotFoundError as e:
        mod = e.name.split('.')[0]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', PIP_NAME.get(mod, mod)], check=False)
import torch
print('torch', torch.__version__, '| pyiqa', pyiqa.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
# ===== Config.py (identical to Config.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - configuration of the revision (INS-D-26-12219)
# --------------------------------------------------------------------------
# Every setting of every run lives in this single Config class (no argparse,
# no sys.argv).  A run is "Config + one line of overrides" (see ABLATION_GROUPS).
# Section 4.1 of the manuscript is generated from the values below and from the
# run.json that each training run writes (GPU, iterations, wall-clock, seeds).
# ==========================================================================
import os
import copy
import json


class Config:
    # ------------------------------------------------------------------ stages
    # main() executes these stages in order; each one is idempotent (a finished run
    # or analysis is detected from its JSON and skipped), so the notebook can be
    # re-run in as many Colab sessions as needed.
    STAGES = ['audit', 'train_full', 'train_main', 'eval_full', 'analyses_full', 'ablations', 'analyses_ablation', 'tables']

    # ---------------------------------------------------------------- hardware
    # Colab "G4" runtime = NVIDIA RTX PRO 6000 Blackwell Server Edition (96 GB).
    # The actual device name, driver, CUDA/cuDNN/PyTorch versions are recorded in run.json.
    DEVICE = 'cuda'
    NUM_WORKERS = min(16, os.cpu_count() or 4)
    PIN_MEMORY = True
    AMP = True                    # mixed precision (paper: FP16)
    AMP_DTYPE = 'float16'         # 'float16' (paper) or 'bfloat16'
    EVAL_AMP = False              # all reported metrics are computed with an FP32 forward pass
    # ---------------------------------------------------------------- inference protocol
    # Several operators of KASIRNet aggregate over the WHOLE input: the SRB resonance gate (mean of the
    # un-normalised FFT amplitude: on a 720x1280 frame it is ~3.75x larger than on the 256x256 training
    # patches), the anti-linear / quadrature part of the SRB complex 1x1 mixing (point reflection and
    # row-wise Hilbert transform of the whole map), the SIM channel attention and Prompt-Learner global
    # pooling, the GCB softmax pooling and the Meshgrid coordinates (stretched over the whole image).
    # A full-image forward pass therefore feeds them statistics never seen in training (cf. TLC, Chu et
    # al., ECCV 2022).  'tiledP' restores every image from overlapping PxP tiles (hop P/2, sin^2 window =
    # exact partition of unity, reflect-padded borders so that every tile has full size): each operator
    # then sees exactly the training regime; images not larger than P go through the model directly.
    # The weights are untouched.  'auto' selects the protocol once, on the held-out validation images of
    # the training pool at full resolution (never on a test set): INFERENCE_CANDIDATES.
    INFERENCE_PROTOCOL = 'auto'   # 'auto' | 'full' | 'tiled256' | 'tiled384' (+ '+x8' = 8-fold self-ensemble)
    INFERENCE_CANDIDATES = ['full', 'tiled256', 'tiled384']
    INFERENCE_SELECT_MAX_SIZE = (720, 1280)   # validation images are centre-cropped to at most this size
    INFERENCE_SELECT_MIN_GAIN = 0.02          # dB; below this the simpler protocol ('full') is kept
    TILE_BATCH = 32
    CUDNN_BENCHMARK = True
    DETERMINISTIC = False         # True -> cudnn.deterministic (slower); seeds are always fixed
    MAX_SESSION_HOURS = 23.5      # save + stop cleanly before a Colab session is recycled
    # Several Colab sessions (each with its own G4) can run main() at the same time on the same
    # Drive folder: a run being trained is locked (heartbeat at every checkpoint) and skipped by the
    # other sessions; a lock older than LOCK_STALE_MINUTES is considered abandoned.
    LOCK_STALE_MINUTES = 45

    # ------------------------------------------------------------------- paths
    DRIVE_MOUNT = '/content/drive'
    OUTPUT_ROOT = '/content/drive/MyDrive/KASIRNet_revision'   # checkpoints, results, tables (persistent)
    DATA_ROOT = '/content/datasets'                            # local disk: downloads and restored caches
    # ---- dataset cache on Google Drive.  First session: every source is downloaded (kagglehub,
    # Hugging Face, Google Drive) and then packed into ONE uncompressed tar per source in
    # DATA_CACHE_ROOT; later sessions extract these tars to local disk instead of downloading.
    #   'used' : only the files the experiments read (frozen training manifest + every benchmark);
    #            if the training manifest has to be rebuilt the full source is downloaded again
    #   'full' : the complete sources (about 55 GB of Drive space)   |   'off': no cache
    DATA_CACHE_ROOT = '/content/drive/MyDrive/KASIRNet_revision/datasets_cache'
    DATA_CACHE_MODE = 'used'
    # Which sources go to the Drive cache.  'fragile' (default): the Google-Drive and Hugging Face downloads
    # (Snow100K test, Snow100K train, DPDD; about 7 GB) - Kaggle sources are fast and mirrored by Colab, so
    # caching them would only fill the Drive quota (about 40 GB).  'all': every source.
    DATA_CACHE_SOURCES = 'fragile'
    DATA_CACHE_REFRESH = False    # True: rebuild the tars of this session's sources
    # Google-Drive archives often fail with "quota exceeded".  A copy of the archive placed by hand in
    # DATA_CACHE_ROOT/archives/<file name> (Drive web page -> "Make a copy" / "Add shortcut") is used first.
    WEIGHTS_CACHE = '/content/drive/MyDrive/KASIRNet_revision/weights_cache'   # pretrained networks (torch hub cache copy)
    LEGACY_CHECKPOINT = '/content/drive/MyDrive/Checkpoints/KASIRNet/best.pth'        # submitted model (audit only)
    LEGACY_NOSIM_CHECKPOINT = ''                                                       # submitted "w/o SIM" model, if kept
    # Competitor outputs held by the authors (restored images named like the test inputs).
    COMPETITOR_ROOT = '/content/drive/MyDrive/KASIRNet_revision/competitors'
    COMPETITOR_OUTPUTS = {          # benchmark -> {method name: folder}   (TODO-12C, TODO-14)
        'hide':     {'Restormer': f'{COMPETITOR_ROOT}/hide/Restormer'},
        'textocr':  {'CNN-ViT': f'{COMPETITOR_ROOT}/textocr/CNN-ViT'},
        'kohler':   {'Rahman et al.': f'{COMPETITOR_ROOT}/kohler/Rahman'},
        'reside6k': {'ADSM': f'{COMPETITOR_ROOT}/reside6k/ADSM'},
        'sidd':     {'Defusion': f'{COMPETITOR_ROOT}/sidd/Defusion'},
    }
    FIGURE_COMPETITORS = {          # supplementary Figs. S1/S2 (TODO-06): up to 3 methods each
        'rain100l':   {'PromptIR': f'{COMPETITOR_ROOT}/rain100l/PromptIR', 'AdaIR': f'{COMPETITOR_ROOT}/rain100l/AdaIR',
                       'Restormer': f'{COMPETITOR_ROOT}/rain100l/Restormer'},
        'snow100k_l': {'TransWeather': f'{COMPETITOR_ROOT}/snow100k_l/TransWeather', 'WGWS-Net': f'{COMPETITOR_ROOT}/snow100k_l/WGWS',
                       'Histoformer': f'{COMPETITOR_ROOT}/snow100k_l/Histoformer'},
    }

    # --------------------------------------------------------- data sources
    # kagglehub.dataset_download(slug) returns the local root of each source.
    KAGGLE_SOURCES = {
        'dblur':    'jishnuparayilshibu/a-curated-list-of-image-deblurring-datasets',  # GoPro train/test, HIDE test only, TextOCR, CelebA, Helen
        'kohler':   'eshapahwa1302/kohlerdeblur',              # 4 scenes x 12 kernels = 48 pairs
        'realblur_j': 'adwythdarsanr/realblurj-image-deblurring-dataset',  # RealBlur-J official split: train 3758 (182 scenes) / test 980 (50 scenes)
        'jrdr':     'shivakanthsujit/jrdr-deraining-dataset',  # RainTrainH / RainTrainL (1800 each)
        'raindrop': 'chenchienying/deraindataset',             # DeRaindrop: train 861 / test_a 58 / test_b 249
        'rain100l': 'adeshpal/rain-100l',                      # Rain100L test (100 pairs, Rain100L/{input,target}/<n>.png, 480x320:
                                                               # the MPRNet/Restormer test layout; input 25.5 dB RGB).  NOT
                                                               # bshaurya/rain-dataset: that one is Rain100H (input 12.1 dB, the
                                                               # same 100 pairs as alicewen820/rain100h)
        'rain100h': 'alicewen820/rain100h',                    # Rain100H test (100)
        'reside6k': 'kmljts/reside-6k',                        # RESIDE-6K train 6000 / test 1000
        'sidd':     'soumikrakshit/smartphone-image-denoising-dataset',  # SIDD-Medium sRGB (320) + SIDD val crops (lmdb)
        'sp_noise': 'rajneesh231/salt-and-pepper-noise-images',
        'docs':     'mohamedkhaled201841/image-denoising',     # noisy documents (train/train_cleaned)
        'alphaisp': 'tenxengineers/denoising-dataset-multiple-iso-levels',
        'cbsd68':   'tarekmebrouk/cbsd68',                     # clean images for Gaussian sigma 15/25/50
        'lol':      'soumikrakshit/lol-dataset',               # LOL eval15
        'live1':    'nikshko/live-1',                          # LIVE1 references (29 bmp)
        'legacy_raining': 'adeshpal/image-deraining',          # audit only: the "raining" category of the submitted pool
        'cdd11':    'haohoangofficial/cdd11-full',             # CDD-11 (OneRestore, ECCV 2024): 1,183 clean x 11 degradation combinations
        'snow_test_l': 'suzukimio/allweather-testing',         # Snow100K-L official test set (16,801 pairs, AllWeather_Testing/Snow100K-L);
                                                               # names and byte sizes identical to the official archive
    }
    # Google-Drive archives (official links of the dataset authors).  Layouts checked on the archives:
    # training tar -> all/{gt,mask,synthetic}; test tar -> media/jdway/GameSSD/overlapping/test/Snow100K-{S,M,L}/{gt,mask,synthetic}
    # (The official Snow100K Google-Drive archives - training 1SGS2w2RBhfwAMn9dMQZUnoQcqCcz1baU, test
    # 15z-A3i3Z2ybdKuyxwYRA8YKxIA8_kTDm - fail repeatedly with "quota exceeded"; they are no longer used.)
    GDOWN_SOURCES = {}
    # Snow100K from Hugging Face (the archive released with ConvIR, no quota): train2500 = 2,500 pairs of the
    # official training set; test2000 = 713 Snow100K-S + 662 Snow100K-M + 625 Snow100K-L official test pairs.
    # Checked against the official archive listings: every file has the official name and byte size.
    HF_ZIP_SOURCES = {
        'snow100k_hf': dict(repo='fancongyi/snow100k', file='Snow100K_ConvIR.zip'),
        # DIV2K training HR images (800 PNG, 2K): clean images of the synthetic categories (same size as the
        # official ETH archive DIV2K_train_HR.zip, 3,530,603,713 bytes)
        'div2k': dict(repo='deepinv/DIV2K', file='DIV2K_train_HR.zip'),
    }
    # Pairs extracted one by one from a remote zip (HTTP range requests): a seeded subset of an official split.
    # RSBlur (Rim et al., ECCV 2022): real hand-held camera shake recorded with a beam-splitter rig, sharp
    # reference aligned with the blurred image (1917x1199).  The real counterpart of Koehler's camera shake;
    # official training list (8,878 pairs, 19 scenes), never its val/test lists.  ~6.6 MB per pair.
    HF_REMOTE_ZIP_SOURCES = {
        'rsblur': dict(repo='rimchang/RSBlur', file='RSBlur.zip', n_pairs=1000, seed=2026,
                       list_url='https://raw.githubusercontent.com/rimchang/RSBlur/main/datalist/RSBlur/RSBlur_real_train.txt'),
    }
    # names of the Snow100K-S / -M official test images contained in test2000 (zlib + base64)
    SNOW_LEVEL_NAMES = {
        'S': (
        'eNqNm21yHDcOhv/vKXKBbBFfBHkalWIrthJFSknyunz75WjImSHx0u74R1wP0CAIgiCaPf7j4f7b++Of357u3v55fHq4S+0//e9f/375zx9RYr'
        'yRSLGNxEmwhEg31og1byTiO2vqtJOUncR0L9nMh9x0J/GEJdz+7CTFNxLispE00U5S60YiZRNR1rrzwEx2krqzto2OpLrxWjRvrInxZhUk76Ij'
        'nnbPeKadpGzWR5NtnlHSvaTuJHk3juTzfD7dv71/bBuiPIMiPoNKdQF9JT89vv+4e324/3zalx6ZJopsDHfLRAKzzIHlvlduWdH4bO1xuWHElg'
        'DTwDJ7ZB58oYJYjePWFPVqnAdL0siUIqthHuwenhWiMA8Z1W5iFsYQ0eCzmIe5aau5kcW4KEsYV0etvGUuDFgYw1JcS7M4jzx2wS0DOZmVIysx'
        'prnGfGlFIMTFiTiyHGLgIiEPXGMMXC3EwI1SZDU8W5yCfzXFXGtbJvhXcy2B1ehzrWFcOh2tgalQYCWt+5donP63TEN9IbKwbi3reZ0vccwDYl'
        'aNzKKehPxrMyOKrKzxI0khViQU6hoJxxi0vZoi8xKYUBwj5gtJrDntXNGgp5xqZBbG1bjfSHOK9kZXdsMsZYqsKmDBF4t1srVQJepZzJdMEuaR'
        'hSOzsKcbizmeK4e5eQr7tzGNeq24B5ZjXDyX4J97DnMrCTANNYJKjv7VFM6KxjzkS6W4PyqXaE9SZNXX/cGt99DALOQaN1dSZDU+WyQ+W0PsGw'
        'vryzT641sW9+CJBXukYZ8zWVg3bgFc15dbj41Y8KV1wmG+nEOf0xpzDj5LO+ECi/nHoL7wpf+9ZU6dvb68vT0+fzl1kaOsXREPT65IiqxImSOq'
        'K7KRKVfkNTx46b8uiJKvDxKPk+6KJNuKNHFAtk67bWNfUeVg/tKPXhCnnAIKfrVMWEdkjw9ettYFyaWpvyL2dY4ioxe6otbTrcjSumgtAVZbrc'
        '7ngDSgErSa9RRQXke0HGJvl27xhF5fzm8hqQamWQOz7sefj69v73dvzy/fP7JTI9ReKyaY++E3Qe83Bv/ev/598vDp5f20MaQi2hdjplaRrvcl'
        'mWlRZKH0rnqmte/ridLoOmY6cmKhhiyMyM60MLIA/T0dKZG2IgZGa10XpL2vnmmG1NHceJwaE5VxNi0UxUFyqogaiG8rK4Rof2uaqfYzaaa5IL'
        'vuYMZakGdakN22EYEFEwYWTCuIjhnK9UxJESVBFO0LTwx0PTPYF14q0q3IM68GLBS4L0q2iijKs9YWAt02GIjD5R3rltLp3EJUE6DsFVAD+duo'
        'xrkRUUGUQT40asAzKiAfWoFBs2itG5hFa6wYUUEWOBOgwsgHpYqoIOqC7Bawj9t7XGJEGekyqBrtLU0U0eyIgt1NsO40mpEPnpG/BeRk63PUES'
        '3AB82EdF2AD1pALWkU5Zmi86JRR7SiFTIyYNcsJ0QL8Mxyr1Fvn76+vDzdPT9++fpxePdpLJix9igGCx73aTNu+yshLP2bw4JH87xiQZ7QeGVf'
        'cM7QSBGE2+7NAFvtvdOMMwsaMo/LqwXn3oav2JFtH29ECy6E/C6jkgRcIa5QmzPyu4w7mBX3YvD2+Pnh+/3T3799f3x+f3j97fcvr/c/3j7dPz'
        '2c/3r63tVfGg6oWipHVUuW46r1qGodifFLVaJUD6v2OnJAlXsLdUTVjwartbiHp6X9bD2g6nrU1/aud1hV9agDbFQOq9rRJWjd8dHMEqlHfZVx'
        'j3dANfeXogOq3s+MA6pVjvqqdHhvNdWj09Jxs3xENR+NQNM8GoH2gn7Y6rhp+rWqpcMlw8SPBstUDzug+Whcczq8DVvrczRdshIfVj28sO1gPG'
        'y1HF6t1inqYdV+efNxBXL//PjP/dPpNOtvRTPl3tEttN95zLT2N4eFjlW8pTR+PTBTd6DLiYC/TP2b+EyNgQ+cC5ib0Gh/JsoKLIgysjC+L3zQ'
        'mzvNKZYTr5BfepET/1i3t4dP748vz6dnSLYyKjf2/vfw9fHTx6f+cU28YPUNPk+4J8ztEL/93q/JhhM/05F+wP5UJ/9aZ3zI+ZmO0699LvrLsW'
        'h8VP2ZDvV25Kxzd/f2/vrw8PFK0Vc/CEZzHgSl/5JkFVCd5nMV8PjSHwQiWwF2V5JXLBh3lavg8vOWIBjfnYOgKp55638FC3hagRtBzjgkmSp2'
        'N+tm8EvBiwLfPOEVz7yQ4SeKFd0IKl6PkhUL6vhK1gV/fHt8+tzrRE4bybgxiZLxLS1IWvLvJKNPApK6k4waBiTzutxIzNJOUmgjyVSwhMcXvS'
        'gx8Y1k/PIgSCTtZiq0m6kU2qycJuOdxDfWlEveSFRoJym7cXLdxE1L3jxjKZedpGxWzjjrVrKJtdnON/N56359+fb2ccIxIaxzxAZ2g9plXuGB'
        '61yROqYEh6TlEBjY5v3YMfNcnC94zoGBCxHEFWrXjCYvbGiWkisKrFRou73hoOkoz/tqYCGBmKFtI4aYUahan41mefnF2YKLQ09KQUPaaDRXnJ'
        'EnJopsm8LpWJ66rLu/7v+9/+ju5owdmOaydMEKtcdPNxZsogiXDI1Uhdp1jmDHlCrSJiqMMRrydOOMsM11fOCSUQSpWgGYkxPCOp84A5sjB3l8'
        'iVuxICPijIxIqUhbl6wamCoasrXuEBfDGNuuFTloCfrdyjVaBhtfbBacSSDGRhwusVVC08nsKIJZqEAMY5IdrqWTICPOhIb0nARiQzHx8fOKBS'
        '9dwsAVBrYkQ2lfKKEItnceNGSZ35cuuCY0y7o06wNbUoiNIC7QtsH8roUwhmtZKzJCp9dUhKkowowKByUljBXiXBxhR1lFRGiTNKiEMTTicDoE'
        '987p9yIMcYba5FCbs0BcIRZUTInHJ8MVo/LYzr+KYiIpIwfFDHkimQ3hShijZCNd3psHZkPTUVgeTz82RLNUTdC2oc3dcEWT14w2YMNwGQyvpU'
        'mC2uNz2IodOWiKtcdF7opRD3H6rAtxJWikChoyw3arYZhsuTg0UlEDRU7oEKB2NjDEjPz2wgViRX4XgtMpimoslaVQPz98v/vx8vr3xwsSbyQ+'
        '7+WrhHxe66uEJflGssTiKhGbi9pV0k6ujQe6lLarxMg2z5jPB/hV0vqAncRlE7dSeONBe1ncRKc1W1MMTr9SODX8iuiS3IPOfc+ZtrLrgMrcxH'
        'Vqgmie033QCnxo7SjwgXMxQD2BuUlCPoiIIGpgtPauiiw4G6KlAFodzE1truxnauNSfaaM4mAw6plSRdTBumXNYBbZMhgt5zk/Oy0J2S1UEDUQ'
        'B08KKdQVBp65o3XzgnTLsmM6Xa5iB62IFpSp1ZCF6jnmWev9xACtIKuptX4JUNW4Fq2tAtlHrCD7TreiBVGwN1trwmA0Xa7NztQY7KzT9zZEHd'
        'QSal0TA2ogS1opKUC3mc2AovylwvNVb/+x99Jbjp+KF0SXy7ZOfX4p63S54DpTygnoSp67+E6XC4czVZr38eku/+NSnAAtc0adKS9rcaats+VI'
        'VQ3YVZtPqU59rr9n6skRXc7gMy3LlVanNQELVQTMolpGunn+stZp4RhJuvzOf6JkNdolduBvOyDn+H5/uH//2v6fWkVKUNBOWoECpjn8F4HY/L'
        'Z1FSyl5SLQ8W9dVsF663gj2Jiy5Wr0IsjJ8DzWjX4VjIr1f3uAgjw='),
        'M': (
        'eNqNm+FyW6kOx7/fp+gL7B2EJISeJuO2buvdbLITp7fTt784PjgG/dmcL23nJyGEACE47ufj4efr6dvPx4fz36fH40NKWfS/f/7z/T+fg0RFVp'
        'JqC0lZWjOvC0mVlbVqKw9qTQuJp7KSlIUHlHzhNZHLUrJqo7rqp+jCN3JaSnhlzVcxyKnmhSTbSqLGC4llWUl84QHnTAuJrDxgpVUbp4UHQqvo'
        'CGd/k3w5nF/fpp8ojyArzUBGwD41kZQmIHUEWqZui8gMJg0jHUFNU7dVpya1bK6fXn8/vBwPXy/jy4Btk3rPWEpgdQvwwBiwEuwRSQbMIgNtcw'
        '6+EAN7t7jfsx7pO2YUmacaGevMMqXQb84S+m0ojCOXFO1Vj/Z6crpj3BQDEwlzySqhXy4e2kqiMF6hGL/GSmRxvYjGuZSag56CPpRr1BPANIc+'
        'Wj4qkcU1pLUGvUIlxL5kDzEtbKHfInFNtj2cIrPYh6pGVqI9o9DWUlzPRnGfW05hXxrH+LUUEXxxSsGei4eYutGsR5dKIbAS9hslC7Fv52tYpw'
        'T2eUsbdY4zZSmh31w16DHV4DPHtUZtz1hgFscmKayrxsKebtlPgz1hCm3b/qiRedRTC+PQEtYVqYc92FZ9jGlhD22buRC/YjF+luM4TELubEzC'
        '/FovHwdmIQZmIeeQVY/MJfriCvTiGqoU9iDVeFY0FvY0tTQOWPTZkwZ7TtE/zx5i4MWCz20PlsjinDdXwvpzD2d3btVNDkwp6sU5avVkiH1jVQ'
        'PzcG7lS0IITELsGwu5ODeDkXHIYY1pGEe713BgNeTExjwF5mEPZqbYlinkjcztQI+MI7MYezYOPnPt6+Dl+Xw+PX1vZVjigKivgjtUAuKsM5I0'
        'I7pVH++o9ER6j0LDGpzIiWZb+Ra0d5T7eXiHgvnMmQPS0CP3/fWObunuHVmZQ8hJA3Kf/RKT2Xw7tGak0g+8d1SDX+pp9qvcTsB3VEIILdFdcF'
        '6er7V+zYEZXcPz7fRyfn04Pz3/uiyCkiK0bbH8c3j569LH4/PrRdUcUN18HKkpstCz+0BbMYEokSOKfCAV4ENLXsBu234F0Lrt04EybdXVSLko'
        'oFKBZ6woklwy8Iydc6StegC9Sa8jJ1pBb1IM6ConMGKtaGylzRGgmcAoSr+4j9QZRMcYzXyrKYBnpg5G0a4AYIbawZ0QRfGtGflbOQELVQT460'
        'zABy+AtorbEqBZIHUFlBOkTIAKpEoZUY9jo7Y3C6JugBIaRSthkd0iwLMWHEXUQCTJBejm/owzUmbgQ3Ywm9QKCge0DRlRkDXaPcOBv6wJRJ2L'
        'gVGwg/xwKZNAb5IZUa7Ah7Z8wQxJSai3UkAkxaAP/QVloCoKdLXv2POXH8/Pjw9Pp+8/LkeDbAfWhHXLMTNWaKRs2XbCthVwE64Z4+1leMTUc+'
        'CMKzJyuQgiXLe0PWE3NJx2t0e2c39Pm3DerhwTroS0S98oI7a+32dMWDsj20aEsUMj/Xo8YcloQZg5xA61K2114/n09fjr8PjXp1+np9fjy6c/'
        'vr8cfp+/HB6P13++vfemvaq21XMfq5LmvVapyn7VbWd9rJoT7/W13ZH2BqudjrtVe8W2QzXTbqs9Z+1R3R3XWzbeodoT4ceqosl3q/ZE8LFq3U'
        '7xHaq+O1jSP4t9rKpptwNaepL8WNX2q1bereq0dwra8ZR2q26F9Q5Vod2qdfeOtZK2RHu5th2eTn8fHt9ubx5pO5ESoHW7yo/U+/Ye6VaBD7Qd'
        'UhVQ3mrtiSLP8m0vj7SAsXG/So9U9U73/mmEGHKmO/3/HX+cvlw/17FDvNXh2zy8/Xk+fnk9PT99+mO7MZN+rNNn4N90+pvev+nU7V3yX3So+B'
        '6deqfz8HB+fTke3y7627EeBGUrGmYB9c+NQWBVsaBuH1ZmQe71dxD0sicI8lbrBgELHmAWWXTeX1eCoH9rCgKvGQo4LYLI1XBIJK0EtOi8lfN4'
        'HNKfvWeB0mLONS/60Co4uu36vxCsWlh/TokCwUG8fd6KAqaFwPDI2+V/0YIFLzgTWpgqtjBlq85Xq732N5Qg6NevWeA58UIwbpzPP0+PX7c0KG'
        'kpqQsJDYnoXtKrtCjRumpTxs1wL7GV16WufOtXmiAhGtLevSSXRQyoP9FFSfHFSG/HZpT4uGLfJZlsJWFeScrSWtHFSHN/RgwSTraSLNcO5ywr'
        'ySrWzLSIDvenPSCpq3503Al3krpqI/31Pkr6xRxIZGWtfxSPEi2rNv0pNkoK5ZWkLtaoVFlZ66VakLSb5yJuKmmY7W+n4+PXtxuwAkxp7OHH88'
        '/z28+hxgzccRmTZsfVC8I+rrsNty4JY2SEqELMgjwhHRdnxybQSE1Iu20KiC0ZxAXZzhVrOwwVZ+h3GyRBjLUtZ4QdBlZUHGJFC0JKRl22yy/E'
        '/bVqxsUgdjSXWsbB/3n45/D09hXTEdZxGjq2cT927JwAJoW2STMyQlNMOvYxghtuxQ5DrIaw5Ipwf2ee8ZjHOzZGEcwuKFScHOL+Y5YJC+ySBU'
        '4Da4YYz87tO/iM2SF2FO+27JG2pITiLVmQJyIMsQrqshXmyG8lQRHU/kuGERcuFWJHw2mFJOrSWAniiobTClg0nCpwsVUV5He7+EDtCj3xxApx'
        'RaN0gl063vOucJRe4U5zR55cvsQZxGjFtqtPgUa0gjRDt2/iEy5FILaCcHWwTigLWlWUTdBw2nGE8Hy9vWG004ipOMKsaPDtlIJGDEawVYEo3k'
        'IonzRsyG+BO62tKYW2C9YucC5bwUEIw1xFeBdTqegkabgKxHDSDG6phgvUzpUhhqvKGC4Im24wHZuhwLbEgfGYIZ6Ovx5+P7/89fb7dV1I6tjx'
        'vSQvrOVpUd9JpoVwJynjIfIuYRoT752EbeEbGy8k4rmsJLLwQPsHRCCxRdzUbBGDUsuiH6O6iI65L9rUbAvfPI0b4/L5+K3wEkB9nJcrzVPNuV'
        'FKwG5mAnZzKUh3evi5Up5eEzdax9x2pe1aCCy0K2GNVKeL70ZNgGeFUcyKZeBZme40nRbgb6sSgGfmY4LuVICFtg0F0fGk3aihOFQTQtTAKHx6'
        '8t1oBT5cfotTAC1MgE71Zqcl+kDt8FZAmRzQKVFv1AlYyIkzoGj9trOvArtSwA5oFI1YSRE1AnbVK7BbpqejKzWFFNo1AzuL4DqjymPlsf3mMY'
        '2H1Ua5It3piN2ojw+UV0paEC3jPfpKuf9fmJFmSK0Af3mqkK9UkgHdVuAg3cwF0fG47XTcWZf35cv3wzGTb3RaO1fKOj7zXWm7+wDdltaBXe2/'
        'lR/p9L3jSosL6M3qWBlcaZWx7tjodMm8UrcxR220UrR7+VkqoHm6SW7UQXQaHYvHK+U8PoL/Oh5ef7S/Lz+MKlgwvS3cBJfKEgvqWM3eBJlHl2'
        '4CntbCTTBfnW8CneJ+L7i6+3/CAaVS'),
    }
    # Hugging Face parquet mirrors.  DPDD test (76 pairs, 1680x1120, original file names, combined image
    # 'source' = inputC, 'target'); the official Google-Drive file (DPDD_GDOWN_ID) is the fallback.
    HF_PARQUET_SOURCES = {
        # DPDD training split (350 pairs, combined image 'source' = train_c/source, 'target' = train_c/target)
        'dpdd_train': dict(repo='JacobLinCool/DPDD', files=[f'combined/train-{i:05d}-of-00012.parquet' for i in range(12)],
                           columns={'source': 'train/inputC', 'target': 'train/target'}, key_column='source'),
        'dpdd_test': dict(repo='JacobLinCool/DPDD', files=[f'combined/test-0000{i}-of-00003.parquet' for i in range(3)],
                          columns={'source': 'test/inputC', 'target': 'test/target'}, key_column='source'),
    }
    DPDD_GDOWN_ID = '1dDWUQ_D93XGtcywoUcZE1HOXCV4EuLyw'

    # -------------------------------------------------------- training pool
    # family is used by the prompt probe and by the per-family reports.
    # match: 'stem' (same file stem) or ('regex', input_regex, gt_regex) -> key = joined groups.
    # exclude_overlap_with: training pairs whose clean image is a near-duplicate of a clean
    # image of these benchmarks are removed (leakage audit, see Datasets.LeakageAuditor).
    _JRDR_MATCH = ('regex', r'norain-(\d+)x2', r'norain-(\d+)')
    TRAIN_CATEGORIES = {
        # (DBlur has no HIDE training split: HIDE is a test set only, the standard GoPro -> HIDE protocol)
        'gopro':      dict(family='blur',  input=('dblur', 'DBlur/Gopro/train/blur'),   gt=('dblur', 'DBlur/Gopro/train/sharp')),
        'textocr':    dict(family='blur',  input=('dblur', 'DBlur/TextOCR/train/blur'), gt=('dblur', 'DBlur/TextOCR/train/sharp')),
        'celeba':     dict(family='blur',  input=('dblur', 'DBlur/CelebA/train/blur'),  gt=('dblur', 'DBlur/CelebA/train/sharp')),
        'helen':      dict(family='blur',  input=('dblur', 'DBlur/Helen/train/blur'),   gt=('dblur', 'DBlur/Helen/train/sharp')),
        # Real recorded camera-shake blur: replaces the Koehler category of the submitted pool (Koehler has no
        # training split; its 48 images are now a test set only).  Official RealBlur-J training split, scenes
        # disjoint from the RealBlur test scenes; checked against the Koehler ground truths by the leakage audit.
        'realblur_j': dict(family='blur',  input=('realblur_j', 'RealBlur/train/blur'), gt=('realblur_j', 'RealBlur/train/gt')),
        # RainTrainH: rainy images are rain/X2/norain-NNNx2.png, clean are norain/norain-NNN.png
        'rain_heavy': dict(family='rain',  input=('jrdr', 'JRDR/rain_data_train_Heavy/rain/X2'), gt=('jrdr', 'JRDR/rain_data_train_Heavy/norain'),
                           match=_JRDR_MATCH, exclude_overlap_with=['rain100h', 'rain100l']),
        'rain_light': dict(family='rain',  input=('jrdr', 'JRDR/rain_data_train_Light/rain'), gt=('jrdr', 'JRDR/rain_data_train_Light/norain'),
                           match=_JRDR_MATCH, exclude_overlap_with=['rain100l', 'rain100h']),
        # raindrops on the glass blur and veil the scene: the clean image has the higher contrast
        'raindrop':   dict(family='rain',  input=('raindrop', 'train/data'), gt=('raindrop', 'train/gt'),
                           match=('regex', r'(\d+)_rain', r'(\d+)_clean'), direction='contrast_gt'),
        'haze':       dict(family='haze',  input=('reside6k', 'RESIDE-6K/train/hazy'), gt=('reside6k', 'RESIDE-6K/train/GT'),
                           exclude_overlap_with=['reside6k']),
        'snow':       dict(family='snow',  input=('snow100k_hf', 'Snow100K/train2500/Snow'), gt=('snow100k_hf', 'Snow100K/train2500/Gt')),
        'sidd':       dict(family='noise', root=('sidd', 'train/train'), input_glob='**/*NOISY_SRGB_*.PNG', gt_replace=('NOISY', 'GT'),
                           repeat=8),   # 320 very large images: 8 random crops of each per epoch
        'sp_noise':   dict(family='noise', input=('sp_noise', 'Noisy_folder'), gt=('sp_noise', 'Ground_truth'),
                           match=('regex', r'noisy_(.+)', r'(.+)')),
        # stained documents: the cleaned page (black text on white) has the higher contrast
        'docs':       dict(family='noise', input=('docs', 'train'), gt=('docs', 'train_cleaned'), direction='contrast_gt'),
        'alphaisp':   dict(family='noise', input=('alphaisp', 'AlphaISP - Denoising Dataset/AlphaISP - Denoising Dataset/PNG Data/2DNR Denoising/Noised'),
                           gt=('alphaisp', 'AlphaISP - Denoising Dataset/AlphaISP - Denoising Dataset/PNG Data/2DNR Denoising/Ground Truth')),
        # ---- synthetic categories, degraded on the fly from clean training images (Datasets, section 5b)
        # Koehler-equivalent camera shake: the image-formation model of Koehler et al. (ECCV 2012), i.e. the
        # mean of homographies K R(t) K^-1 along a 6-DOF rotational hand-shake trajectory, integrated in linear
        # light, non-uniform over a virtual full frame, mild sensor noise / JPEG.  Random trajectories (no
        # Koehler image or trajectory), clean images = GoPro training sharp frames.
        'camera_shake': dict(family='blur', synth='camera_shake', repeat=3,
                             input=('div2k', 'DIV2K_train_HR'), gt=('div2k', 'DIV2K_train_HR')),
        # Real counterpart: RSBlur real hand-held camera shake (1,000 pairs of the official training split)
        'rsblur':     dict(family='blur', root=('rsblur', 'RSBlur'), input_glob='*/*/real_blur/real_blur.png',
                           gt_replace=('path', 'real_blur/real_blur.png', 'gt/gt_sharp.png'), repeat=2),
        # CDD-11 training split: low light, haze, rain, snow and their 7 combinations on the same 1,183 scenes
        'cdd11':      dict(family='mixed', root=('cdd11', 'CDD-11_train/CDD-11_train'), input_glob='*/*.png',
                           gt_replace=('sibling', 'clear')),
        # LOL our485 (low light) and DPDD training split (defocus): the leakage audit removes the our485 captures that
        # duplicate an eval15 scene, so LOL eval15 stays scene-disjoint
        'lol':        dict(family='lowlight', input=('lol', 'lol_dataset/our485/low'), gt=('lol', 'lol_dataset/our485/high')),
        'dpdd':       dict(family='defocus', input=('dpdd_train', 'train/inputC'), gt=('dpdd_train', 'train/target'), repeat=2),
        # Composite degradations: 1-3 of haze, rain streaks, snow, defocus, camera shake, low light,
        # Poisson-Gaussian noise, JPEG, applied in physical order to the same image.
        'composite':  dict(family='mixed', synth='composite', repeat=3,
                           input=('div2k', 'DIV2K_train_HR'), gt=('div2k', 'DIV2K_train_HR'),
                           exclude_overlap_with=['gopro', 'hide', 'textocr', 'kohler', 'rain100l', 'rain100h', 'reside6k',
                                                 'snow100k_l', 'snow100k_s', 'snow100k_m', 'cbsd68_g25', 'live1_q10', 'lol', 'dpdd']),
    }
    # Koehler has no official training split.  'test_only' (default, standard protocol):
    # all 48 images are a test set and none is trained on.  'scene_split': scenes 1-3 are a
    # training category, scene 4 (12 images) is the test set.
    KOHLER_PROTOCOL = 'test_only'
    MAX_PER_CATEGORY = 2500       # seeded random subset when a category is larger
    VAL_PER_CATEGORY = 10         # held out from training, used only to select the checkpoint
    DATA_SUBSET_SEED = 2026
    # Leakage audit: every training candidate is compared with the clean images of every benchmark of
    # the same degradation family (and of `exclude_overlap_with`); near-duplicates are removed before
    # the per-category cap is applied.  Screening: correlation of 24x24 grey thumbnails > OVERLAP_THRESHOLD,
    # confirmation: 64x64 thumbnails > OVERLAP_CONFIRM and aspect ratio within 5 %.
    EXCLUDE_TEST_OVERLAP = True
    # 'all': every training category is compared with the clean images of EVERY benchmark, not only those of its
    # family (the JRDR rain training backgrounds are 300 BSDS500 images and include the 68 CBSD68 test images
    # pixel for pixel: a family-only audit kept 528 such pairs).  'family': same-family benchmarks only.
    LEAKAGE_SCOPE = 'all'
    OVERLAP_THRESHOLD = 0.97
    OVERLAP_CONFIRM = 0.95
    # Pair validation (audit stage): per category / benchmark, VALIDATE_SAMPLES pairs are checked for
    # readability, size agreement, alignment (64x64 grey correlation >= MIN_PAIR_CORR) and degradation
    # direction (input blurrier / noisier / hazier than the ground truth).  A category whose median
    # correlation or direction rate is below the limits stops the run (MISPAIR / SWAP).
    VALIDATE_SAMPLES = 40
    MIN_PAIR_CORR = 0.5
    MIN_DIRECTION_RATE = 0.6
    VALIDATION_STRICT = True          # stop before training if a training category is flagged
    ALLOW_MISSING_CATEGORIES = False  # a training category without data is an error (not silently skipped)

    # ------------------------------------------------------------- benchmarks
    _KOHLER_MATCH = ('regex', r'Blurry(\d+_\d+)', r'Sharp(\d+_\d+)')
    BENCHMARKS = {
        # ---- the eight benchmarks of Tables 4-5
        'gopro':      dict(family='blur',  input=('dblur', 'DBlur/Gopro/test/blur'),   gt=('dblur', 'DBlur/Gopro/test/sharp')),
        'hide':       dict(family='blur',  input=('dblur', 'DBlur/HIDE/test/blur'),    gt=('dblur', 'DBlur/HIDE/test/sharp')),
        'textocr':    dict(family='blur',  input=('dblur', 'DBlur/TextOCR/test/blur'), gt=('dblur', 'DBlur/TextOCR/test/sharp')),
        # Koehler: the 12 blurred images of a scene share ONE sharp reference and are shifted from it by up to
        # 10 px, so a perfectly deblurred image still scores below the blurred one.  align='ecc' registers every
        # restored image (and the input) to the reference before scoring, as the RealBlur protocol (Rim et al.,
        # ECCV 2020): least-squares intensity scaling + ECC homography, metrics on the valid region.
        'kohler':     dict(family='blur',  input=('kohler', 'kohler/blur'), gt=('kohler', 'kohler/sharp'), match=_KOHLER_MATCH,
                           align='ecc'),
        'rain100l':   dict(family='rain',  input=('rain100l', 'Rain100L/input'), gt=('rain100l', 'Rain100L/target')),
        'reside6k':   dict(family='haze',  input=('reside6k', 'RESIDE-6K/test/hazy'), gt=('reside6k', 'RESIDE-6K/test/GT')),
        'snow100k_l': dict(family='snow',  input=('snow_test_l', 'AllWeather_Testing/Snow100K-L/input'), gt=('snow_test_l', 'AllWeather_Testing/Snow100K-L/gt')),
        'sidd':       dict(family='noise', lmdb=('sidd', 'SIDD-val-lmdb/SIDD/val/input_crops.lmdb'), lmdb_gt=('sidd', 'SIDD-val-lmdb/SIDD/val/gt_crops.lmdb')),
        # ---- TODO-02
        'rain100h':   dict(family='rain',  input=('rain100h', 'Rain100H-test/rain'), gt=('rain100h', 'Rain100H-test/norain')),
        # ---- TODO-09 Part A severity levels
        # severity levels: the official S / M test images contained in the Hugging Face test2000 archive
        'snow100k_s': dict(family='snow',  input=('snow100k_hf', 'Snow100K/test2000/Snow'), gt=('snow100k_hf', 'Snow100K/test2000/Gt'), names='S'),
        'snow100k_m': dict(family='snow',  input=('snow100k_hf', 'Snow100K/test2000/Snow'), gt=('snow100k_hf', 'Snow100K/test2000/Gt'), names='M'),
        'cbsd68_g15': dict(family='noise', clean=('cbsd68', 'dataset/train'), degrade=[('gaussian', 15)]),
        'cbsd68_g25': dict(family='noise', clean=('cbsd68', 'dataset/train'), degrade=[('gaussian', 25)]),
        'cbsd68_g50': dict(family='noise', clean=('cbsd68', 'dataset/train'), degrade=[('gaussian', 50)]),
        # ---- TODO-17 zero-shot (degradations never trained on)
        'live1_q10':  dict(family='jpeg',     clean=('live1', 'live1'), degrade=[('jpeg', 10)]),
        'lol':        dict(family='lowlight', input=('lol', 'lol_dataset/eval15/low'), gt=('lol', 'lol_dataset/eval15/high')),
        'dpdd':       dict(family='defocus',  input=('dpdd_test', 'test/inputC'), gt=('dpdd_test', 'test/target')),
        # ---- TODO-16 mixed degradations (second degradation added to the first one)
        'mix_gopro_g25':      dict(family='mixed', base='gopro', degrade=[('gaussian', 25)], families=('blur', 'noise')),
        'mix_rain100l_haze':  dict(family='mixed', base='rain100l', degrade=[('haze', 1.0, 0.8)], families=('rain', 'haze')),
        'mix_snow100kl_g25':  dict(family='mixed', base='snow100k_l', degrade=[('gaussian', 25)], families=('snow', 'noise')),
    }
    # Fingerprints of the real test sets: (number of pairs, mean input PSNR in dB, RGB, full resolution)
    # measured on the verified sources.  validate_data() compares a sample of each benchmark with them and
    # stops the audit when a source holds another test set (e.g. Rain100H served as Rain100L).
    BENCHMARK_FINGERPRINTS = {
        'gopro': (1111, 25.11), 'hide': (2025, 23.95), 'textocr': (500, 22.07), 'kohler': (48, 20.27),
        'rain100l': (100, 25.52), 'rain100h': (100, 12.13), 'reside6k': (1000, 13.96), 'snow100k_l': (16801, 18.68),
        'sidd': (1280, 23.66), 'snow100k_s': (713, 25.07), 'snow100k_m': (662, 22.90), 'lol': (15, 7.77), 'dpdd': (76, 23.89),
    }
    FINGERPRINT_TOL_DB = 2.0      # a sample of VALIDATE_SAMPLES pairs; Rain100L vs Rain100H differ by 13 dB
    MAIN_BENCHMARKS = ['gopro', 'hide', 'textocr', 'kohler', 'rain100l', 'reside6k', 'snow100k_l', 'sidd']
    EXTRA_BENCHMARKS = ['rain100h', 'snow100k_s', 'snow100k_m', 'cbsd68_g15', 'cbsd68_g25', 'cbsd68_g50',
                        'live1_q10', 'lol', 'dpdd', 'mix_gopro_g25', 'mix_rain100l_haze', 'mix_snow100kl_g25']
    ABLATION_BENCHMARKS = ['gopro', 'rain100l', 'sidd']      # reference = mean over these three (Table 6)
    SYNTHETIC_SEED = 1234         # per-image seeds of synthetic noise = SYNTHETIC_SEED + crc32(file name)
    HAZE_DEPTH = (0.5, 1.0)       # pseudo-depth ramp (top -> bottom = far -> near) of the ASM haze

    # ----------------------------------------------- training (Section 4.1)
    PATCH_SIZE = 256
    BATCH_SIZE = 64               # G4 (96 GB); Section 4.1 must state batch 64 (the submitted paper said 6)
    # Learning rate of every run: the paper value 1e-4 was set for batch 6; at batch 64 it is scaled by the
    # square-root rule (1e-4 x sqrt(64/6) = 3.3e-4; Malladi et al., NeurIPS 2022), with a linear warm-up
    # (Goyal et al., 2017) because the first updates of a transformer at this rate are unstable in FP16.
    LR = 3e-4
    WARMUP_ITERS = 1500
    BETAS = (0.9, 0.999)
    WEIGHT_DECAY = 0.0
    ETA_MIN = 1e-7                # cosine annealing per iteration from LR to ETA_MIN over the whole budget
    GRAD_CLIP = 1.0
    # Budget.  One epoch = one pass over the (capped, repeated) training pool, i.e. floor(N_train / BATCH_SIZE)
    # iterations (~30,400 samples with the two synthetic categories -> ~474 iterations at batch 64).
    # KASIRNet_full and every ablation run: 150 epochs (~71,000 iterations), the same recipe, so every ablation
    # is compared with a reference trained exactly like it; "full, seed SEED" is KASIRNet_full itself (trained
    # once).  main() prints the measured s/iteration and the GPU-hours left before the ablations.
    EPOCHS = 150                  # reference full model KASIRNet_full (= "full, seed 1" of every ablation table)
    ABLATION_EPOCHS = 150         # every ablation run: the same budget and recipe as KASIRNet_full
    # Final model of Tables 4-5 (run MAIN_RUN_ID): KASIRNet_full continued by a second cosine cycle (warm
    # restart, SGDR, Loshchilov & Hutter, ICLR 2017) of MAIN_EPOCHS epochs, i.e. 300 epochs / ~142,000 updates
    # in total, about the budget of comparable all-in-one models (PromptIR, AdaIR) and ~4x the first G4 run
    # (35,700 updates at 1e-4).  MAIN_INIT_FROM = None trains it from scratch (then set MAIN_EPOCHS = 300);
    # MAIN_EPOCHS = 0 reports KASIRNet_full itself.
    MAIN_RUN_ID = 'KASIRNet_main'
    MAIN_INIT_FROM = 'KASIRNet_full'
    MAIN_EPOCHS = 150             # KASIRNet_main = KASIRNet_full + 150 epochs (warm restart) = 300 epochs in total
    MAIN_LR = 3e-4
    MAIN_WARMUP_ITERS = 1500
    CKPT_EVERY_ITERS = 2000       # resumable checkpoint (atomic write) every N iterations
    VAL_EVERY_ITERS = 5000
    LOG_EVERY_ITERS = 100
    MODEL_SELECTION = 'best_val'  # 'best_val' (held-out split of the training pool, never a test set) or 'last'
    LOSS_WEIGHTS = dict(pix=5.0, fft=0.5, perc=1.0, ssim=1.0, id=1.0, edge=0.5)
    LOSS_CALIBRATION = True       # weights act on scale-calibrated terms (Loss.py); False = raw terms
    SEED = 1

    # ------------------------------------------------------------------ model
    MODEL = dict(base_channels=64, prompt_dim=256, use_sim=True, extractor='vgg19', extractor_pretrained=True,
                 use_srb=True, srb_replacement='plain_conv', use_scwam=True, scwam_replacement='conv',
                 use_meshgrid=True, window_size=8, shift_size=4, masked_shift=False, rel_pos_bias=False,
                 use_prompt=True, use_gcb=True, num_scales=4)

    # ----------------------------------------------------------- run plan
    FULL_RUN_ID = 'KASIRNet_full'
    ABLATION_SEEDS = (1, 2, 3)
    RUN_ABLATION_GROUPS = ['TODO12A_seeds', 'TODO09B_extractor', 'TODO12B_interactions', 'TODO15_loss_sensitivity',
                           'T6_components', 'T7_loss_terms', 'SCWAM_design']

    # --------------------------------------------------------------- analyses
    BOOTSTRAP_RESAMPLES = 10000
    BOOTSTRAP_SEED = 7
    PROBE_IMAGES_PER_FAMILY = 100
    PROBE_FAMILIES = {'blur': 'gopro', 'rain': 'rain100l', 'haze': 'reside6k', 'snow': 'snow100k_l', 'noise': 'sidd'}
    SEVERITY_RUNS = ('full_s1', 'noSIM_s1')   # TODO-09A: same reduced budget, with / without SIM
    PROFILE_SIZES = [(256, 256), (720, 1280)]
    PROFILE_WARMUP, PROFILE_REPEATS = 10, 50
    EXPLAIN_IMAGES_PER_BENCHMARK = 3
    EXPLAIN_OCCLUSION = False
    SAVE_RESTORED_IMAGES = True

    # ======================================================================
    # helpers
    # ======================================================================
    def __init__(self, **overrides):
        for k, v in overrides.items():
            if not hasattr(type(self), k):
                raise AttributeError(f'Unknown Config field {k}')
            setattr(self, k, copy.deepcopy(v))
        self.MODEL = copy.deepcopy(self.MODEL)
        self.LOSS_WEIGHTS = copy.deepcopy(self.LOSS_WEIGHTS)

    def report_run_id(self):
        """The model reported in Tables 4-5 and analysed (final model if configured, else KASIRNet_full)."""
        return self.MAIN_RUN_ID if self.MAIN_EPOCHS > 0 else self.FULL_RUN_ID

    def main_run(self):
        return dict(run_id=self.MAIN_RUN_ID, group='main', seed=self.SEED, epochs=self.MAIN_EPOCHS, model={}, loss={},
                    lr=self.MAIN_LR, warmup=self.MAIN_WARMUP_ITERS, init_from=self.MAIN_INIT_FROM)

    # paths
    def full_alias(self):
        """Ablation run id that is identical to the full model (same seed, budget, model and loss), or None."""
        if self.ABLATION_EPOCHS == self.EPOCHS and self.SEED in self.ABLATION_SEEDS:
            return f'full_s{self.SEED}'
        return None

    def resolve_run(self, run_id):
        return self.FULL_RUN_ID if run_id == self.full_alias() else run_id

    def ckpt_dir(self, run_id):   return os.path.join(self.OUTPUT_ROOT, 'checkpoints', self.resolve_run(run_id))
    def run_dir(self, run_id):    return os.path.join(self.OUTPUT_ROOT, 'runs', self.resolve_run(run_id))
    def analysis_dir(self, name): return os.path.join(self.OUTPUT_ROOT, 'analysis', name)
    @property
    def tables_dir(self):         return os.path.join(self.OUTPUT_ROOT, 'tables')
    @property
    def manifest_dir(self):       return os.path.join(self.OUTPUT_ROOT, 'data_manifest')

    def train_categories(self):
        cats = copy.deepcopy(self.TRAIN_CATEGORIES)
        if self.KOHLER_PROTOCOL == 'scene_split':
            cats['kohler'] = dict(family='blur', input=('kohler', 'kohler/blur'), gt=('kohler', 'kohler/sharp'),
                                  match=self._KOHLER_MATCH, include=r'Blurry[123]_')
        return cats

    def benchmarks(self):
        b = copy.deepcopy(self.BENCHMARKS)
        if self.KOHLER_PROTOCOL == 'scene_split':
            b['kohler']['include'] = r'Blurry4_'
        return b

    # run plan ------------------------------------------------------------
    def full_run(self):
        return dict(run_id=self.FULL_RUN_ID, group='full', seed=self.SEED, epochs=self.EPOCHS, model={}, loss={})

    def ablation_groups(self):
        S = self.ABLATION_SEEDS
        E = self.ABLATION_EPOCHS
        w = self.LOSS_WEIGHTS

        def run(rid, group, seed=1, model=None, loss=None):
            return dict(run_id=rid, group=group, seed=seed, epochs=E, model=model or {}, loss=loss or {})

        variants = [('full', {}), ('noSIM', dict(use_sim=False)), ('noSRB', dict(use_srb=False)),
                    ('noSCWAM', dict(use_scwam=False)), ('noPrompt', dict(use_prompt=False))]
        return {
            # TODO-12 Part A: 5 variants x 3 seeds (also the VGG-19 reference of TODO-09B and default of TODO-15)
            'TODO12A_seeds': [run(f'{n}_s{s}', 'TODO12A_seeds', s, m) for n, m in variants for s in S],
            # TODO-09 Part B: alternative frozen extractors in the SIM
            'TODO09B_extractor': [run('ext_resnet50_s1', 'TODO09B_extractor', model=dict(extractor='resnet50')),
                                  run('ext_convnextT_s1', 'TODO09B_extractor', model=dict(extractor='convnext_t'))],
            # TODO-12 Part B: pairwise interactions
            'TODO12B_interactions': [run('noSIM+noSRB_s1', 'TODO12B_interactions', model=dict(use_sim=False, use_srb=False)),
                                     run('noSIM+noSCWAM_s1', 'TODO12B_interactions', model=dict(use_sim=False, use_scwam=False)),
                                     run('noSRB+noSCWAM_s1', 'TODO12B_interactions', model=dict(use_srb=False, use_scwam=False)),
                                     run('noSRB+noFFT_s1', 'TODO12B_interactions', model=dict(use_srb=False), loss=dict(fft=0.0))],
            # TODO-15: each loss weight halved and doubled
            'TODO15_loss_sensitivity': [run(f'w_{k}_x{f:g}_s1', 'TODO15_loss_sensitivity', loss={k: w[k] * f})
                                        for k in w for f in (0.5, 2.0)],
            # Table 6 remaining rows under the same budget
            'T6_components': [run('noMeshgrid_s1', 'T6_components', model=dict(use_meshgrid=False)),
                              run('noGCB_s1', 'T6_components', model=dict(use_gcb=False)),
                              run('depth3_s1', 'T6_components', model=dict(num_scales=3)),
                              run('width32_s1', 'T6_components', model=dict(base_channels=32))],
            # Table 7: each loss term removed
            'T7_loss_terms': [run(f'no_{k}_s1', 'T7_loss_terms', loss={k: 0.0}) for k in w],
            # SCWAM vs Swin design choices (Reviewer 3 comment 1, Reviewer 5 comment 1)
            'SCWAM_design': [run('scwam_noShift_s1', 'SCWAM_design', model=dict(shift_size=0)),
                             run('scwam_swinMask_s1', 'SCWAM_design', model=dict(masked_shift=True)),
                             run('scwam_RPB_s1', 'SCWAM_design', model=dict(use_meshgrid=False, rel_pos_bias=True))],
        }

    def ablation_plan(self):
        groups = self.ablation_groups()
        plan, seen = [], set()
        seen.add(self.full_alias())   # trained once as the full model, never again
        for g in self.RUN_ABLATION_GROUPS:
            for r in groups[g]:
                if r['run_id'] not in seen:
                    seen.add(r['run_id']); plan.append(r)
        return plan

    def all_runs(self):
        runs = {self.FULL_RUN_ID: self.full_run()}
        if self.MAIN_EPOCHS > 0:
            runs[self.MAIN_RUN_ID] = self.main_run()
        for g in self.ablation_groups().values():
            for r in g:
                runs.setdefault(r['run_id'], r)
        return runs

    def model_kwargs(self, run):
        m = dict(self.MODEL)
        m.update(run.get('model', {}))
        return m

    def loss_weights(self, run):
        w = dict(self.LOSS_WEIGHTS)
        w.update(run.get('loss', {}))
        return w

    def to_dict(self):
        d = {}
        for k in dir(type(self)):
            if k.isupper():
                v = getattr(self, k)
                try:
                    json.dumps(v)
                    d[k] = v
                except TypeError:
                    d[k] = str(v)
        return d

In [ ]:
# ===== metrics.py (identical to metrics.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - THE metric function (TODO-01)
# --------------------------------------------------------------------------
# One implementation, used everywhere (validation, every benchmark, every
# ablation, every competitor output).  It is the pyiqa 0.1.16 implementation
# that produced the figures of the submitted manuscript, called with its
# default options:
#
#   PSNR  pyiqa 'psnr' : RGB, all three channels, data range 1, MSE over C x H x W,
#                        10 log10(1 / (MSE + 1e-8)), no border crop.
#   SSIM  pyiqa 'ssim' : luminance Y of YIQ (0.299 R + 0.587 G + 0.114 B) scaled to
#                        [0, 255] and rounded (MATLAB uint8 convention), float64,
#                        Gaussian window 11 x 11, sigma 1.5, 'valid' filtering,
#                        no down-sampling, C1 = (0.01*255)^2, C2 = (0.03*255)^2,
#                        no border crop.
#   LPIPS pyiqa 'lpips': AlexNet, v0.1.         DISTS pyiqa 'dists'.
#
# Inputs: restored output clamped to [0, 1] and NOT quantised to 8 bit; ground
# truth read from the 8-bit file; full resolution (no patches, no resizing,
# except that a ground truth whose size differs from the input is resized to
# the input size, the rule of the submitted code; such images are counted).
# ==========================================================================
import torch
import torch.nn.functional as F

PROTOCOL = {
    'library': 'pyiqa==0.1.16 (create_metric defaults)',
    'psnr': 'RGB, data range 1, no border crop, eps 1e-8, full resolution, float output clamped to [0,1] (not quantised)',
    'ssim': 'Y of YIQ scaled to [0,255] and rounded, Gaussian 11x11 sigma 1.5, valid filtering, no downsampling, no border crop',
    'lpips': 'AlexNet v0.1 (pyiqa lpips)',
    'dists': 'pyiqa dists',
    'psnr_y / ssim_y': 'additional columns: same pyiqa functions on the Y channel of YCbCr (BT.601, MATLAB rgb2ycbcr), '
                       'the convention of most deraining papers; reported only for like-with-like comparisons',
    'resolution': 'full image, batch 1, no tiling',
    'averaging': 'arithmetic mean of per-image values over all test images',
}


CUSTOM_METRICS = {'psnr_y': ('psnr', dict(test_y_channel=True, color_space='ycbcr')),
                  'ssim_y': ('ssim', dict(test_y_channel=True, color_space='ycbcr'))}


class MetricSuite:
    """metrics(restored, gt) -> {'psnr': [...], 'ssim': [...], ...} (one value per image)."""

    def __init__(self, device, names=('psnr', 'ssim')):
        import pyiqa
        self.names = tuple(names)
        self.device = device
        self.m = {}
        for n in self.names:
            base, opts = CUSTOM_METRICS.get(n, (n, {}))
            self.m[n] = pyiqa.create_metric(base, device=device, **opts)

    @torch.no_grad()
    def __call__(self, restored, gt):
        restored = restored.to(self.device, torch.float32).clamp(0, 1)
        gt = gt.to(self.device, torch.float32).clamp(0, 1)
        if restored.dim() == 3:
            restored, gt = restored.unsqueeze(0), gt.unsqueeze(0)
        out = {}
        for n, fn in self.m.items():
            v = fn(restored, gt)
            out[n] = [float(t) for t in v.reshape(-1).cpu()]
        return out


def psnr_torch(x, y):
    """Same formula as pyiqa 'psnr' (used only for the cheap training-batch log)."""
    mse = ((x.float() - y.float()) ** 2).mean(dim=[1, 2, 3])
    return 10 * torch.log10(1.0 / (mse + 1e-8))


# ------------------------------------------------------------------
# Differentiable SSIM for the loss: identical definition (Y of YIQ, Gaussian
# 11x11, sigma 1.5, valid filtering) but float32 and without the uint8 rounding.
# ------------------------------------------------------------------
_YIQ = torch.tensor([0.299, 0.587, 0.114])


def _gauss_win(size=11, sigma=1.5, device='cpu', dtype=torch.float32):
    ax = torch.arange(size, dtype=dtype, device=device) - (size - 1) / 2
    g = torch.exp(-(ax ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    return (g[:, None] @ g[None, :]).view(1, 1, size, size)


def ssim_y(x, y, size=11, sigma=1.5):
    x = x.float()
    y = y.float()
    w = _YIQ.to(x.device).view(1, 3, 1, 1)
    x = (x * w).sum(1, keepdim=True)
    y = (y * w).sum(1, keepdim=True)
    win = _gauss_win(size, sigma, x.device)
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu1, mu2 = F.conv2d(x, win), F.conv2d(y, win)
    s11 = F.conv2d(x * x, win) - mu1 ** 2
    s22 = F.conv2d(y * y, win) - mu2 ** 2
    s12 = F.conv2d(x * y, win) - mu1 * mu2
    cs = F.relu((2 * s12 + C2) / (s11 + s22 + C2))
    ssim_map = ((2 * mu1 * mu2 + C1) / (mu1 ** 2 + mu2 ** 2 + C1)) * cs
    return ssim_map.mean(dim=[1, 2, 3])

In [ ]:
# ===== Model.py (identical to Model.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet (Knowledge-Augmented Single Image Revamp Network) - model
# --------------------------------------------------------------------------
# With every flag at its default value the network built here is exactly the
# architecture of the submitted paper (same modules, same parameter names, same
# shapes, same forward computation): a submitted checkpoint loads with
# strict=True.  The flags only exist so that every ablation of the revision is
# "one line of Config" instead of a copy of this file:
#
#   use_sim / extractor            : Semantic Injection Module + frozen extractor
#                                    ('vgg19' = paper, 'resnet50', 'convnext_t')
#   use_srb / srb_replacement      : Spectral Resonance Block or its replacement
#                                    ('plain_conv' = same spatial branch, no FFT)
#   use_scwam / scwam_replacement  : shifted cyclic window attention or a local
#                                    residual conv block ('conv') / 'identity'
#   use_meshgrid                   : CoordConv-like Meshgrid coordinate embedding
#   shift_size / masked_shift /    : SCWAM design variants used to document how
#   rel_pos_bias                     SCWAM differs from Swin (mask, RPB)
#   use_prompt                     : Prompt Learner + the four SFT layers
#   use_gcb                        : Global Context Block in the bottleneck
#   num_scales / base_channels     : depth / width ablations
#
# Numerical notes (they do not change the function that is computed):
#   * the FFT path of the SRB always runs in float32, also under FP16 autocast
#     (cuFFT half precision only supports power-of-two sizes and the
#     un-normalised amplitude overflows the FP16 range);
#   * window attention uses torch's fused scaled_dot_product_attention, which
#     computes softmax(QK^T / sqrt(d)) V exactly like the original code.
# ==========================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


# ==========================================
# 1. Core Components (SCWAM & Meshgrid)
# ==========================================

class MesgridCoordinateEmbedder(nn.Module):
    """Normalised (x, y) meshgrid in [-1, 1] embedded by a 1x1-conv MLP (CoordConv-like).
    The embedding is added to the features before window partitioning, so every
    token carries its absolute image position (no relative position bias table)."""

    def __init__(self, embed_dim):
        super(MesgridCoordinateEmbedder, self).__init__()
        self.embed = nn.Sequential(
            nn.Conv2d(2, embed_dim // 4, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(embed_dim // 4, embed_dim, 1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        y_coords = torch.linspace(-1, 1, H, device=x.device, dtype=x.dtype).view(1, 1, H, 1).expand(1, 1, H, W)
        x_coords = torch.linspace(-1, 1, W, device=x.device, dtype=x.dtype).view(1, 1, 1, W).expand(1, 1, H, W)
        coords = torch.cat([x_coords, y_coords], dim=1)
        return self.embed(coords).expand(B, -1, -1, -1)    # identical for every sample: computed once


class WindowAttention(nn.Module):
    """Multi-head self-attention inside one window.  `rel_pos_bias=True` adds a
    Swin relative-position-bias table (ablation only, not used by KASIRNet)."""
    use_fused = True  # fused SDPA kernel; mathematically identical to the explicit form

    def __init__(self, dim, window_size, num_heads, rel_pos_bias=False):
        super(WindowAttention, self).__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)

        self.rel_pos_bias = rel_pos_bias
        if rel_pos_bias:
            ws = window_size
            self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * ws - 1) * (2 * ws - 1), num_heads))
            nn.init.trunc_normal_(self.relative_position_bias_table, std=.02)
            coords = torch.stack(torch.meshgrid(torch.arange(ws), torch.arange(ws), indexing='ij')).flatten(1)
            rel = (coords[:, :, None] - coords[:, None, :]).permute(1, 2, 0).contiguous()
            rel[:, :, 0] += ws - 1
            rel[:, :, 1] += ws - 1
            rel[:, :, 0] *= 2 * ws - 1
            self.register_buffer('relative_position_index', rel.sum(-1), persistent=False)

    def _bias(self, N):
        if not self.rel_pos_bias:
            return None
        b = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        return b.view(N, N, -1).permute(2, 0, 1).contiguous()  # heads, N, N

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        bias = self._bias(N)
        if mask is not None:  # Swin-style shifted-window mask (ablation only)
            nW = mask.shape[0]
            m = mask.to(q.dtype).unsqueeze(1).repeat(B_ // nW, 1, 1, 1)  # B_, 1, N, N
            bias = m if bias is None else bias.unsqueeze(0).to(q.dtype) + m
        elif bias is not None:
            bias = bias.unsqueeze(0).to(q.dtype)

        if self.use_fused:
            # CUDA fused kernels launch one block per (window, head): their grid is limited to 65,535 in that
            # dimension (batch 64 x 1,024 windows x 2 heads exceeds it -> 'CUDA error: invalid argument').
            # The windows are therefore processed in chunks; the result is identical.
            step = max(1, 65535 // (2 * self.num_heads))
            if B_ <= step:
                x = F.scaled_dot_product_attention(q, k, v, attn_mask=bias)
            else:
                x = torch.cat([F.scaled_dot_product_attention(
                    q[s:s + step], k[s:s + step], v[s:s + step],
                    attn_mask=None if bias is None else (bias if bias.shape[0] == 1 else bias[s:s + step]))
                    for s in range(0, B_, step)], dim=0)
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            if bias is not None:
                attn = attn + bias
            attn = attn.softmax(dim=-1)
            x = attn @ v

        x = x.transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        return x


class SCWAMLayer(nn.Module):
    """Shifted Cyclic Window Attention with Meshgrid.
    Differences with a Swin block (all are switches so that they can be ablated):
      * the window grid is cyclically rolled by `shift_size` in EVERY layer and no
        attention mask is applied, so tokens on opposite borders attend to each
        other (wrap-around context);  `masked_shift=True` restores the Swin mask;
      * absolute position comes from the Meshgrid embedding instead of a
        relative-position-bias table (`rel_pos_bias=True` adds the Swin table)."""

    def __init__(self, dim, num_heads, window_size=8, shift_size=4, use_meshgrid=True,
                 masked_shift=False, rel_pos_bias=False):
        super(SCWAMLayer, self).__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.use_meshgrid = use_meshgrid
        self.masked_shift = masked_shift

        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(dim, window_size, num_heads, rel_pos_bias=rel_pos_bias)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )
        if use_meshgrid:
            self.grid_embedder = MesgridCoordinateEmbedder(dim)
        self._mask_cache = {}

    def _swin_mask(self, Hp, Wp, device):
        key = (Hp, Wp, str(device))
        if key not in self._mask_cache:
            ws, s = self.window_size, self.shift_size
            img_mask = torch.zeros((1, Hp, Wp, 1), device=device)
            cnt = 0
            for h in (slice(0, -ws), slice(-ws, -s), slice(-s, None)):
                for w in (slice(0, -ws), slice(-ws, -s), slice(-s, None)):
                    img_mask[:, h, w, :] = cnt
                    cnt += 1
            mw = self.window_partition(img_mask, ws).squeeze(-1)
            m = mw.unsqueeze(1) - mw.unsqueeze(2)
            self._mask_cache[key] = m.masked_fill(m != 0, -100.0).masked_fill(m == 0, 0.0)
        return self._mask_cache[key]

    def forward(self, x):
        B, C, H, W = x.shape

        if self.use_meshgrid:
            grid_emb = self.grid_embedder(x)
            x = x + grid_emb
        shortcut = x

        pad_l = pad_t = 0
        pad_r = (self.window_size - W % self.window_size) % self.window_size
        pad_b = (self.window_size - H % self.window_size) % self.window_size
        x = torch.nn.functional.pad(x, (pad_l, pad_r, pad_t, pad_b))
        _, _, Hp, Wp = x.shape

        x = x.flatten(2).transpose(1, 2)
        x = self.norm1(x).view(B, Hp, Wp, C)

        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x

        mask = self._swin_mask(Hp, Wp, x.device) if (self.masked_shift and self.shift_size > 0) else None
        x_windows = self.window_partition(shifted_x, self.window_size)
        attn_windows = self.attn(x_windows, mask=mask)

        shifted_x = self.window_reverse(attn_windows, self.window_size, Hp, Wp)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x

        if pad_r > 0 or pad_b > 0:
            x = x[:, :H, :W, :].contiguous()

        x = x.view(B, H * W, C)

        x = shortcut.flatten(2).transpose(1, 2) + x
        x = x + self.mlp(self.norm2(x))

        return x.transpose(1, 2).view(B, C, H, W)

    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size * window_size, C)
        return windows

    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x


class LocalConvBlock(nn.Module):
    """Replacement used by the 'w/o SCWAM' ablation: residual depthwise 3x3 +
    pointwise MLP (local mixing only, no attention, no coordinates)."""

    def __init__(self, dim):
        super(LocalConvBlock, self).__init__()
        self.body = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim),
            nn.Conv2d(dim, dim * 2, 1),
            nn.GELU(),
            nn.Conv2d(dim * 2, dim, 1)
        )

    def forward(self, x):
        return x + self.body(x)


# ==========================================
# 2. Spectral Resonance Block
# ==========================================

class SpectralResonanceBlock(nn.Module):
    """F = rFFT2(X);  A = |F|, Phi = angle(F)
       G = sigmoid(W2 ReLU(W1 GAP(A)))                  (channel gate computed on the amplitude)
       A' = A (1 + G)                                    (phase-preserving amplitude resonance)
       [R; I] = Conv1x1([A' cos Phi ; A' sin Phi])       (complex 1x1 mixing: MAY change the phase)
       Y = X + DWConv-PWConv(irFFT2(R + iI))
    Only the amplitude gate is phase-preserving; the learned complex mixing is not.
    The gate is not zero-initialised (G ~ 0.5 at initialisation)."""

    def __init__(self, dim):
        super(SpectralResonanceBlock, self).__init__()
        self.freq_conv = nn.Conv2d(dim * 2, dim * 2, kernel_size=1)
        self.resonance_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(dim, dim // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 4, dim, 1),
            nn.Sigmoid()
        )
        self.spatial_restore = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim),
            nn.Conv2d(dim, dim, 1)
        )

    def _spectral(self, x, trace=False):
        B, C, H, W = x.shape
        with torch.autocast(device_type=x.device.type, enabled=False):
            x32 = x.float()
            x_fft = torch.fft.rfft2(x32, norm='backward')
            mag = torch.abs(x_fft)
            pha = torch.angle(x_fft)

            gate = self.resonance_gate(mag)
            mag_resonant = mag * (1.0 + gate)

            real_res = mag_resonant * torch.cos(pha)
            imag_res = mag_resonant * torch.sin(pha)

            f_stack = torch.cat([real_res, imag_res], dim=1)
            f_stack = self.freq_conv(f_stack)

            real_final, imag_final = torch.chunk(f_stack, 2, dim=1)
            x_fft_final = torch.complex(real_final, imag_final)

            x_restored = torch.fft.irfft2(x_fft_final, s=(H, W), norm='backward')
        if trace:
            return dict(F_in=x_fft, F_gated=torch.polar(mag_resonant, pha), F_out=x_fft_final, gate=gate)
        return x_restored.to(x.dtype)

    def forward(self, x):
        identity = x
        x_restored = self._spectral(x)
        return identity + self.spatial_restore(x_restored)

    @torch.no_grad()
    def trace(self, x):
        """Intermediate spectra used by the phase-fidelity analysis (no side effect)."""
        return self._spectral(x, trace=True)


class PlainConvBlock(nn.Module):
    """Replacement used by the 'w/o SRB' ablation: the FFT/gate/complex-mixing path
    is replaced by a spatial 1x1 conv, the spatial branch is kept as in the SRB."""

    def __init__(self, dim):
        super(PlainConvBlock, self).__init__()
        self.mix = nn.Conv2d(dim, dim, 1)
        self.spatial_restore = nn.Sequential(
            nn.Conv2d(dim, dim, 3, 1, 1, groups=dim),
            nn.Conv2d(dim, dim, 1)
        )

    def forward(self, x):
        return x + self.spatial_restore(self.mix(x))


# ==========================================
# 3. Frozen semantic feature extractors
# ==========================================

class _FrozenExtractor(nn.Module):
    """Frozen ImageNet backbone returning four feature maps; always in eval mode."""

    def _freeze(self):
        for param in self.parameters():
            param.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.train(False)            # eval mode from construction on (modules are built in training mode)

    def train(self, mode=True):  # BatchNorm of ResNet must never switch to batch statistics
        return super().train(False)


class VGG19SemanticExtractor(_FrozenExtractor):
    """VGG-19 (ImageNet IMAGENET1K_V1), blocks conv1_1..pool1 / conv2_1..pool2 /
    conv3_1..pool3 / conv4_1..pool4 (feature index 0..27) -> 64/128/256/512 channels at
    1/2, 1/4, 1/8, 1/16 of the input resolution."""
    dims = [64, 128, 256, 512]

    def __init__(self, pretrained=True):
        super(VGG19SemanticExtractor, self).__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1 if pretrained else None).features
        self.scale1 = nn.Sequential(*[vgg[i] for i in range(5)])
        self.scale2 = nn.Sequential(*[vgg[i] for i in range(5, 10)])
        self.scale3 = nn.Sequential(*[vgg[i] for i in range(10, 19)])
        self.scale4 = nn.Sequential(*[vgg[i] for i in range(19, 28)])
        self._freeze()

    def forward(self, x):
        x = (x - self.mean) / self.std
        f1 = self.scale1(x)
        f2 = self.scale2(f1)
        f3 = self.scale3(f2)
        f4 = self.scale4(f3)
        return [f1, f2, f3, f4]


class ResNet50SemanticExtractor(_FrozenExtractor):
    """ResNet-50 (ImageNet IMAGENET1K_V1) layer1..layer4 -> 256/512/1024/2048 channels."""
    dims = [256, 512, 1024, 2048]

    def __init__(self, pretrained=True):
        super(ResNet50SemanticExtractor, self).__init__()
        r = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4
        self._freeze()

    def forward(self, x):
        x = self.stem((x - self.mean) / self.std)
        f1 = self.layer1(x)
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        return [f1, f2, f3, f4]


class ConvNeXtTSemanticExtractor(_FrozenExtractor):
    """ConvNeXt-T (ImageNet IMAGENET1K_V1) four stages -> 96/192/384/768 channels."""
    dims = [96, 192, 384, 768]

    def __init__(self, pretrained=True):
        super(ConvNeXtTSemanticExtractor, self).__init__()
        c = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1 if pretrained else None)
        self.features = c.features
        self._freeze()

    def forward(self, x):
        x = (x - self.mean) / self.std
        feats = []
        for i, blk in enumerate(self.features):
            x = blk(x)
            if i in (1, 3, 5, 7):
                feats.append(x)
        return feats


SEMANTIC_EXTRACTORS = {
    'vgg19': VGG19SemanticExtractor,
    'resnet50': ResNet50SemanticExtractor,
    'convnext_t': ConvNeXtTSemanticExtractor,
}


# ==========================================
# 4. Semantic Injection Module
# ==========================================

class SemanticInjection(nn.Module):
    def __init__(self, restore_dim, semantic_dim):
        super(SemanticInjection, self).__init__()
        self.semantic_proj = nn.Conv2d(semantic_dim, restore_dim, 1)
        self.channel_attn = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(restore_dim, restore_dim // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(restore_dim // 4, restore_dim, 1),
            nn.Sigmoid()
        )
        self.spatial_attn = nn.Sequential(
            nn.Conv2d(restore_dim, 1, 7, 1, 3),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(restore_dim * 2 + semantic_dim, restore_dim, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(restore_dim, restore_dim, 3, 1, 1)
        )

    def forward(self, restore_feat, semantic_feat):
        B, C, H, W = restore_feat.shape
        identity = restore_feat

        sem_proj = self.semantic_proj(semantic_feat)
        if sem_proj.shape[2:] != restore_feat.shape[2:]:
            sem_proj = torch.nn.functional.interpolate(sem_proj, size=(H, W), mode='bilinear', align_corners=False)
            semantic_feat = torch.nn.functional.interpolate(semantic_feat, size=(H, W), mode='bilinear', align_corners=False)

        ch_attn = self.channel_attn(sem_proj)
        restore_ch = restore_feat * ch_attn

        sp_attn = self.spatial_attn(sem_proj)
        restore_sp = restore_feat * sp_attn

        fused = self.fusion(torch.cat([semantic_feat, restore_ch, restore_sp], dim=1))
        return identity + fused


# ==========================================
# 5. Encoder / Decoder blocks
# ==========================================

def _make_srb(dim, use_srb, replacement):
    if use_srb:
        return SpectralResonanceBlock(dim)
    if replacement == 'plain_conv':
        return PlainConvBlock(dim)
    return nn.Identity()


def _make_scwam(dim, num_heads, opts):
    if opts.get('use_scwam', True):
        return SCWAMLayer(dim, num_heads=num_heads, window_size=opts.get('window_size', 8),
                          shift_size=opts.get('shift_size', 4), use_meshgrid=opts.get('use_meshgrid', True),
                          masked_shift=opts.get('masked_shift', False), rel_pos_bias=opts.get('rel_pos_bias', False))
    if opts.get('scwam_replacement', 'conv') == 'conv':
        return LocalConvBlock(dim)
    return nn.Identity()


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels, semantic_dim, num_heads=8, use_semantic=True, opts=None):
        super(EncoderBlock, self).__init__()
        opts = opts or {}
        self.use_semantic = use_semantic
        self.downsample = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.srb = _make_srb(out_channels, opts.get('use_srb', True), opts.get('srb_replacement', 'plain_conv'))
        self.scwam = _make_scwam(out_channels, num_heads, opts)
        if use_semantic:
            self.sim = SemanticInjection(out_channels, semantic_dim)

    def forward(self, x, semantic_feat=None):
        x = self.downsample(x)
        x = self.srb(x)
        if self.use_semantic and semantic_feat is not None:
            x = self.sim(x, semantic_feat)
        x = self.scwam(x)
        return x


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels, semantic_dim, prompt_dim=256, num_heads=8,
                 use_semantic=True, use_prompt=True, opts=None):
        super(DecoderBlock, self).__init__()
        opts = opts or {}
        self.use_semantic = use_semantic
        self.use_prompt = use_prompt
        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.fusion = nn.Sequential(
            nn.Conv2d(out_channels + skip_channels, out_channels, 1),
            nn.LeakyReLU(0.2, inplace=True)
        )
        self.srb = _make_srb(out_channels, opts.get('use_srb', True), opts.get('srb_replacement', 'plain_conv'))
        self.scwam = _make_scwam(out_channels, num_heads, opts)
        if use_prompt:
            self.sft = SFTLayer(out_channels, prompt_dim)
        if use_semantic:
            self.sim = SemanticInjection(out_channels, semantic_dim)

    def forward(self, x, skip, prompt_emb, semantic_feat=None):
        x = self.upsample(x)
        if x.size(2) != skip.size(2) or x.size(3) != skip.size(3):
            x = torch.nn.functional.interpolate(x, size=(skip.size(2), skip.size(3)), mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.fusion(x)
        x = self.srb(x)
        if self.use_semantic and semantic_feat is not None:
            x = self.sim(x, semantic_feat)
        x = self.scwam(x)
        if self.use_prompt and prompt_emb is not None:
            x = self.sft(x, prompt_emb)
        return x


class SFTLayer(nn.Module):
    """Channel-wise affine modulation by the 256-D visual descriptor p:
       y = x * (1 + gamma(p)) + beta(p).  gamma, beta are 1x1 convs applied to p broadcast
       over H x W, i.e. spatially constant (channel-wise SFT)."""

    def __init__(self, in_channels, prompt_dim=256):
        super(SFTLayer, self).__init__()
        self.sft_scale = nn.Conv2d(prompt_dim, in_channels, 1)
        self.sft_shift = nn.Conv2d(prompt_dim, in_channels, 1)

    def forward(self, x, prompt_emb):
        # gamma(p), beta(p) are spatially constant: the 1x1 convs are applied once to the B x 256 x 1 x 1
        # descriptor and broadcast (identical result to applying them on the H x W broadcast descriptor)
        prompt = prompt_emb.view(prompt_emb.size(0), prompt_emb.size(1), 1, 1).to(x.dtype)
        scale = self.sft_scale(prompt)
        shift = self.sft_shift(prompt)
        return x * (scale + 1) + shift


class GlobalContextBlock(nn.Module):
    def __init__(self, in_channels):
        super(GlobalContextBlock, self).__init__()
        self.conv_mask = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.softmax = nn.Softmax(dim=2)
        self.channel_add_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // 2, kernel_size=1),
            nn.LayerNorm([in_channels // 2, 1, 1]),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // 2, in_channels, kernel_size=1)
        )

    def forward(self, x):
        b, c, h, w = x.size()
        input_x = x
        input_x = input_x.view(b, c, h * w).unsqueeze(1)
        mask = self.conv_mask(x).view(b, 1, h * w)
        mask = self.softmax(mask).unsqueeze(-1)
        context = torch.matmul(input_x, mask)
        context = context.view(b, c, 1, 1)
        return x + self.channel_add_conv(context)


class PromptLearner(nn.Module):
    """Visual degradation descriptor p (256-D) computed from the degraded image by a
    small CNN (no text, no degradation label at inference)."""

    def __init__(self, in_channels=3, prompt_dim=256):
        super(PromptLearner, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Sequential(
            nn.Linear(256, prompt_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(prompt_dim, prompt_dim)
        )

    def forward(self, x):
        feat = self.features(x)
        feat = feat.view(feat.size(0), -1)
        return self.fc(feat)


# ==========================================
# 6. KASIRNet Definition
# ==========================================

MODEL_DEFAULTS = dict(
    in_channels=3, out_channels=3, base_channels=64, prompt_dim=256,
    use_sim=True, extractor='vgg19', extractor_pretrained=True,
    use_srb=True, srb_replacement='plain_conv',
    use_scwam=True, scwam_replacement='conv', use_meshgrid=True,
    window_size=8, shift_size=4, masked_shift=False, rel_pos_bias=False,
    use_prompt=True, use_gcb=True, num_scales=4,
)


class KASIRNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, base_channels=64, prompt_dim=256, **flags):
        super(KASIRNet, self).__init__()
        unknown = set(flags) - set(MODEL_DEFAULTS)
        if unknown:
            raise ValueError(f'Unknown KASIRNet flags: {sorted(unknown)}')
        o = dict(MODEL_DEFAULTS, in_channels=in_channels, out_channels=out_channels,
                 base_channels=base_channels, prompt_dim=prompt_dim, **flags)
        self.flags = o
        self.use_sim, self.use_prompt, self.num_scales = o['use_sim'], o['use_prompt'], o['num_scales']
        assert self.num_scales in (3, 4), 'num_scales must be 3 or 4'
        self.prompt_dim = prompt_dim
        # input padding: a multiple of window_size x 2^num_scales, so that no SCWAM layer pads internally
        # (internal zero padding is normalised to the LayerNorm bias and the cyclic shift wraps into it)
        self.pad_multiple = o.get('window_size', 8) * 2 ** self.num_scales

        if self.use_sim:
            ext = SEMANTIC_EXTRACTORS[o['extractor']](pretrained=o['extractor_pretrained'])
            sem_dims = ext.dims
            if o['extractor'] == 'vgg19':
                self.vgg_extractor = ext          # parameter names of the submitted model
            else:
                self.semantic_extractor = ext
        else:
            sem_dims = [0, 0, 0, 0]
        ch = [base_channels, base_channels*2, base_channels*4, base_channels*6, base_channels*8]
        if self.use_prompt:
            self.prompt_learner = PromptLearner(in_channels, prompt_dim)
        us, up = self.use_sim, self.use_prompt

        self.head_conv = nn.Sequential(
            nn.Conv2d(in_channels, ch[0], 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch[0], ch[0], 3, 1, 1)
        )
        self.head_attn = _make_scwam(ch[0], 2, o)

        self.encoder1 = EncoderBlock(ch[0], ch[1], sem_dims[0], num_heads=4, use_semantic=us, opts=o)
        self.encoder2 = EncoderBlock(ch[1], ch[2], sem_dims[1], num_heads=4, use_semantic=us, opts=o)
        self.encoder3 = EncoderBlock(ch[2], ch[3], sem_dims[2], num_heads=8, use_semantic=us, opts=o)
        if self.num_scales == 4:
            self.encoder4 = EncoderBlock(ch[3], ch[4], sem_dims[3], num_heads=8, use_semantic=us, opts=o)
        cb = ch[4] if self.num_scales == 4 else ch[3]

        self.bottleneck = nn.Sequential(
            GlobalContextBlock(cb) if o['use_gcb'] else nn.Identity(),
            _make_srb(cb, o['use_srb'], o['srb_replacement']),
            _make_scwam(cb, 8, o)
        )

        if self.num_scales == 4:
            self.decoder4 = DecoderBlock(ch[4], ch[3], ch[3], sem_dims[3], prompt_dim, num_heads=8, use_semantic=us, use_prompt=up, opts=o)
        self.decoder3 = DecoderBlock(ch[3], ch[2], ch[2], sem_dims[2], prompt_dim, num_heads=8, use_semantic=us, use_prompt=up, opts=o)
        self.decoder2 = DecoderBlock(ch[2], ch[1], ch[1], sem_dims[1], prompt_dim, num_heads=4, use_semantic=us, use_prompt=up, opts=o)
        self.decoder1 = DecoderBlock(ch[1], ch[0], ch[0], sem_dims[0], prompt_dim, num_heads=4, use_semantic=us, use_prompt=up, opts=o)

        self.tail = nn.Sequential(
            nn.Conv2d(ch[0], ch[0], 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ch[0], out_channels, 3, 1, 1)
        )

    @property
    def extractor_module(self):
        return getattr(self, 'vgg_extractor', None) or getattr(self, 'semantic_extractor', None)

    def check_image_size(self, x):
        _, _, h, w = x.size()
        m = getattr(self, 'pad_multiple', 16)
        mod_pad_h = (m - h % m) % m
        mod_pad_w = (m - w % m) % m
        x = torch.nn.functional.pad(x, (0, mod_pad_w, 0, mod_pad_h), mode='replicate')
        return x, h, w

    def forward(self, x, prompt_override=None):
        """Returns (restored image, 256-D descriptor p).  `prompt_override` replaces p
        (used only by the mixed-degradation control experiment); p is None when the
        Prompt Learner is ablated."""
        x_padded, H_orig, W_orig = self.check_image_size(x)
        if self.use_sim:
            with torch.no_grad():
                sem = self.extractor_module(x_padded)
        else:
            sem = [None, None, None, None]

        if prompt_override is not None:
            prompt_emb = prompt_override
        elif self.use_prompt:
            prompt_emb = self.prompt_learner(x_padded)
        else:
            prompt_emb = None

        x0 = self.head_conv(x_padded)
        x0 = self.head_attn(x0)

        x1 = self.encoder1(x0, sem[0])
        x2 = self.encoder2(x1, sem[1])
        x3 = self.encoder3(x2, sem[2])
        if self.num_scales == 4:
            x4 = self.encoder4(x3, sem[3])
            x_mid = self.bottleneck(x4)
            d4 = self.decoder4(x_mid, x3, prompt_emb, sem[3])
        else:
            d4 = self.bottleneck(x3)
        d3 = self.decoder3(d4, x2, prompt_emb, sem[2])
        d2 = self.decoder2(d3, x1, prompt_emb, sem[1])
        d1 = self.decoder1(d2, x0, prompt_emb, sem[0])

        out = self.tail(d1)
        out = out + x_padded
        out = out[:, :, :H_orig, :W_orig]
        if not self.training:   # clamping during training would zero the gradient of every overshooting pixel
            out = out.clamp(0, 1)

        return out, prompt_emb


# ==========================================
# 7. Parameter accounting (Table 10)
# ==========================================

def component_parameter_table(model):
    """Trainable parameters per component, frozen extractor parameters (millions)."""
    groups = {'SCWAM (incl. Meshgrid)': (SCWAMLayer, LocalConvBlock), 'SIM': (SemanticInjection,),
              'SRB': (SpectralResonanceBlock, PlainConvBlock), 'Prompt Learner': (PromptLearner,),
              'SFT': (SFTLayer,), 'GCB': (GlobalContextBlock,)}
    seen, out = set(), {}
    for gname, types in groups.items():
        n = 0
        for m in model.modules():
            if isinstance(m, types):
                for p in m.parameters():
                    if id(p) not in seen and p.requires_grad:
                        seen.add(id(p)); n += p.numel()
        out[gname] = n / 1e6
    other = sum(p.numel() for p in model.parameters() if p.requires_grad and id(p) not in seen)
    out['head/tail/sampling/fusion'] = other / 1e6
    out['trainable total'] = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    out['frozen extractor'] = sum(p.numel() for p in model.parameters() if not p.requires_grad) / 1e6
    out['loaded at inference'] = out['trainable total'] + out['frozen extractor']
    return out

In [ ]:
# ===== Loss.py (identical to Loss.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - six-term Omni-Restoration loss (Section 3.2)
#   L = w_pix L_pix + w_fft L_fft + w_perc L_perc + w_ssim L_ssim + w_id L_id + w_edge L_edge
#   defaults (paper): 5.0, 0.5, 1.0, 1.0, 1.0, 0.5
#   L_pix  : L1(pred, gt)
#   L_fft  : L1(|rFFT2(pred)|, |rFFT2(gt)|), orthonormal FFT (norm='ortho': the value does not grow with
#            the patch size; with the unnormalised FFT its gradient was 26x the pixel gradient at 256 px)
#   L_perc : sum_k 2^-(k-1) L1(phi_k(pred), phi_k(gt)), phi_k = frozen ImageNet VGG-19 blocks
#            ending at pool1..pool4 (same cut points as the SIM extractor), k = 1..4
#   L_ssim : 1 - SSIM_Y(pred, gt) (Gaussian 11x11, sigma 1.5; metrics.ssim_y, differentiable)
#   L_id   : L1 between the 2048-D global features of a frozen ImageNet ResNet-50 (no classifier)
#   L_edge : L1 between Sobel gradient magnitudes sqrt(gx^2 + gy^2 + 1e-6)
# Frozen networks are ImageNet-normalised, in eval mode from construction on (the ResNet-50
# BatchNorm used batch statistics in the first revision code), used only at training time.
# A term whose weight is 0 is not computed.
# Calibrated weights (Config.LOSS_CALIBRATION): every term k is multiplied by a fixed scale
# s_k = g_pix / g_k, g_k = mean L2 norm of dL_k/d(pred) at pred = degraded input on the held-out
# validation patches (model-independent, the same for every run).  The paper weights w_k then
# set the share of the gradient each term contributes (pix 5/9 = 56 %) instead of being
# overridden by the raw scales of the terms (uncalibrated, the pixel term had ~1 % of the
# gradient and the identity term 37-91 %).
# ==========================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


_MEAN = (0.485, 0.456, 0.406)
_STD = (0.229, 0.224, 0.225)


class _Frozen(nn.Module):
    def _finish(self):
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor(_MEAN).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(_STD).view(1, 3, 1, 1))
        self.train(False)       # BatchNorm statistics frozen from construction on

    def train(self, mode=True):
        return super().train(False)


class VGGFeatureExtractor(_Frozen):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice1 = nn.Sequential(*[vgg[i] for i in range(5)])
        self.slice2 = nn.Sequential(*[vgg[i] for i in range(5, 10)])
        self.slice3 = nn.Sequential(*[vgg[i] for i in range(10, 19)])
        self.slice4 = nn.Sequential(*[vgg[i] for i in range(19, 28)])
        self._finish()

    def forward(self, x):
        h1 = self.slice1((x - self.mean) / self.std)
        h2 = self.slice2(h1)
        h3 = self.slice3(h2)
        h4 = self.slice4(h3)
        return [h1, h2, h3, h4]


class IdentityFeature(_Frozen):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        self._finish()

    def forward(self, x):
        return self.backbone((x - self.mean) / self.std).flatten(1)


class OmniRestorationLoss(nn.Module):
    TERMS = ('pix', 'fft', 'perc', 'ssim', 'id', 'edge')

    def __init__(self, weights, amp=False, amp_dtype=torch.float16, scales=None):
        super().__init__()
        self.w = {k: float(weights.get(k, 0.0)) for k in self.TERMS}
        self.s = {k: float((scales or {}).get(k, 1.0)) for k in self.TERMS}
        self.amp, self.amp_dtype = amp, amp_dtype
        self.vgg = VGGFeatureExtractor() if self.w['perc'] > 0 else None
        self.idnet = IdentityFeature() if self.w['id'] > 0 else None
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1))

    def compute_edge(self, x):
        ex = F.conv2d(x, self.sobel_x, padding=1, groups=3)
        ey = F.conv2d(x, self.sobel_y, padding=1, groups=3)
        return torch.sqrt(ex ** 2 + ey ** 2 + 1e-6)

    def terms(self, preds, targets, only=None):
        """Unweighted terms (those with a non-zero weight, or those in `only`)."""
        preds, targets = preds.float(), targets.float()
        on = (lambda k: k in only) if only is not None else (lambda k: self.w[k] > 0)
        terms = {}
        if on('pix'):
            terms['pix'] = F.l1_loss(preds, targets)
        if on('fft'):
            terms['fft'] = F.l1_loss(torch.abs(torch.fft.rfft2(preds, norm='ortho')),
                                     torch.abs(torch.fft.rfft2(targets, norm='ortho')))
        dev = preds.device.type
        with torch.autocast(device_type=dev, dtype=self.amp_dtype, enabled=self.amp):
            if self.vgg is not None and on('perc'):
                with torch.no_grad():
                    tf = self.vgg(targets)
                pf = self.vgg(preds)
                terms['perc'] = sum(F.l1_loss(a.float(), b.float()) * (1.0 / (2 ** i)) for i, (a, b) in enumerate(zip(pf, tf)))
            if self.idnet is not None and on('id'):
                with torch.no_grad():
                    t_id = self.idnet(targets)
                terms['id'] = F.l1_loss(self.idnet(preds).float(), t_id.float())
        if on('ssim'):
            terms['ssim'] = 1.0 - ssim_y(preds, targets).mean()
        if on('edge'):
            terms['edge'] = F.l1_loss(self.compute_edge(preds), self.compute_edge(targets))
        return terms

    def forward(self, preds, targets):
        terms = self.terms(preds, targets)
        total = sum(self.w[k] * self.s[k] * v for k, v in terms.items())
        logs = {k: float(v.detach()) for k, v in terms.items()}
        logs['total'] = float(total.detach())
        return total, logs

In [ ]:
# ===== Datasets.py (identical to Datasets.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - data: sources, pairing, leakage audit, training pool, test sets
# --------------------------------------------------------------------------
# * Pairs are matched by an explicit key (file stem or a regex), never by the
#   position in two independently sorted lists (sorted-order zipping mis-paired
#   all 1800 RainTrainH pairs, 'norain-10.png' <-> 'norain-1001x2.png').
# * A training pair whose clean image is a near-duplicate of a benchmark clean
#   image is removed (LeakageAuditor); the audit is written to data_manifest/.
# * The training pool (seeded subset, held-out validation split) is computed once
#   and frozen in data_manifest/train_manifest.json so that every run of the
#   revision sees exactly the same images.
# ==========================================================================
import os
import io
import math
import re
import glob
import json
import zlib
import random
import hashlib
import tarfile
import zipfile
import time
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image, ImageFile
import torchvision.transforms.functional as TF


ImageFile.LOAD_TRUNCATED_IMAGES = True
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')


# ==========================================
# 1. Sources: kagglehub / Hugging Face / Google Drive, and the Drive cache
# ==========================================

def _extract(archive, dest, stream=False):
    if archive.endswith('.zip'):
        with zipfile.ZipFile(archive) as z:
            z.extractall(dest)
        return
    with tarfile.open(archive, 'r|*' if stream else 'r:*') as t:
        try:
            t.extractall(dest, filter='data')
        except TypeError:          # Python < 3.12
            t.extractall(dest)


def _done(dest):
    return os.path.join(dest, '.extracted')


def _download_gdown(cfg, key):
    src = cfg.GDOWN_SOURCES[key] if key in cfg.GDOWN_SOURCES else key
    dest = os.path.join(cfg.DATA_ROOT, src['file'].split('.')[0] if isinstance(key, dict) else key)
    if os.path.exists(_done(dest)):
        return dest
    os.makedirs(dest, exist_ok=True)
    manual = os.path.join(cfg.DATA_CACHE_ROOT, 'archives', src['file'])
    archive = manual if os.path.exists(manual) else os.path.join(cfg.DATA_ROOT, src['file'])
    if not os.path.exists(archive):
        import gdown
        out = gdown.download(id=src['id'], output=archive, quiet=False)
        if out is None or not os.path.exists(archive):
            raise RuntimeError(f'Google-Drive download of {src["file"]} failed (often "quota exceeded", resets within 24 h). '
                               f'Either run again later, or open https://drive.google.com/file/d/{src["id"]} in the browser, '
                               f'download it and upload it to {manual}')
    print(f'[data] extracting {archive} -> {dest}')
    _extract(archive, dest, stream=not archive.endswith('.zip'))
    open(_done(dest), 'w').write('ok')
    if archive != manual:
        os.remove(archive)
    return dest


def _download_hf_parquet(cfg, key):
    """Hugging Face parquet mirror -> image files <dir>/<stem of key column>.<ext> (same stem in every column)."""
    spec = cfg.HF_PARQUET_SOURCES[key]
    dest = os.path.join(cfg.DATA_ROOT, key)
    if os.path.exists(_done(dest)):
        return dest
    import urllib.request
    import pyarrow.parquet as pq
    os.makedirs(dest, exist_ok=True)
    n = 0
    for f in spec['files']:
        local = os.path.join(cfg.DATA_ROOT, key + '_' + os.path.basename(f))
        if not os.path.exists(local):
            url = f'https://huggingface.co/datasets/{spec["repo"]}/resolve/main/{f}'
            print(f'[data] downloading {url}')
            urllib.request.urlretrieve(url, local + '.part')
            os.replace(local + '.part', local)
        table = pq.read_table(local)
        for row in table.to_pylist():
            stem = os.path.splitext(os.path.basename(row[spec['key_column']]['path']))[0]
            for col, sub in spec['columns'].items():
                ext = os.path.splitext(row[col]['path'])[1] or '.png'
                out = os.path.join(dest, sub, stem + ext)
                os.makedirs(os.path.dirname(out), exist_ok=True)
                with open(out, 'wb') as fh:
                    fh.write(row[col]['bytes'])
            n += 1
        os.remove(local)
    print(f'[data] {key}: {n} pairs written to {dest}')
    open(_done(dest), 'w').write('ok')
    return dest


def _download_hf_zip(cfg, key):
    spec = cfg.HF_ZIP_SOURCES[key]
    dest = os.path.join(cfg.DATA_ROOT, key)
    if os.path.exists(_done(dest)):
        return dest
    import urllib.request
    os.makedirs(dest, exist_ok=True)
    local = os.path.join(cfg.DATA_ROOT, key + '_' + spec['file'])
    url = f'https://huggingface.co/datasets/{spec["repo"]}/resolve/main/{spec["file"]}'
    print(f'[data] downloading {url}')
    urllib.request.urlretrieve(url, local + '.part')
    os.replace(local + '.part', local)
    _extract(local, dest)
    os.remove(local)
    open(_done(dest), 'w').write('ok')
    return dest


class _HTTPRangeFile(io.RawIOBase):
    """Seekable read-only view of a remote file through HTTP range requests (zipfile reads the central
    directory and single members of a 100 GB archive without downloading it)."""

    def __init__(self, url):
        import urllib.request
        self._ur = urllib.request
        r = urllib.request.urlopen(urllib.request.Request(url, method='HEAD', headers={'User-Agent': 'kasirnet'}), timeout=120)
        self.url, self.size, self.pos = r.geturl(), int(r.headers['Content-Length']), 0

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, o, whence=0):
        self.pos = o if whence == 0 else self.pos + o if whence == 1 else self.size + o
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else n
        if n == 0 or self.pos >= self.size:
            return b''
        end = min(self.pos + n, self.size) - 1
        for attempt in range(5):
            try:
                req = self._ur.Request(self.url, headers={'Range': f'bytes={self.pos}-{end}', 'User-Agent': 'kasirnet'})
                b = self._ur.urlopen(req, timeout=300).read()
                break
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** attempt)
        self.pos += len(b)
        return b

    def readinto(self, buf):
        d = self.read(len(buf))
        buf[:len(d)] = d
        return len(d)


def _download_hf_remote_zip(cfg, key):
    """A seeded subset of the pairs listed in an official split file, extracted member by member from a
    remote zip (Hugging Face) with HTTP range requests (e.g. 1,000 RSBlur training pairs out of 115 GB)."""
    import urllib.request
    import zipfile
    from concurrent.futures import ThreadPoolExecutor
    spec = cfg.HF_REMOTE_ZIP_SOURCES[key]
    dest = os.path.join(cfg.DATA_ROOT, key)
    if os.path.exists(_done(dest)):
        return dest
    lines = urllib.request.urlopen(spec['list_url'], timeout=120).read().decode().split('\n')
    pairs = sorted({tuple(ln.split()) for ln in lines if len(ln.split()) == 2})
    pairs = random.Random(spec.get('seed', 2026)).sample(pairs, min(spec['n_pairs'], len(pairs)))
    members = sorted({m for p in pairs for m in p})
    url = f'https://huggingface.co/datasets/{spec["repo"]}/resolve/main/{spec["file"]}'
    print(f'[data] {key}: {len(pairs)} pairs ({len(members)} files) from {url} by range requests')

    def worker(chunk):
        z = zipfile.ZipFile(io.BufferedReader(_HTTPRangeFile(url), buffer_size=1 << 22))
        for m in chunk:
            out = os.path.join(dest, m)
            if os.path.exists(out):
                continue
            os.makedirs(os.path.dirname(out), exist_ok=True)
            with open(out + '.part', 'wb') as fh:
                fh.write(z.read(m))
            os.replace(out + '.part', out)
        return len(chunk)

    nw = spec.get('workers', 16)
    with ThreadPoolExecutor(nw) as ex:
        done = sum(ex.map(worker, [members[i::nw] for i in range(nw)]))
    print(f'[data] {key}: {done} files written to {dest}')
    open(_done(dest), 'w').write('ok')
    return dest


def download_source(cfg, key):
    if key in cfg.HF_ZIP_SOURCES:
        return _download_hf_zip(cfg, key)
    if key in getattr(cfg, 'HF_REMOTE_ZIP_SOURCES', {}):
        return _download_hf_remote_zip(cfg, key)
    if key in cfg.KAGGLE_SOURCES:
        import kagglehub
        return kagglehub.dataset_download(cfg.KAGGLE_SOURCES[key])
    if key in cfg.GDOWN_SOURCES:
        return _download_gdown(cfg, key)
    if key in cfg.HF_PARQUET_SOURCES:
        try:
            return _download_hf_parquet(cfg, key)
        except Exception as e:
            if key == 'dpdd_test':   # official archive (zip with test/inputC, test/target)
                print(f'[data] Hugging Face mirror failed ({e}); trying the official Google-Drive archive')
                return _download_gdown(cfg, dict(id=cfg.DPDD_GDOWN_ID, file='dpdd_test.zip'))
            raise
    raise KeyError(f'Unknown data source {key}')


class DatasetCache:
    """One uncompressed tar per source on Google Drive (DATA_CACHE_ROOT/<key>.tar + <key>.json).
    'used' tars hold only the files read by the experiments; they are never used to rebuild the
    training manifest (need_full=True), which requires the complete source."""

    def __init__(self, cfg):
        self.cfg, self.mode, self.root = cfg, cfg.DATA_CACHE_MODE, cfg.DATA_CACHE_ROOT
        self.local = os.path.join(cfg.DATA_ROOT, 'from_drive_cache')

    def _tar(self, key):
        return os.path.join(self.root, f'{key}.tar')

    def info(self, key):
        p = os.path.join(self.root, f'{key}.json')
        if os.path.exists(p) and os.path.exists(self._tar(key)):
            return json.load(open(p))
        return None

    def restore(self, key, need_full=False):
        if self.mode == 'off' or self.cfg.DATA_CACHE_REFRESH:
            return None
        m = self.info(key)
        if m is None or (need_full and m['mode'] != 'full'):
            return None
        if os.path.getsize(self._tar(key)) < m['bytes']:
            print(f'[cache] !! {key}: the tar on Drive is incomplete (upload interrupted or Drive full) -> downloaded again')
            return None
        dest = os.path.join(self.local, key)
        marker = os.path.join(dest, '.cache_meta.json')
        if os.path.exists(marker) and json.load(open(marker)).get('created') == m['created']:
            return dest
        import shutil
        shutil.rmtree(dest, ignore_errors=True)
        os.makedirs(dest)
        print(f'[cache] restoring {key} ({m["n_files"]} files, {m["bytes"] / 1e9:.2f} GB, mode {m["mode"]}) from Google Drive')
        _extract(self._tar(key), dest, stream=True)
        json.dump(m, open(marker, 'w'))
        return dest

    def wanted(self, key):
        if self.mode == 'off':
            return False
        if getattr(self.cfg, 'DATA_CACHE_SOURCES', 'fragile') == 'all':
            return True
        return (key in self.cfg.GDOWN_SOURCES or key in self.cfg.HF_PARQUET_SOURCES or key in self.cfg.HF_ZIP_SOURCES
                or key in getattr(self.cfg, 'HF_REMOTE_ZIP_SOURCES', {}))

    def inventory(self, keys):
        """Print what the Drive cache holds (helps to see a cache lost to a full Drive)."""
        import shutil
        try:
            free = shutil.disk_usage(self.root if os.path.isdir(self.root) else os.path.dirname(self.root)).free / 1e9
        except OSError:
            free = float('nan')
        print(f'[cache] {self.root}  (free space reported for this Drive: {free:.1f} GB)')
        for k in sorted(set(keys)):
            m, t = self.info(k), self._tar(k)
            state = (f'cached, {m["mode"]}, {m["n_files"]} files, {m["bytes"] / 1e9:.2f} GB' if m else
                     ('tar without metadata (incomplete upload?)' if os.path.exists(t) else
                      ('not cached' if self.wanted(k) else 'not cached (Kaggle source, downloaded each session)')))
            if m and os.path.getsize(t) < m['bytes']:
                state += f'  !! tar is only {os.path.getsize(t) / 1e9:.2f} GB: incomplete, will be rebuilt'
            print(f'          {k:16s} {state}')

    def store(self, key, root, relfiles=None):
        """relfiles=None or mode 'full' -> the whole source.  Never overwrites an existing cache
        unless DATA_CACHE_REFRESH (a 'full' cache is never replaced by a 'used' one)."""
        if not self.wanted(key) or root.startswith(self.local):
            return
        m = self.info(key)
        if m is not None and not self.cfg.DATA_CACHE_REFRESH:
            return
        if self.mode == 'full' or relfiles is None:
            files, mode = [], 'full'
            for dp, _, fs in os.walk(root):
                files += [os.path.relpath(os.path.join(dp, f), root) for f in fs if not f.startswith('.cache_meta')]
        else:
            files, mode = list(relfiles), 'used'
        files = sorted(set(files))
        os.makedirs(self.root, exist_ok=True)
        need = sum(os.path.getsize(os.path.join(root, f)) for f in files)
        try:
            import shutil
            free = shutil.disk_usage(self.root).free
            if free < need * 1.05:
                print(f'[cache] !! {key}: needs {need / 1e9:.2f} GB but the Drive has {free / 1e9:.2f} GB free -> not cached')
                return
        except OSError:
            pass
        tmp = self._tar(key) + '.tmp'
        total = 0
        try:
            with tarfile.open(tmp, 'w') as t:
                for rel in files:
                    p = os.path.join(root, rel)
                    t.add(p, arcname=rel, recursive=False)
                    total += os.path.getsize(p)
            os.replace(tmp, self._tar(key))
        except OSError as e:
            print(f'[cache] !! could not cache {key} on Drive ({e}); it will be downloaded again next session')
            if os.path.exists(tmp):
                os.remove(tmp)
            return
        meta = dict(key=key, mode=mode, n_files=len(files), bytes=total, created=time.time(), source_root=root)
        with open(os.path.join(self.root, f'{key}.json'), 'w') as f:
            json.dump(meta, f, indent=1)
        print(f'[cache] {key}: {len(files)} files ({total / 1e9:.2f} GB, {mode}) cached in {self.root}')


def resolve_sources(cfg, keys, cache=None, need_full=False):
    """{source key: local root}.  The Drive cache is used first, then the original download."""
    roots = {}
    for key in sorted(set(keys)):
        r = cache.restore(key, need_full) if cache is not None else None
        roots[key] = r or download_source(cfg, key)
    return roots


def spec_sources(spec, benchmarks=None):
    keys = []
    for f in ('input', 'gt', 'root', 'clean', 'lmdb', 'lmdb_gt'):
        if f in spec:
            keys.append(spec[f][0])
    if 'base' in spec and benchmarks is not None:
        keys += spec_sources(benchmarks[spec['base']], benchmarks)
    return keys


def resolve_path(roots, ref, must_exist=True):
    """ref = (source key, relative path); a relative path containing '*' is searched
    recursively and the first (sorted) matching directory is returned."""
    key, rel = ref
    if key not in roots:
        raise FileNotFoundError(f'data source {key!r} is not available (download failed or not requested)')
    root = roots[key]
    if '*' in rel:
        hits = sorted(h for h in glob.glob(os.path.join(root, rel), recursive=True) if os.path.isdir(h))
        hits = [h for h in hits if '__MACOSX' not in h]
        if not hits:
            raise FileNotFoundError(f'No directory matches {rel} under {root}')
        return hits[0]
    p = os.path.normpath(os.path.join(root, rel))
    if must_exist and not os.path.exists(p):
        raise FileNotFoundError(f'{p} does not exist (source {key!r})')
    return p


def list_images(d, recursive=False):
    if recursive:
        files = glob.glob(os.path.join(d, '**', '*'), recursive=True)
    else:
        files = glob.glob(os.path.join(d, '*'))
    return sorted(f for f in files if f.lower().endswith(IMG_EXT) and os.path.isfile(f))


# ==========================================
# 2. Pairing
# ==========================================

def _key(path, rx):
    stem = os.path.splitext(os.path.basename(path))[0]
    if rx is None:
        return stem.lower()
    m = re.search(rx, stem)
    return None if m is None else '_'.join(m.groups()).lower()


def pair_files(input_dir, gt_dir, match='stem', include=None):
    """Return (pairs, report).  pairs = [(input_path, gt_path, key)].  A missing directory or an
    empty one is an error (the submitted pool silently skipped empty folders)."""
    for d in (input_dir, gt_dir):
        if not os.path.isdir(d):
            raise FileNotFoundError(f'directory {d} does not exist')
    rx_in = rx_gt = None
    if isinstance(match, (tuple, list)) and match[0] == 'regex':
        rx_in, rx_gt = match[1], match[2]
    ins, gts = list_images(input_dir), list_images(gt_dir)
    gt_map, n_gt_keys = {}, 0
    for g in gts:
        k = _key(g, rx_gt)
        if k is not None:
            n_gt_keys += 1
            gt_map.setdefault(k, g)
    pairs, unmatched, n_in_keys = [], [], 0
    for f in ins:
        k = _key(f, rx_in)
        if k is None:
            continue
        if include and not re.search(include, os.path.basename(f)):
            continue
        n_in_keys += 1
        if k in gt_map and os.path.abspath(gt_map[k]) != os.path.abspath(f):
            pairs.append((f, gt_map[k], k))
        else:
            unmatched.append(os.path.basename(f))
    report = dict(input_dir=input_dir, gt_dir=gt_dir, n_input_files=len(ins), n_gt_files=len(gts),
                  n_input_keys=n_in_keys, n_gt_keys=n_gt_keys, n_pairs=len(pairs), n_unmatched=len(unmatched),
                  unmatched_examples=unmatched[:5])
    if not pairs:
        raise FileNotFoundError(f'no pairs found between {input_dir} and {gt_dir} (match={match})')
    return pairs, report


def pair_by_replace(root, input_glob, gt_replace):
    if not os.path.isdir(root):
        raise FileNotFoundError(f'directory {root} does not exist')
    """gt_replace: (old, new) replaced in the file name | ('path', old, new) replaced in the path relative to
    root (key = that relative path) | ('sibling', dir) same file name in the sibling folder `dir` (inputs in
    `dir` itself are skipped; key = '<input folder>/<stem>')."""
    ins = sorted(glob.glob(os.path.join(root, input_glob), recursive=True))
    pairs, missing = [], 0
    for f in ins:
        rel = os.path.relpath(f, root)
        if gt_replace[0] == 'path':
            g, key = os.path.join(root, rel.replace(gt_replace[1], gt_replace[2])), os.path.splitext(rel)[0]
        elif gt_replace[0] == 'sibling':
            parent = os.path.basename(os.path.dirname(f))
            if parent == gt_replace[1]:
                continue
            g = os.path.join(os.path.dirname(os.path.dirname(f)), gt_replace[1], os.path.basename(f))
            key = f'{parent}/{os.path.splitext(os.path.basename(f))[0]}'
        else:
            g = os.path.join(os.path.dirname(f), os.path.basename(f).replace(gt_replace[0], gt_replace[1]))
            key = os.path.splitext(os.path.basename(f))[0]
        if os.path.exists(g) and os.path.abspath(g) != os.path.abspath(f):
            pairs.append((f, g, key))
        else:
            missing += 1
    if not pairs:
        raise FileNotFoundError(f'no {input_glob} pairs under {root}')
    return pairs, dict(root=root, n_input_files=len(ins), n_input_keys=len(ins), n_gt_keys=len(pairs),
                       n_pairs=len(pairs), n_unmatched=missing)


def level_names(cfg_names, level):
    import base64
    return {n for n in zlib.decompress(base64.b64decode(''.join(cfg_names[level]))).decode().split('\n') if n}


def spec_pairs(spec, roots):
    if spec.get('synth'):    # synthetic category: the clean images only (degraded on the fly)
        d = resolve_path(roots, spec['gt'])
        files = list_images(d)
        if not files:
            raise FileNotFoundError(f'no clean images in {d}')
        pairs = [(f, f, os.path.splitext(os.path.basename(f))[0]) for f in files]
        return pairs, dict(input_dir=d, gt_dir=d, n_input_files=len(files), n_gt_files=len(files), n_input_keys=len(files),
                           n_gt_keys=len(files), n_pairs=len(pairs), n_unmatched=0, synthetic=spec['synth'])
    if 'root' in spec:
        return pair_by_replace(resolve_path(roots, spec['root']), spec['input_glob'], spec['gt_replace'])
    pairs, rep = pair_files(resolve_path(roots, spec['input']), resolve_path(roots, spec['gt']),
                            spec.get('match', 'stem'), spec.get('include'))
    if spec.get('names'):   # subset defined by a list of file names (e.g. Snow100K-S images of test2000)
        keep = level_names(Config.SNOW_LEVEL_NAMES, spec['names'])
        pairs = [p for p in pairs if os.path.basename(p[0]) in keep]
        rep = dict(rep, n_pairs=len(pairs), n_input_keys=len(keep), n_gt_keys=len(keep), subset=f'names {spec["names"]}')
    return pairs, rep


def spec_keys(spec):
    """(source of the inputs, source of the ground truths)."""
    if 'root' in spec:
        return spec['root'][0], spec['root'][0]
    return spec['input'][0], spec['gt'][0]


def benchmark_files(cfg, roots, name):
    """[(source key, relative path)] of every file a benchmark reads (for the Drive cache)."""
    spec = cfg.benchmarks()[name]
    out = []
    if 'base' in spec:
        return out
    if 'lmdb' in spec:
        for ref in (spec['lmdb'], spec['lmdb_gt']):
            d = resolve_path(roots, ref)
            out += [(ref[0], os.path.relpath(os.path.join(d, f), roots[ref[0]])) for f in os.listdir(d)]
        return out
    if 'clean' in spec:
        k = spec['clean'][0]
        return [(k, os.path.relpath(f, roots[k])) for f in list_images(resolve_path(roots, spec['clean']))]
    pairs, _ = spec_pairs(spec, roots)
    ki, kg = spec_keys(spec)
    for a, g, _ in pairs:
        out += [(ki, os.path.relpath(a, roots[ki])), (kg, os.path.relpath(g, roots[kg]))]
    return out


# ==========================================
# 3. Leakage audit (near-duplicate clean images between train and test)
# ==========================================

class LeakageAuditor:
    """Two-stage near-duplicate test on the clean images.  Screening: zero-mean / unit-norm
    24x24 grey thumbnails, correlation > `threshold`.  Confirmation of every screened pair:
    64x64 thumbnails, correlation > `confirm`, aspect ratios within 5 %.  (On RainTrainH vs
    Rain100H this finds exactly the 546 duplicated backgrounds reported by the PReNet authors.)"""

    def __init__(self, cache_dir, threshold=0.97, confirm=0.95, workers=8):
        self.cache_dir, self.threshold, self.confirm, self.workers = cache_dir, threshold, confirm, max(1, workers)
        os.makedirs(cache_dir, exist_ok=True)
        self._fine = {}

    @staticmethod
    def _vec(im, s):
        v = np.asarray(im.resize((s, s), Image.BILINEAR), np.float32).ravel()
        v = v - v.mean()
        return v / (np.linalg.norm(v) + 1e-6)

    def _thumb(self, f):
        im = Image.open(f).convert('L')
        return self._vec(im, 24), im.size[0] / im.size[1]

    def _fine_thumb(self, f):
        if f not in self._fine:
            self._fine[f] = self._vec(Image.open(f).convert('L'), 64)
        return self._fine[f]

    def thumbs(self, files, tag, ids=None):
        ids = ids or files
        h = hashlib.md5('\n'.join(ids).encode()).hexdigest()[:12]
        cache = os.path.join(self.cache_dir, f'thumbs_{tag}_{h}.npz')
        if os.path.exists(cache):
            z = np.load(cache)
            return z['X'], z['A']
        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(self.workers) as ex:
            res = list(ex.map(self._thumb, files))
        X, A = np.stack([r[0] for r in res]), np.array([r[1] for r in res], np.float32)
        np.savez(cache, X=X, A=A)
        return X, A

    def duplicates(self, train_gts, test_gts, tag_train, tag_test, train_ids=None, test_ids=None):
        (Xa, Aa), (Xb, Ab) = self.thumbs(train_gts, tag_train, train_ids), self.thumbs(test_gts, tag_test, test_ids)
        dup = np.zeros(len(Xa), bool)
        match = [None] * len(Xa)
        for i in range(0, len(Xa), 2048):
            S = Xa[i:i + 2048] @ Xb.T
            for r, c in zip(*np.nonzero(S > self.threshold)):
                ia = i + r
                if dup[ia] or abs(Aa[ia] / Ab[c] - 1) > 0.05:
                    continue
                if float(self._fine_thumb(train_gts[ia]) @ self._fine_thumb(test_gts[c])) > self.confirm:
                    dup[ia], match[ia] = True, os.path.basename(test_gts[c])
        return dup, match


# ==========================================
# 4. Training pool manifest (paths stored relative to their source)
# ==========================================

def manifest_signature(cfg):
    return hashlib.md5(json.dumps([cfg.train_categories(), cfg.MAX_PER_CATEGORY, cfg.VAL_PER_CATEGORY, cfg.DATA_SUBSET_SEED,
                                   cfg.EXCLUDE_TEST_OVERLAP, cfg.OVERLAP_THRESHOLD, cfg.OVERLAP_CONFIRM, cfg.KOHLER_PROTOCOL, cfg.LEAKAGE_SCOPE,
                                   sorted(overlap_benchmarks(cfg).items())],
                                  sort_keys=True, default=str).encode()).hexdigest()


def manifest_path(cfg):
    return os.path.join(cfg.manifest_dir, 'train_manifest.json')


def manifest_is_current(cfg):
    p = manifest_path(cfg)
    return os.path.exists(p) and json.load(open(p)).get('signature') == manifest_signature(cfg)


def overlap_benchmarks(cfg):
    """family -> benchmarks whose clean images training data must not duplicate."""
    out = {}
    for name, s in cfg.benchmarks().items():
        if 'base' in s or 'lmdb' in s:
            continue
        out.setdefault(s['family'], []).append(name)
    return {k: sorted(v) for k, v in out.items()}


def resolve_manifest(man, roots):
    """Copy of the manifest with absolute paths for this session."""
    missing = sorted({it[k] for s in ('train', 'val') for it in man[s] for k in ('src_in', 'src_gt')} - set(roots))
    if missing:
        raise FileNotFoundError(f'data sources needed by the training manifest are not available: {missing} '
                                f'(see the download messages above)')
    out = dict(man)
    for split in ('train', 'val'):
        out[split] = [dict(it, input=os.path.join(roots[it['src_in']], it['input']),
                           gt=os.path.join(roots[it['src_gt']], it['gt'])) for it in man[split]]
    return out


def build_training_manifest(cfg, roots, force=False):
    """Pairs of every training category -> seeded shuffle -> leakage filtering of the candidates ->
    held-out validation split + capped training subset.  Written once (relative paths) to
    data_manifest/train_manifest.json; returned with absolute paths for this session."""
    os.makedirs(cfg.manifest_dir, exist_ok=True)
    path = manifest_path(cfg)
    sig = manifest_signature(cfg)
    if os.path.exists(path) and not force:
        man = json.load(open(path))
        if man.get('signature') == sig:
            return resolve_manifest(man, roots)
        print('[data] configuration changed -> rebuilding the training manifest')
    cats = cfg.train_categories()
    fam_bench = overlap_benchmarks(cfg)
    auditor = LeakageAuditor(os.path.join(cfg.manifest_dir, 'thumb_cache'), cfg.OVERLAP_THRESHOLD, cfg.OVERLAP_CONFIRM,
                             workers=cfg.NUM_WORKERS)
    rng = random.Random(cfg.DATA_SUBSET_SEED)
    man = dict(signature=sig, categories={}, train=[], val=[], audit={})
    bench_gts = {}
    errors = []
    for ci, (name, spec) in enumerate(cats.items()):
        try:
            pairs, rep = spec_pairs(spec, roots)
        except FileNotFoundError as e:
            errors.append(f'{name}: {e}')
            man['categories'][name] = dict(available=False, error=str(e))
            continue
        ki, kg = spec_keys(spec)
        pairs = sorted(pairs)
        rng.shuffle(pairs)
        n_val = min(cfg.VAL_PER_CATEGORY, max(1, len(pairs) // 10)) if len(pairs) >= 5 else 0
        need = n_val + cfg.MAX_PER_CATEGORY
        scope = sorted({b for v in fam_bench.values() for b in v}) if cfg.LEAKAGE_SCOPE == 'all' else fam_bench.get(spec['family'], [])
        checks = sorted(set(spec.get('exclude_overlap_with', [])) | set(scope)) if cfg.EXCLUDE_TEST_OVERLAP else []
        for b in checks:
            if b not in bench_gts:
                try:
                    bs = cfg.benchmarks()[b]
                    kb = bs['clean'][0] if 'clean' in bs else spec_keys(bs)[1]
                    files = benchmark_gt_paths(cfg, roots, b)
                    bench_gts[b] = (files, [os.path.relpath(f, roots[kb]) for f in files])
                except (FileNotFoundError, KeyError) as e:
                    print(f'[data] overlap check vs {b} skipped: {e}')
                    bench_gts[b] = ([], [])
        kept, removed, pos = [], [], 0
        while len(kept) < need and pos < len(pairs):
            chunk = pairs[pos:pos + int(1.2 * (need - len(kept))) + 32]
            pos += len(chunk)
            dup = np.zeros(len(chunk), bool)
            ids = [f'{ki}/{os.path.relpath(g, roots[kg])}' for _, g, _ in chunk]
            for b in checks:
                files, rel = bench_gts[b]
                if not files:
                    continue
                d, match = auditor.duplicates([g for _, g, _ in chunk], files, name, b, ids, rel)
                a = man['audit'].setdefault(f'{name}_vs_{b}', dict(n_checked=0, n_duplicates=0, examples=[]))
                a['n_checked'] += len(chunk)
                a['n_duplicates'] += int(d.sum())
                a['examples'] = (a['examples'] + [f'{os.path.basename(chunk[i][1])} = {b}/{match[i]}' for i in np.nonzero(d)[0]])[:10]
                dup |= d
            removed += [chunk[i] for i in np.nonzero(dup)[0]]
            kept += [chunk[i] for i in np.nonzero(~dup)[0]]
        kept = kept[:need]
        val, train = kept[:n_val], sorted(kept[n_val:])
        rep = dict(rep, available=True, family=spec['family'], index=ci, n_removed_overlap=len(removed),
                   n_candidates_checked=pos, overlap_checked_against=checks,
                   n_train=len(train), n_val=len(val), repeat=spec.get('repeat', 1))
        man['categories'][name] = rep

        def item(a, g):
            it = dict(src_in=ki, input=os.path.relpath(a, roots[ki]), src_gt=kg, gt=os.path.relpath(g, roots[kg]),
                      category=name, family=spec['family'])
            if spec.get('synth'):      # degraded on the fly from the clean image (Section 5b)
                it['synth'] = spec['synth']
            return it
        man['train'] += [dict(item(a, g), repeat=spec.get('repeat', 1)) for a, g, _ in train]
        man['val'] += [item(a, g) for a, g, _ in val]
        print(f'[data] {name:11s} pairs={rep["n_pairs"]:6d}  checked={pos:6d}  overlap-removed={len(removed):4d}  '
              f'train={len(train):5d}  val={len(val):3d}  repeat={spec.get("repeat", 1)}')
    if errors and not cfg.ALLOW_MISSING_CATEGORIES:
        raise RuntimeError('training categories without data:\n  ' + '\n  '.join(errors))
    with open(path + '.tmp', 'w') as f:   # atomic: several sessions may build it at the same time
        json.dump(man, f, indent=1)
    os.replace(path + '.tmp', path)
    return resolve_manifest(man, roots)


def benchmark_gt_paths(cfg, roots, name):
    spec = cfg.benchmarks()[name]
    if 'clean' in spec:
        return list_images(resolve_path(roots, spec['clean']))
    pairs, _ = spec_pairs(spec, roots)
    return [g for _, g, _ in pairs]


def manifest_used_files(cfg, roots, man):
    """{source key: set(relative paths)} read by the experiments (Drive cache in 'used' mode)."""
    used = {}
    for split in ('train', 'val'):
        for it in man[split]:
            used.setdefault(it['src_in'], set()).add(os.path.relpath(it['input'], roots[it['src_in']]))
            used.setdefault(it['src_gt'], set()).add(os.path.relpath(it['gt'], roots[it['src_gt']]))
    for name in cfg.benchmarks():
        try:
            for k, r in benchmark_files(cfg, roots, name):
                used.setdefault(k, set()).add(r)
        except (FileNotFoundError, KeyError):
            pass
    return used


# ==========================================
# 4b. Pair validation (alignment, direction of the degradation, sizes, readability)
# ==========================================

DIRECTION = {'blur': 'sharper_gt', 'defocus': 'sharper_gt', 'rain': 'cleaner_gt', 'snow': 'cleaner_gt',
             'noise': 'cleaner_gt', 'haze': 'contrast_gt', 'lowlight': 'brighter_gt'}


def _pair_stats(a_path, g_path, a_img=None, g_img=None):
    a = a_img or Image.open(a_path).convert('RGB')
    g = g_img or Image.open(g_path).convert('RGB')
    same = a.size == g.size
    if not same:
        g = g.resize(a.size, Image.BICUBIC)
    mse_full = float(((np.asarray(a, np.float32) - np.asarray(g, np.float32)) ** 2).mean()) / 255. ** 2
    s = 512 / max(a.size)
    if s < 1:
        a, g = a.resize((max(1, int(a.size[0] * s)), max(1, int(a.size[1] * s))), Image.BILINEAR), \
               g.resize((max(1, int(a.size[0] * s)), max(1, int(a.size[1] * s))), Image.BILINEAR)
    A = np.asarray(a.convert('L'), np.float32) / 255.
    G = np.asarray(g.convert('L'), np.float32) / 255.
    hf = lambda x: float(np.abs(np.diff(x, axis=0)).mean() + np.abs(np.diff(x, axis=1)).mean())
    va, vg = LeakageAuditor._vec(Image.fromarray((A * 255).astype(np.uint8)), 64), LeakageAuditor._vec(Image.fromarray((G * 255).astype(np.uint8)), 64)
    mse = float(((np.asarray(a, np.float32) - np.asarray(g, np.float32)) ** 2).mean()) / 255. ** 2
    return dict(same_size=same, corr=float(va @ vg), psnr=10 * np.log10(1 / (mse + 1e-10)),
                psnr_full=10 * np.log10(1 / (mse_full + 1e-10)), hf_in=hf(A), hf_gt=hf(G),
                std_in=float(A.std()), std_gt=float(G.std()), mean_in=float(A.mean()), mean_gt=float(G.mean()))


def _direction_ok(st, rule):
    return {'sharper_gt': st['hf_in'] < st['hf_gt'], 'cleaner_gt': st['hf_in'] > st['hf_gt'],
            'contrast_gt': st['std_in'] < st['std_gt'], 'brighter_gt': st['mean_in'] < st['mean_gt']}[rule]


def _summarise(name, family, stats, rule, cfg, extra=None):
    corr = [s['corr'] for s in stats]
    r = dict(name=name, family=family, n_checked=len(stats), median_corr=float(np.median(corr)) if corr else None,
             min_corr=float(np.min(corr)) if corr else None, median_input_psnr=float(np.median([s['psnr'] for s in stats])) if stats else None,
             n_low_corr=int(sum(c < cfg.MIN_PAIR_CORR for c in corr)), n_size_mismatch=int(sum(not s['same_size'] for s in stats)),
             direction_rule=rule, **(extra or {}))
    r['direction_rate'] = float(np.mean([_direction_ok(s, rule) for s in stats])) if (rule and stats) else None
    flags = []
    if r['median_corr'] is not None and r['median_corr'] < cfg.MIN_PAIR_CORR:
        flags.append('MISPAIR')
    if r['direction_rate'] is not None and r['direction_rate'] < cfg.MIN_DIRECTION_RATE:
        flags.append('SWAP?')
    if r['n_size_mismatch']:
        flags.append('SIZE')
    if r.get('n_unreadable'):
        flags.append('UNREADABLE')
    if r.get('pair_rate') is not None and r['pair_rate'] < 0.9:
        flags.append('UNMATCHED')
    # fingerprint of a known benchmark (number of pairs, mean input PSNR at full resolution): catches a
    # source that holds another test set (e.g. Rain100H served as Rain100L: 12.1 dB instead of 25.5 dB)
    fp = cfg.BENCHMARK_FINGERPRINTS.get(name) if extra and 'n_pairs' in extra else None
    if fp and stats:
        r['mean_input_psnr_full'] = float(np.mean([s['psnr_full'] for s in stats]))
        r['fingerprint'] = dict(n_pairs=fp[0], mean_input_psnr=fp[1])
        if r['n_pairs'] != fp[0] or abs(r['mean_input_psnr_full'] - fp[1]) > cfg.FINGERPRINT_TOL_DB:
            flags.append('FINGERPRINT')
    r['flags'] = flags
    return r


def validation_signature(cfg):
    """Definitions checked by validate_data(); a change re-runs the validation."""
    return hashlib.md5(json.dumps([cfg.train_categories(), cfg.benchmarks(), cfg.BENCHMARK_FINGERPRINTS, cfg.FINGERPRINT_TOL_DB,
                                   [[k, getattr(cfg, 'KAGGLE_SOURCES', {}).get(k)] for k in sorted(getattr(cfg, 'KAGGLE_SOURCES', {}))]],
                                  sort_keys=True, default=str).encode()).hexdigest()


def validate_data(cfg, roots, man):
    """Checks a sample of every training category and every benchmark; writes
    data_manifest/data_validation.json.  Training categories flagged MISPAIR or SWAP? stop the run."""
    rng = random.Random(cfg.DATA_SUBSET_SEED + 1)
    rows = []

    def sample_stats(pairs):
        stats, bad = [], 0
        for a, g in rng.sample(pairs, min(cfg.VALIDATE_SAMPLES, len(pairs))):
            try:
                stats.append(_pair_stats(a, g))
            except Exception:
                bad += 1
        return stats, bad

    for name, spec in cfg.train_categories().items():
        items = [(it['input'], it['gt']) for it in man['train'] if it['category'] == name]
        if not items:
            continue
        if spec.get('synth'):   # check the generated pairs themselves (generator, sizes, direction)
            ds = UniversalRestorationDataset([dict(it, repeat=1) for it in man['train'] if it['category'] == name][:cfg.VALIDATE_SAMPLES],
                                             cfg.PATCH_SIZE, False)
            stats, bad = [], 0
            for k in range(len(ds)):
                try:
                    b = ds[k]
                    stats.append(_pair_stats(None, None, TF.to_pil_image(b['input']), TF.to_pil_image(b['gt'])))
                except Exception:
                    bad += 1
        else:
            stats, bad = sample_stats(items)
        cat = man['categories'][name]
        pr = cat['n_pairs'] / max(1, min(cat.get('n_input_keys', cat['n_pairs']), cat.get('n_gt_keys', cat['n_pairs'])))
        r = dict(_summarise(name, spec['family'], stats, spec.get('direction', DIRECTION.get(spec['family'])), cfg,
                            dict(n_unreadable=bad, pair_rate=pr)), kind='train')
        if spec.get('synth'):   # pairs correct by construction: pairing / direction statistics are informative only
            r['flags'] = [f for f in r['flags'] if f not in ('MISPAIR', 'SWAP?')]
            r['synthetic'] = spec['synth']
        rows.append(r)
    for name, spec in cfg.benchmarks().items():
        if 'base' in spec or 'clean' in spec:
            continue
        try:
            if 'lmdb' in spec:
                rd = LMDBPairReader(resolve_path(roots, spec['lmdb']), resolve_path(roots, spec['lmdb_gt']))
                keys = rng.sample(rd.keys, min(cfg.VALIDATE_SAMPLES, len(rd.keys)))
                stats = [_pair_stats(None, None, *rd.get(k)) for k in keys]
                bad, extra = 0, dict(n_pairs=len(rd.keys))
            else:
                pairs, rep = spec_pairs(spec, roots)
                stats, bad = sample_stats([(a, g) for a, g, _ in pairs])
                extra = dict(n_pairs=rep['n_pairs'], pair_rate=rep['n_pairs'] / max(1, min(rep['n_input_keys'], rep['n_gt_keys'])))
        except (FileNotFoundError, KeyError) as e:
            rows.append(dict(name=name, kind='benchmark', family=spec['family'], flags=['MISSING'], error=str(e)))
            continue
        rows.append(dict(_summarise(name, spec['family'], stats, spec.get('direction', DIRECTION.get(spec['family'])), cfg,
                                    dict(extra, n_unreadable=bad)), kind='benchmark'))
    os.makedirs(cfg.manifest_dir, exist_ok=True)
    with open(os.path.join(cfg.manifest_dir, 'data_validation.json'), 'w') as f:
        json.dump(rows, f, indent=1)
    print(f'{"set":22s} {"kind":9s} {"n":>4s} {"corr":>6s} {"dir":>5s} {"psnr_in":>7s}  flags')
    for r in rows:
        print(f'{r["name"]:22s} {r["kind"]:9s} {r.get("n_checked", 0):4d} {r.get("median_corr") or 0:6.3f} '
              f'{(r.get("direction_rate") if r.get("direction_rate") is not None else float("nan")):5.2f} '
              f'{r.get("median_input_psnr") or 0:7.2f}  {",".join(r["flags"])}')
    fatal = [r['name'] for r in rows if r['kind'] == 'train' and set(r['flags']) & {'MISPAIR', 'SWAP?', 'UNREADABLE'}]
    if fatal and cfg.VALIDATION_STRICT:
        raise RuntimeError(f'training data failed validation: {fatal} (see data_manifest/data_validation.json)')
    wrong = [f'{r["name"]} (n={r.get("n_pairs")}, input {r.get("mean_input_psnr_full", float("nan")):.2f} dB; expected '
             f'n={r["fingerprint"]["n_pairs"]}, {r["fingerprint"]["mean_input_psnr"]:.2f} dB)' for r in rows if 'FINGERPRINT' in r['flags']]
    if wrong and cfg.VALIDATION_STRICT:
        raise RuntimeError('benchmark data do not match the known test set: ' + '; '.join(wrong) +
                           ' (wrong source or folder; see data_manifest/data_validation.json)')
    return rows


# ==========================================
# 5. Synthetic degradations (test-time only, deterministic per image)
# ==========================================

def image_seed(base, name):
    return (base + zlib.crc32(os.path.basename(name).encode())) % (2 ** 31)


def degrade(img_t, ops, seed, haze_depth=(0.5, 1.0)):
    """img_t: (3,H,W) float in [0,1].  ops: [('gaussian', sigma_255) | ('jpeg', q) | ('haze', beta, A)]."""
    x = img_t.clone()
    g = torch.Generator().manual_seed(seed)
    for op in ops:
        if op[0] == 'gaussian':
            x = (x + torch.randn(x.shape, generator=g) * (op[1] / 255.0)).clamp(0, 1)
        elif op[0] == 'jpeg':
            buf = io.BytesIO()
            TF.to_pil_image(x.clamp(0, 1)).save(buf, format='JPEG', quality=int(op[1]))
            x = TF.to_tensor(Image.open(io.BytesIO(buf.getvalue())).convert('RGB'))
        elif op[0] == 'haze':  # atmospheric scattering model I = J t + A (1 - t), t = exp(-beta d)
            beta, A = op[1], op[2]
            H = x.shape[1]
            far, near = haze_depth[1], haze_depth[0]
            d = torch.linspace(far, near, H).view(1, H, 1)
            t = torch.exp(-beta * d)
            x = (x * t + A * (1 - t)).clamp(0, 1)
        else:
            raise ValueError(op)
    return x


# ==========================================
# 5b. Physically based synthetic degradations for TRAINING (camera shake, composites)
# ==========================================
# Applied on the fly to clean training images (a new draw every epoch; validation items use a
# fixed seed per image).  Everything is computed in linear light (sRGB gamma 2.2 removed), as a
# sensor integrates light, then re-encoded.

def _srgb_to_lin(x):
    return x.clamp(0, 1) ** 2.2


def _lin_to_srgb(x):
    return x.clamp(0, 1) ** (1 / 2.2)


def _rot(ax, ay, az):
    cx, sx, cy, sy, cz, sz = math.cos(ax), math.sin(ax), math.cos(ay), math.sin(ay), math.cos(az), math.sin(az)
    Rx = torch.tensor([[1, 0, 0], [0, cx, -sx], [0, sx, cx]], dtype=torch.float32)
    Ry = torch.tensor([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]], dtype=torch.float32)
    Rz = torch.tensor([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]], dtype=torch.float32)
    return Rz @ Ry @ Rx


def shake_trajectory(rng, n=48, extent_px=20.0, focal=1000.0, roll_ratio=0.3):
    """6-DOF hand-shake model, rotations only (translations are negligible for scenes farther than
    ~1 m, as in Koehler et al., ECCV 2012).  Angular velocity is a random walk with inertia plus rare
    impulsive jerks (Boracchi & Foi, TIP 2012, extended to 3 axes).  The path is centred on its mean pose
    (the sharp reference is the mean pose) and scaled so that the image-centre blur extent is `extent_px`."""
    w = np.zeros(3)
    ang = np.zeros((n, 3))
    for t in range(1, n):
        jerk = rng.normal(0, 1.0, 3) * (rng.random() < 0.08) * 3.0
        w = 0.75 * w + rng.normal(0, 1.0, 3) + jerk
        ang[t] = ang[t - 1] + w
    ang -= ang.mean(0)
    span = np.ptp(ang[:, :2], axis=0).max() + 1e-9
    ang[:, :2] *= (extent_px / focal) / span          # pitch / yaw: displacement ~ focal * angle
    ang[:, 2] *= roll_ratio * (extent_px / focal) / (np.ptp(ang[:, 2]) + 1e-9) * rng.random()
    return ang


def camera_shake(clean_lin, rng, out=256, extent_px=None, frame=None):
    """clean_lin: (3, S, S) linear-light patch with S = out + 2 m.  Returns the (3, out, out) blurred
    linear patch = mean over the trajectory of the patch warped by H_t = K R_t K^-1, the patch being at
    a random position of a virtual full frame (spatially varying blur, strongest away from the centre
    for the roll component)."""
    S = clean_lin.shape[-1]
    m = (S - out) // 2
    extent_px = extent_px if extent_px is not None else float(rng.uniform(4, 40))
    Wf = frame or float(rng.uniform(800, 1600))
    focal = Wf * float(rng.uniform(0.8, 1.4))
    ox, oy = rng.uniform(-Wf / 2, Wf / 2 - out), rng.uniform(-Wf / 2.6, Wf / 2.6 - out)   # top-left of the patch in centred frame coords
    ang = shake_trajectory(rng, n=int(rng.integers(24, 64)), extent_px=extent_px, focal=focal)
    ys, xs = torch.meshgrid(torch.arange(out, dtype=torch.float32), torch.arange(out, dtype=torch.float32), indexing='ij')
    P = torch.stack([xs + ox, ys + oy, torch.full_like(xs, focal)], 0).view(3, -1)    # rays of the output pixels
    grids = []
    for a in ang:
        q = _rot(*a) @ P
        u = focal * q[0] / q[2] - ox + m        # source pixel in the margin-padded patch
        v = focal * q[1] / q[2] - oy + m
        grids.append(torch.stack([u / (S - 1) * 2 - 1, v / (S - 1) * 2 - 1], -1).view(out, out, 2))
    grid = torch.stack(grids)
    src = clean_lin.unsqueeze(0).expand(len(grids), -1, -1, -1)
    warped = F.grid_sample(src, grid, mode='bilinear', padding_mode='reflection', align_corners=True)
    return warped.mean(0)


def _disk(r):
    k = int(math.ceil(r))
    y, x = torch.meshgrid(torch.arange(-k, k + 1, dtype=torch.float32), torch.arange(-k, k + 1, dtype=torch.float32), indexing='ij')
    d = (r + 0.5 - torch.sqrt(x ** 2 + y ** 2)).clamp(0, 1)
    return d / d.sum()


def _line(length, angle_deg):
    k = int(math.ceil(length / 2)) + 1
    ker = torch.zeros(2 * k + 1, 2 * k + 1)
    a = math.radians(angle_deg)
    for t in np.linspace(-length / 2, length / 2, int(length * 3) + 1):
        ker[int(round(k + t * math.sin(a))), int(round(k + t * math.cos(a)))] += 1
    return ker / ker.sum()


def _conv(x, ker):
    k = ker.shape[-1] // 2
    return F.conv2d(F.pad(x.unsqueeze(0), (k, k, k, k), mode='reflect'), ker.view(1, 1, *ker.shape).repeat(x.shape[0], 1, 1, 1),
                    groups=x.shape[0])[0]


def _smooth_field(rng, H, W, cells=4):
    f = torch.from_numpy(rng.random((1, 1, cells, cells)).astype(np.float32))
    return F.interpolate(f, size=(H, W), mode='bicubic', align_corners=False)[0, 0].clamp(0, 1)


def _gen(rng):
    return torch.Generator().manual_seed(int(rng.integers(0, 2 ** 31 - 1)))


SYNTH_TYPES = ('haze', 'rain', 'snow', 'defocus', 'shake', 'lowlight', 'noise', 'jpeg')
_SYNTH_ORDER = {t: i for i, t in enumerate(SYNTH_TYPES)}    # physical order: scene -> optics -> exposure -> sensor -> codec


def synth_degrade(clean, kind, rng, out=256, types=SYNTH_TYPES):
    """clean: (3, S, S) sRGB patch, S = out + 2 * SYNTH_MARGIN.  kind 'camera_shake': camera-shake blur +
    sensor noise (+ JPEG); kind 'composite': 1-3 degradations of `types` in physical order.
    Returns (degraded, gt), both (3, out, out) in [0, 1]."""
    S = clean.shape[-1]
    m = (S - out) // 2
    gt = clean[:, m:m + out, m:m + out]
    g = _gen(rng)
    if kind == 'camera_shake':
        ops = ['shake', 'noise'] + (['jpeg'] if rng.random() < 0.5 else [])      # Koehler-like: low ISO, mild noise
        params = dict(noise=float(rng.uniform(0.0, 0.006)), jpeg=int(rng.integers(75, 96)))
    else:
        pool = [t for t in types]
        k = int(rng.choice([1, 2, 3], p=[0.25, 0.45, 0.30]))
        ops = []
        while len(ops) < k and pool:
            t = pool.pop(int(rng.integers(len(pool))))
            if (t == 'defocus' and 'shake' in ops) or (t == 'shake' and 'defocus' in ops):
                continue
            ops.append(t)
        ops.sort(key=_SYNTH_ORDER.get)
        params = dict(noise=float(rng.uniform(0.01, 0.12)), jpeg=int(rng.integers(10, 60)))
    x = _srgb_to_lin(clean)
    for op in ops:
        if op == 'haze':      # ASM with a smooth random depth field: I = J t + A (1 - t)
            d = 0.4 + 0.6 * _smooth_field(rng, S, S)
            t = torch.exp(-float(rng.uniform(0.6, 2.5)) * d)
            A = float(rng.uniform(0.6, 1.0))
            x = x * t + _srgb_to_lin(torch.tensor(A)) * (1 - t)
        elif op == 'rain':    # sparse drops motion-blurred into streaks, added to the scene
            dens = float(rng.uniform(0.002, 0.012))
            seeds = (torch.rand((1, S, S), generator=g) < dens).float() * torch.rand((1, S, S), generator=g)
            streak = _conv(seeds, _line(float(rng.uniform(10, 45)), float(90 + rng.uniform(-30, 30))))
            streak = (streak / (streak.max() + 1e-6)) * float(rng.uniform(0.4, 0.9))
            x = 1 - (1 - x) * (1 - streak)            # screen blend
        elif op == 'snow':    # opaque flakes of several sizes, slightly motion-blurred
            layer = torch.zeros(1, S, S)
            for r, dens in ((1.0, 0.004), (2.0, 0.0015), (3.5, 0.0005)):
                pts = (torch.rand((1, S, S), generator=g) < dens * float(rng.uniform(0.5, 2))).float()
                layer = torch.maximum(layer, _conv(pts, _disk(r)) * (2 * r + 1) ** 2 * 0.6)
            layer = _conv(layer.clamp(0, 1), _line(float(rng.uniform(1, 6)), float(rng.uniform(60, 120))))
            x = x * (1 - layer) + layer * float(rng.uniform(0.8, 1.0))
        elif op == 'defocus':
            x = _conv(x, _disk(float(rng.uniform(1.5, 8))))
        elif op == 'shake':   # whole S x S patch (the outer margin absorbs the border of the warps)
            x = camera_shake(F.pad(x.unsqueeze(0), (16, 16, 16, 16), mode='reflect')[0], rng, out=S,
                             extent_px=float(rng.uniform(4, 40)))
        elif op == 'lowlight':
            x = x * float(rng.uniform(0.02, 0.25))
            params['noise'] = max(params['noise'], float(rng.uniform(0.01, 0.05)))
            if 'noise' not in ops:
                ops.append('noise')
        elif op == 'noise':   # Poisson-Gaussian sensor noise in linear light
            a, b = params['noise'] * float(rng.uniform(0.2, 1.0)), params['noise']
            x = x + torch.sqrt(a * x.clamp(min=0) + b ** 2) * torch.randn(x.shape, generator=g)
        elif op == 'jpeg':
            pass                # applied after re-encoding to sRGB
    y = _lin_to_srgb(x)[:, m:m + out, m:m + out]
    if 'jpeg' in ops:
        buf = io.BytesIO()
        TF.to_pil_image(y.clamp(0, 1)).save(buf, format='JPEG', quality=params['jpeg'])
        y = TF.to_tensor(Image.open(io.BytesIO(buf.getvalue())).convert('RGB'))
    return y.clamp(0, 1), gt


SYNTH_MARGIN = 48   # clean patches are cropped with this margin so that blur kernels never see a border


# ==========================================
# 6. Datasets
# ==========================================

def load_rgb(path):
    return Image.open(path).convert('RGB')


class UniversalRestorationDataset(Dataset):
    """Training / validation patches.  Train: random patch_size crop + horizontal and
    vertical flips.  Images whose shorter side is below the patch size are first up-scaled
    (bicubic, aspect ratio kept; the submitted code stretched them to a square).
    Validation: centre crop."""

    def __init__(self, items, patch_size=256, is_train=True, families=None):
        self.patch_size, self.is_train = patch_size, is_train
        self.samples = []
        for it in items:
            self.samples += [it] * (int(it.get('repeat', 1)) if is_train else 1)
        cats = sorted({it['category'] for it in items})
        self.task_to_id = {c: i for i, c in enumerate(cats)}
        self.families = families or sorted({it['family'] for it in items})
        self.family_to_id = {f: i for i, f in enumerate(self.families)}

    def __len__(self):
        return len(self.samples)

    def transform(self, image, gt):
        if gt.size != image.size:
            gt = gt.resize(image.size, Image.BICUBIC)
        w, h = image.size
        p = self.patch_size
        if w < p or h < p:   # e.g. CelebA 178x218: bicubic up-scaling that keeps the aspect ratio (short side = p)
            sc = p / min(w, h)
            new_w, new_h = max(p, round(w * sc)), max(p, round(h * sc))
            image = image.resize((new_w, new_h), Image.BICUBIC)
            gt = gt.resize((new_w, new_h), Image.BICUBIC)
            w, h = image.size
        if self.is_train:
            left, top = random.randint(0, w - p), random.randint(0, h - p)
            image = image.crop((left, top, left + p, top + p))
            gt = gt.crop((left, top, left + p, top + p))
            if random.random() > 0.5:
                image, gt = TF.hflip(image), TF.hflip(gt)
            if random.random() > 0.5:
                image, gt = TF.vflip(image), TF.vflip(gt)
        else:
            left, top = (w - p) // 2, (h - p) // 2
            image = image.crop((left, top, left + p, top + p))
            gt = gt.crop((left, top, left + p, top + p))
        return TF.to_tensor(image), TF.to_tensor(gt)

    def synth_item(self, item):
        """Clean patch with a margin -> on-the-fly degradation (random every epoch; fixed seed for validation)."""
        img = load_rgb(item['gt'])
        S = self.patch_size + 2 * SYNTH_MARGIN
        w, h = img.size
        if w < S or h < S:
            sc = S / min(w, h)
            img = img.resize((max(S, round(w * sc)), max(S, round(h * sc))), Image.BICUBIC)
            w, h = img.size
        if self.is_train:
            left, top = random.randint(0, w - S), random.randint(0, h - S)
            rng = np.random.default_rng(random.getrandbits(62))
        else:
            left, top = (w - S) // 2, (h - S) // 2
            rng = np.random.default_rng(image_seed(Config.SYNTHETIC_SEED, item['gt']))
        c = TF.to_tensor(img.crop((left, top, left + S, top + S)))
        if self.is_train:
            if random.random() > 0.5:
                c = torch.flip(c, dims=[2])
            if random.random() > 0.5:
                c = torch.flip(c, dims=[1])
        return synth_degrade(c, item['synth'], rng, out=self.patch_size)

    def __getitem__(self, idx):
        item = self.samples[idx]
        if item.get('synth'):
            inp, gt = self.synth_item(item)
        else:
            inp, gt = self.transform(load_rgb(item['input']), load_rgb(item['gt']))
        return {'input': inp, 'gt': gt, 'task_id': self.task_to_id[item['category']],
                'family_id': self.family_to_id[item['family']]}


class LMDBPairReader:
    """BasicSR-style lmdb (SIDD validation crops): keys from meta_info.txt, PNG bytes."""

    def __init__(self, lq_path, gt_path):
        self.lq_path, self.gt_path = lq_path, gt_path
        with open(os.path.join(lq_path, 'meta_info.txt')) as f:
            self.keys = [ln.split(' ')[0].rsplit('.', 1)[0] for ln in f if ln.strip()]
        self._env = None

    def _open(self):
        import lmdb
        self._env = (lmdb.open(self.lq_path, readonly=True, lock=False, readahead=False),
                     lmdb.open(self.gt_path, readonly=True, lock=False, readahead=False))

    def get(self, key):
        if self._env is None:
            self._open()
        out = []
        for env in self._env:
            with env.begin(write=False) as txn:
                out.append(Image.open(io.BytesIO(txn.get(key.encode('ascii')))).convert('RGB'))
        return out


class BenchmarkDataset(Dataset):
    """Full-resolution test set.  Items: name, input, gt, and for mixed sets the
    singly-degraded input ('first') used by the prompt-override control."""

    def __init__(self, cfg, roots, name, max_images=None):
        self.cfg, self.name = cfg, name
        bench = cfg.benchmarks()
        self.spec = spec = bench[name]
        self.family = spec['family']
        self.lmdb = None
        self.base = None
        self.report = {}
        if 'lmdb' in spec:
            self.lmdb = LMDBPairReader(resolve_path(roots, spec['lmdb']), resolve_path(roots, spec['lmdb_gt']))
            self.items = [(k, None, k) for k in self.lmdb.keys]
            self.report = dict(n_pairs=len(self.items), source='lmdb')
        elif 'clean' in spec:
            files = list_images(resolve_path(roots, spec['clean']))
            self.items = [(f, f, os.path.splitext(os.path.basename(f))[0]) for f in files]
            self.report = dict(n_pairs=len(self.items), source='synthetic', degrade=spec['degrade'])
        elif 'base' in spec:
            self.base = BenchmarkDataset(cfg, roots, spec['base'])
            self.items = list(range(len(self.base)))
            self.report = dict(n_pairs=len(self.items), source='mixed', base=spec['base'], degrade=spec['degrade'])
        else:
            pairs, self.report = spec_pairs(spec, roots)
            self.items = [(a, b, k) for a, b, k in pairs]
        limit = max_images or spec.get('max_images')
        if limit and len(self.items) > limit:
            idx = sorted(random.Random(cfg.DATA_SUBSET_SEED).sample(range(len(self.items)), limit))
            self.items = [self.items[i] for i in idx]
            self.report['subset'] = limit

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        spec = self.spec
        if self.base is not None:
            b = self.base[self.items[idx]]
            x = degrade(b['input'], spec['degrade'], image_seed(self.cfg.SYNTHETIC_SEED, b['name']), self.cfg.HAZE_DEPTH)
            return {'name': b['name'], 'key': b['key'], 'gt_name': b['gt_name'], 'input': x, 'gt': b['gt'],
                    'first': b['input'], 'gt_resized': b['gt_resized']}
        a, g, k = self.items[idx]
        if self.lmdb is not None:
            inp, gt = self.lmdb.get(a)
            inp, gt = TF.to_tensor(inp), TF.to_tensor(gt)
        elif 'clean' in spec:
            gt = TF.to_tensor(load_rgb(g))
            inp = degrade(gt, spec['degrade'], image_seed(self.cfg.SYNTHETIC_SEED, g), self.cfg.HAZE_DEPTH)
        else:
            inp, gt = TF.to_tensor(load_rgb(a)), TF.to_tensor(load_rgb(g))
        resized = False
        if gt.shape[1:] != inp.shape[1:]:  # same rule as the submitted inference code; counted in the report
            gt = TF.resize(gt, list(inp.shape[1:]), antialias=True).clamp(0, 1)
            resized = True
        is_file = isinstance(a, str) and os.sep in a
        return {'name': os.path.basename(a) if is_file else str(k), 'key': str(k),
                'gt_name': os.path.basename(g) if isinstance(g, str) and os.sep in g else str(k),
                'input': inp, 'gt': gt, 'gt_resized': resized}


def collate_single(batch):
    return batch[0]

In [ ]:
# ===== Trainer.py (identical to Trainer.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - Trainer (resumable, iteration-based, FP16, curves, run.json)
# --------------------------------------------------------------------------
# * Budget = EPOCHS x floor(N_train / BATCH_SIZE) iterations; Adam, per-iteration
#   cosine annealing, gradient clipping 1.0, FP16 autocast + GradScaler.
# * Automatic resume: last.pth (model, optimiser, scheduler, scaler, data-order
#   position, RNG states, history, accumulated wall-clock) is written atomically
#   every CKPT_EVERY_ITERS iterations, so an epoch may span several sessions and
#   a resumed run follows exactly the same data order.
# * Model selection: best mean PSNR on the held-out validation split of the
#   training pool (never a test set), or the last iterate.  Stopping criterion:
#   the fixed iteration budget (no early stopping).
# * Writes runs/<run_id>/run.json, history.csv and curves.png (train vs val).
# ==========================================================================
import os
import csv
import json
import math
import time
import uuid
import random
import platform
import warnings
import numpy as np

# With FP16 the GradScaler skips the optimiser step of the first iterations (inf gradients while it
# calibrates its scale); PyTorch then warns that the scheduler stepped first.  Harmless and expected.
warnings.filterwarnings('ignore', message='Detected call of `lr_scheduler.step\\(\\)` before `optimizer.step\\(\\)`')
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Sampler


FAMILIES = ['blur', 'rain', 'haze', 'snow', 'noise', 'mixed', 'lowlight', 'defocus', 'jpeg']


def seed_everything(seed, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def environment_info():
    info = dict(python=platform.python_version(), torch=torch.__version__, platform=platform.platform(),
                cpu_count=os.cpu_count())
    try:
        import torchvision
        info['torchvision'] = torchvision.__version__
    except Exception:
        pass
    try:
        import pyiqa
        info['pyiqa'] = pyiqa.__version__
    except Exception:
        pass
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        info.update(gpu=p.name, gpu_count=torch.cuda.device_count(), gpu_memory_gb=round(p.total_memory / 2 ** 30, 1),
                    cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version())
        try:
            import subprocess
            info['driver'] = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                                                     text=True).strip().splitlines()[0]
        except Exception:
            pass
    return info


def atomic_save(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    torch.save(obj, tmp)
    os.replace(tmp, path)


def write_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = path + '.tmp'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2, default=str)
    os.replace(tmp, path)


def read_json(path, default=None):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return default


class ResumableSampler(Sampler):
    """Epoch-seeded permutation; starts at `start` so that a resumed epoch continues
    with the very same sample order."""

    def __init__(self, n, seed, epoch, start=0):
        self.n, self.seed, self.epoch, self.start = n, seed, epoch, start

    def __iter__(self):
        g = torch.Generator().manual_seed(self.seed * 100003 + self.epoch)
        return iter(torch.randperm(self.n, generator=g)[self.start:].tolist())

    def __len__(self):
        return self.n - self.start


SESSION_ID = uuid.uuid4().hex[:12]


def lock_path(cfg, run_id):
    return os.path.join(cfg.run_dir(run_id), 'LOCK.json')


def try_lock(cfg, run_id):
    """True if this session may train `run_id` (no other live session holds it)."""
    lk = read_json(lock_path(cfg, run_id))
    if lk and lk.get('session') != SESSION_ID and time.time() - lk.get('heartbeat', 0) < cfg.LOCK_STALE_MINUTES * 60:
        return False
    write_json(dict(session=SESSION_ID, host=platform.node(), heartbeat=time.time()), lock_path(cfg, run_id))
    return True


def release_lock(cfg, run_id):
    lk = read_json(lock_path(cfg, run_id))
    if lk and lk.get('session') == SESSION_ID:
        os.remove(lock_path(cfg, run_id))


def build_model(cfg, run):
    return KASIRNet(**cfg.model_kwargs(run))


def load_weights(model, path, device='cpu'):
    ck = torch.load(path, map_location=device, weights_only=False)
    sd = ck.get('model_state_dict', ck) if isinstance(ck, dict) else ck
    model.load_state_dict(sd, strict=True)
    return ck


LOSS_CALIBRATION_VERSION = 1


def loss_scales(cfg, manifest, device):
    """Fixed per-term scales of the loss (Loss.py, 'Calibrated weights'): s_k = g_pix / g_k with g_k the mean
    norm of dL_k/d(pred) at pred = degraded input over the held-out validation patches.  Model-independent,
    computed once per training pool and shared by every run (data_manifest/loss_calibration.json)."""
    terms = OmniRestorationLoss.TERMS
    if not cfg.LOSS_CALIBRATION:
        return {k: 1.0 for k in terms}
    path = os.path.join(cfg.manifest_dir, 'loss_calibration.json')
    key = [manifest['signature'], LOSS_CALIBRATION_VERSION, cfg.PATCH_SIZE]
    rec = read_json(path, {})
    if rec.get('key') == key:
        return rec['scales']
    crit = OmniRestorationLoss({k: 1.0 for k in terms}).to(device)
    ds = UniversalRestorationDataset(manifest['val'], cfg.PATCH_SIZE, False, FAMILIES)
    norms = {k: [] for k in terms}
    for b in DataLoader(ds, batch_size=16, shuffle=False, num_workers=min(4, cfg.NUM_WORKERS)):
        x, y = b['input'].to(device), b['gt'].to(device)
        for k in terms:
            p = x.clone().requires_grad_(True)
            val = crit.terms(p, y, only={k})[k]
            g, = torch.autograd.grad(val, p)
            norms[k].append(float(g.norm()))
    g = {k: float(np.mean(v)) for k, v in norms.items()}
    scales = {k: (g['pix'] / g[k] if g[k] > 0 else 1.0) for k in terms}
    w = cfg.LOSS_WEIGHTS
    share = {k: w[k] / sum(w.values()) for k in terms}
    raw = {k: w[k] * g[k] / sum(w[j] * g[j] for j in terms) for k in terms}
    write_json(dict(key=key, grad_norm_at_input=g, scales=scales, gradient_share_uncalibrated=raw,
                    gradient_share_calibrated=share, n_patches=len(ds)), path)
    print('[loss] calibrated scales ' + ', '.join(f'{k} {v:.3g}' for k, v in scales.items()) +
          ' | gradient share without calibration ' + ', '.join(f'{k} {100 * v:.0f}%' for k, v in raw.items()))
    del crit
    return scales


class Trainer:
    def __init__(self, cfg, run, manifest, device=None):
        self.cfg, self.run = cfg, run
        self.run_id = run['run_id']
        self.device = torch.device(device or (cfg.DEVICE if torch.cuda.is_available() else 'cpu'))
        self.ckdir, self.rundir = cfg.ckpt_dir(self.run_id), cfg.run_dir(self.run_id)
        os.makedirs(self.ckdir, exist_ok=True)
        os.makedirs(self.rundir, exist_ok=True)
        self.json_path = os.path.join(self.rundir, 'run.json')
        self.record = read_json(self.json_path, {})
        if self.record.get('status') == 'complete':
            return
        seed_everything(run['seed'], cfg.DETERMINISTIC)
        torch.backends.cudnn.benchmark = cfg.CUDNN_BENCHMARK and not cfg.DETERMINISTIC

        self.manifest = manifest
        self.train_ds = UniversalRestorationDataset(manifest['train'], cfg.PATCH_SIZE, True, FAMILIES)
        self.val_ds = UniversalRestorationDataset(manifest['val'], cfg.PATCH_SIZE, False, FAMILIES)
        self.ipe = len(self.train_ds) // cfg.BATCH_SIZE
        self.total_iters = int(run['epochs'] * self.ipe)

        self.model = build_model(cfg, run).to(self.device)
        self.amp_dtype = torch.bfloat16 if cfg.AMP_DTYPE == 'bfloat16' else torch.float16
        self.use_amp = bool(cfg.AMP and self.device.type == 'cuda')
        self.loss_scales = loss_scales(cfg, manifest, self.device)
        self.criterion = OmniRestorationLoss(cfg.loss_weights(run), amp=self.use_amp, amp_dtype=self.amp_dtype,
                                             scales=self.loss_scales).to(self.device).eval()
        params = [p for p in self.model.parameters() if p.requires_grad]
        self.lr = float(run.get('lr', cfg.LR))
        self.warmup = int(run.get('warmup', cfg.WARMUP_ITERS))
        self.optimizer = torch.optim.Adam(params, lr=self.lr, betas=tuple(cfg.BETAS), weight_decay=cfg.WEIGHT_DECAY)
        floor = cfg.ETA_MIN / self.lr
        T, W = max(1, self.total_iters), min(self.warmup, max(0, self.total_iters - 1))

        def lr_factor(it):   # linear warm-up over W iterations, then cosine from lr to ETA_MIN over the rest
            if it < W:
                return (it + 1) / W
            return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(it - W, T - W) / max(1, T - W)))
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, lr_factor)
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.use_amp and self.amp_dtype == torch.float16)
        self.metrics = MetricSuite(self.device, ('psnr', 'ssim'))

        self.iteration, self.best_val, self.best_iter, self.train_seconds = 0, -1.0, 0, 0.0
        self.history = dict(train=[], val=[])
        self.skipped_steps = 0
        self.init_from = None
        self._resume()
        if self.iteration == 0 and run.get('init_from'):
            self._initialise_from(run['init_from'])

    # ------------------------------------------------------------------ io
    def _ckpt(self, name):
        return os.path.join(self.ckdir, name)

    def _state(self, full=True):
        s = dict(run_id=self.run_id, iteration=self.iteration, total_iters=self.total_iters,
                 manifest_signature=self.manifest['signature'],
                 model_state_dict=self.model.state_dict(), model_flags=self.cfg.model_kwargs(self.run),
                 loss_weights=self.cfg.loss_weights(self.run), best_val_psnr=self.best_val, best_iter=self.best_iter,
                 init_from=self.init_from)
        if full:
            s.update(optimizer_state_dict=self.optimizer.state_dict(), scheduler_state_dict=self.scheduler.state_dict(),
                     scaler_state_dict=self.scaler.state_dict(), history=self.history, train_seconds=self.train_seconds,
                     skipped_steps=self.skipped_steps,
                     rng=dict(python=random.getstate(), numpy=np.random.get_state(), torch=torch.get_rng_state(),
                              cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None))
        return s

    def _resume(self):
        path = self._ckpt('last.pth')
        if not os.path.exists(path):
            return
        ck = torch.load(path, map_location=self.device, weights_only=False)
        prev = ck.get('manifest_signature') or self.record.get('data', {}).get('manifest_signature')
        if prev and prev != self.manifest['signature']:
            # the training pool changed: resuming would mix two pools inside one run -> restart this run
            for n in ('last.pth', 'best.pth'):
                if os.path.exists(self._ckpt(n)):
                    os.replace(self._ckpt(n), self._ckpt(n.replace('.pth', '_previous_pool.pth')))
            print(f'[{self.run_id}] !! the training pool changed since this checkpoint (iteration {ck["iteration"]}): '
                  f'the run restarts from scratch so that it is trained on one pool only (old checkpoint kept as last_previous_pool.pth)')
            self.record = {}
            return
        if ck.get('total_iters') != self.total_iters:
            print(f'[{self.run_id}] !! the budget changed since the checkpoint ({ck.get("total_iters")} -> {self.total_iters} '
                  f'iterations, e.g. EPOCHS or the training pool changed); the cosine schedule follows the new budget')
        self.model.load_state_dict(ck['model_state_dict'])
        self.optimizer.load_state_dict(ck['optimizer_state_dict'])
        self.scheduler.load_state_dict(ck['scheduler_state_dict'])
        self.scaler.load_state_dict(ck['scaler_state_dict'])
        self.iteration, self.best_val, self.best_iter = ck['iteration'], ck['best_val_psnr'], ck['best_iter']
        self.history, self.train_seconds = ck['history'], ck.get('train_seconds', 0.0)
        self.skipped_steps = ck.get('skipped_steps', 0)
        self.init_from = ck.get('init_from')
        r = ck.get('rng')
        if r:
            random.setstate(r['python'])
            np.random.set_state(r['numpy'])
            torch.set_rng_state(r['torch'].cpu())
            if r.get('cuda') is not None and torch.cuda.is_available():
                torch.cuda.set_rng_state_all([t.cpu() for t in r['cuda']])
        print(f'[{self.run_id}] resumed at iteration {self.iteration}/{self.total_iters}')

    def _initialise_from(self, src_id):
        """Warm restart: start from the selected weights of another (complete) run with the same model."""
        path = os.path.join(self.cfg.ckpt_dir(src_id), 'final.pth')
        if not os.path.exists(path):
            raise FileNotFoundError(f'{self.run_id} starts from {src_id}, whose final.pth does not exist yet')
        ck = torch.load(path, map_location=self.device, weights_only=False)
        if ck.get('manifest_signature') not in (None, self.manifest['signature']):
            raise RuntimeError(f'{src_id} was trained on another training pool than the current manifest')
        self.model.load_state_dict(ck['model_state_dict'], strict=True)
        self.init_from = dict(run_id=src_id, selected=ck.get('selected'))
        print(f'[{self.run_id}] initialised from {src_id} ({ck.get("selected")}); warm restart at lr {self.lr:g}, '
              f'warm-up {self.warmup} iterations')

    # ---------------------------------------------------------- validation
    @torch.no_grad()
    def validate(self):
        if len(self.val_ds) == 0:
            return dict(psnr=float('nan'), ssim=float('nan'), per_category={})
        self.model.eval()
        loader = DataLoader(self.val_ds, batch_size=8, shuffle=False, num_workers=min(4, self.cfg.NUM_WORKERS))
        per_cat, ps, ss = {}, [], []
        inv = {v: k for k, v in self.val_ds.task_to_id.items()}
        for b in loader:
            out, _ = self.model(b['input'].to(self.device))
            m = self.metrics(out, b['gt'])
            ps += m['psnr']
            ss += m['ssim']
            for t, p in zip(b['task_id'].tolist(), m['psnr']):
                per_cat.setdefault(inv[t], []).append(p)
        self.model.train()
        return dict(psnr=float(np.mean(ps)), ssim=float(np.mean(ss)), per_category={k: float(np.mean(v)) for k, v in per_cat.items()})

    # ---------------------------------------------------------------- curves
    def _write_curves(self):
        rows = self.history['train']
        if rows:
            with open(os.path.join(self.rundir, 'history_train.csv'), 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=list(rows[-1].keys()))
                w.writeheader()
                w.writerows(rows)
        vrows = [{k: v for k, v in r.items() if k != 'per_category'} for r in self.history['val']]
        if vrows:
            with open(os.path.join(self.rundir, 'history_val.csv'), 'w', newline='') as f:
                w = csv.DictWriter(f, fieldnames=list(vrows[-1].keys()))
                w.writeheader()
                w.writerows(vrows)
        try:
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
            it = [r['iteration'] for r in rows]
            for k in ('total', 'pix', 'fft', 'perc', 'ssim', 'id', 'edge'):
                if rows and k in rows[0]:
                    ax[0].plot(it, [r[k] for r in rows], label=k, lw=1)
            ax[0].set_yscale('log'); ax[0].set_title('training loss terms (unweighted) and total'); ax[0].legend(fontsize=7)
            ax[1].plot(it, [r['train_psnr'] for r in rows], label='train (random patches)', lw=1)
            ax[1].plot([r['iteration'] for r in vrows], [r['psnr'] for r in vrows], 'o-', label='validation (held-out)', ms=3)
            ax[1].set_title('PSNR (dB)'); ax[1].legend()
            ax[2].plot(it, [r['lr'] for r in rows]); ax[2].set_title('learning rate')
            for a in ax:
                a.set_xlabel('iteration'); a.grid(alpha=.3)
            fig.suptitle(self.run_id)
            fig.tight_layout()
            fig.savefig(os.path.join(self.rundir, 'curves.png'), dpi=120)
            plt.close(fig)
        except Exception as e:
            print('[curves] skipped:', e)

    def _write_record(self, status):
        rec = dict(self.record)
        rec.update(run_id=self.run_id, group=self.run['group'], seed=self.run['seed'], status=status,
                   budget=dict(epochs=self.run['epochs'], iters_per_epoch=self.ipe, total_iters=self.total_iters,
                               iterations_done=self.iteration, batch_size=self.cfg.BATCH_SIZE, patch=self.cfg.PATCH_SIZE,
                               samples_seen=self.iteration * self.cfg.BATCH_SIZE),
                   optimisation=dict(optimizer='Adam', lr=self.lr, betas=list(self.cfg.BETAS), weight_decay=self.cfg.WEIGHT_DECAY,
                                     warmup_iters=self.warmup, initialised_from=self.init_from,
                                     schedule=(f'linear warm-up {self.warmup} it, then ' if self.warmup else '') +
                                              f'cosine per iteration, {self.lr} -> {self.cfg.ETA_MIN}', grad_clip=self.cfg.GRAD_CLIP,
                                     amp=self.use_amp, amp_dtype=str(self.amp_dtype), skipped_nonfinite_steps=self.skipped_steps),
                   model_flags=self.cfg.model_kwargs(self.run), loss_weights=self.cfg.loss_weights(self.run),
                   loss_scales=self.loss_scales,
                   parameters_M=component_parameter_table(self.model),
                   data=dict(n_train_items=len(self.manifest['train']), n_train_samples_per_epoch=len(self.train_ds),
                             n_val=len(self.val_ds), manifest_signature=self.manifest['signature'],
                             categories={k: v.get('n_train') for k, v in self.manifest['categories'].items()}),
                   selection=dict(rule=self.cfg.MODEL_SELECTION, best_val_psnr=self.best_val, best_iter=self.best_iter,
                                  stopping='fixed iteration budget'),
                   wall_clock_hours=round(self.train_seconds / 3600, 3),
                   checkpoint=self._ckpt('final.pth') if status == 'complete' else self._ckpt('last.pth'),
                   environment=environment_info())
        if self.history['val']:
            rec['last_val'] = self.history['val'][-1]
        self.record = rec
        write_json(rec, self.json_path)

    # ------------------------------------------------------------------ fit
    def fit(self):
        if self.record.get('status') == 'complete':
            print(f'[{self.run_id}] already complete -> skipped')
            return self.record
        cfg = self.cfg
        print(f'[{self.run_id}] {len(self.train_ds)} samples/epoch, {self.ipe} it/epoch, budget {self.total_iters} it '
              f'({self.run["epochs"]} epochs), batch {cfg.BATCH_SIZE}, AMP={self.use_amp}')
        session_start = time.time()
        self._t_base, self._t_start = self.train_seconds, session_start
        self.model.train()
        acc, n_acc, t_last = {}, 0, time.time()
        while self.iteration < self.total_iters:
            epoch, pos = divmod(self.iteration, self.ipe)
            g = torch.Generator().manual_seed(self.run['seed'] * 7919 + epoch * 131 + pos)
            loader = DataLoader(self.train_ds, batch_size=cfg.BATCH_SIZE,
                                sampler=ResumableSampler(len(self.train_ds), self.run['seed'], epoch, pos * cfg.BATCH_SIZE),
                                num_workers=cfg.NUM_WORKERS, pin_memory=cfg.PIN_MEMORY and self.device.type == 'cuda',
                                drop_last=True, generator=g, persistent_workers=False)
            for batch in loader:
                inputs = batch['input'].to(self.device, non_blocking=True)
                targets = batch['gt'].to(self.device, non_blocking=True)
                self.optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=self.device.type, dtype=self.amp_dtype, enabled=self.use_amp):
                    restored, _ = self.model(inputs)
                loss, logs = self.criterion(restored, targets)
                if not torch.isfinite(loss):
                    self.skipped_steps += 1
                    if self.skipped_steps > 1000:
                        raise RuntimeError('too many non-finite losses')
                    self.iteration += 1
                    self.scheduler.step()
                    continue
                self.scaler.scale(loss).backward()
                self.scaler.unscale_(self.optimizer)
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=cfg.GRAD_CLIP)
                self.scaler.step(self.optimizer)
                self.scaler.update()
                self.scheduler.step()
                self.iteration += 1
                with torch.no_grad():
                    logs['train_psnr'] = float(psnr_torch(restored.detach(), targets).mean())
                for k, v in logs.items():
                    acc[k] = acc.get(k, 0.0) + v
                n_acc += 1
                self.train_seconds = self._t_base + (time.time() - self._t_start)  # wall-clock incl. data loading, validation, checkpoints

                if self.iteration % cfg.LOG_EVERY_ITERS == 0:
                    row = {k: v / n_acc for k, v in acc.items()}
                    row.update(iteration=self.iteration, epoch=self.iteration / self.ipe, lr=self.scheduler.get_last_lr()[0],
                               sec_per_iter=(time.time() - t_last) / n_acc)
                    self.history['train'].append(row)
                    eta_h = row['sec_per_iter'] * (self.total_iters - self.iteration) / 3600
                    print(f'[{self.run_id}] it {self.iteration}/{self.total_iters} ep {row["epoch"]:.2f} loss {row["total"]:.4f} '
                          f'psnr {row["train_psnr"]:.2f} lr {row["lr"]:.2e} {row["sec_per_iter"]:.3f}s/it ETA {eta_h:.1f}h')
                    acc, n_acc, t_last = {}, 0, time.time()
                if self.iteration % cfg.VAL_EVERY_ITERS == 0 or self.iteration == self.total_iters:
                    self._validate_and_select()
                if self.iteration % cfg.CKPT_EVERY_ITERS == 0:
                    atomic_save(self._state(), self._ckpt('last.pth'))
                    write_json(dict(session=SESSION_ID, host=platform.node(), heartbeat=time.time()), lock_path(cfg, self.run_id))
                    self._write_record('running')
                    self._write_curves()
                if self.iteration >= self.total_iters:
                    break
                if (time.time() - session_start) / 3600 > cfg.MAX_SESSION_HOURS:
                    atomic_save(self._state(), self._ckpt('last.pth'))
                    self._write_record('paused')
                    self._write_curves()
                    print(f'[{self.run_id}] session time limit reached -> paused at {self.iteration}; re-run to resume')
                    return self.record
        if not self.history['val'] or self.history['val'][-1]['iteration'] != self.iteration:
            self._validate_and_select()
        self._finalise()
        return self.record

    def _validate_and_select(self):
        v = self.validate()
        v['iteration'] = self.iteration
        self.history['val'].append(v)
        print(f'[{self.run_id}] validation @ {self.iteration}: PSNR {v["psnr"]:.3f} SSIM {v["ssim"]:.4f}')
        if v['psnr'] == v['psnr'] and v['psnr'] > self.best_val:
            self.best_val, self.best_iter = v['psnr'], self.iteration
            atomic_save(self._state(full=False), self._ckpt('best.pth'))

    def _finalise(self):
        """Keep only the selected weights as final.pth (weights only) to save Drive space."""
        if self.cfg.MODEL_SELECTION == 'best_val' and os.path.exists(self._ckpt('best.pth')):
            sel = torch.load(self._ckpt('best.pth'), map_location='cpu', weights_only=False)
            sel['selected'] = f'best_val @ iteration {self.best_iter}'
        else:
            sel = self._state(full=False)
            sel['selected'] = f'last @ iteration {self.iteration}'
        sel['training_complete'] = True
        atomic_save(sel, self._ckpt('final.pth'))
        self._write_curves()
        self._write_record('complete')
        self.record['selection']['selected'] = sel['selected']
        write_json(self.record, self.json_path)
        for n in ('last.pth', 'best.pth'):
            if os.path.exists(self._ckpt(n)):
                os.remove(self._ckpt(n))
        print(f'[{self.run_id}] complete: {sel["selected"]}, {self.train_seconds / 3600:.2f} h of training')

In [ ]:
# ===== Inferencer.py (identical to Inferencer.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - Inferencer: benchmark evaluation + explainability suite
# --------------------------------------------------------------------------
# Evaluator.evaluate() restores every image of a benchmark at full resolution
# (batch 1, FP32), scores it with metrics.MetricSuite (the single metric
# function) and writes runs/<run_id>/eval/<benchmark>/per_image.csv with
#   image, key, PSNR, SSIM, LPIPS, DISTS, input PSNR/SSIM, H, W, latency
# plus summary.json; the run.json of the run receives the benchmark means.
# Competitor outputs held by the authors are scored with the same function.
# KASIRNetInferencer (explainability figures of the paper) is kept from the
# submitted notebook and runs on a few images per benchmark.
# ==========================================================================
import os
import csv
import glob
import json
import math
import time
import hashlib
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
from PIL import Image
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt



def save_png(t, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    arr = (t.detach().float().clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255.0 + 0.5).astype(np.uint8)
    Image.fromarray(arr).save(path)


def write_rows(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not rows:
        return
    keys = list(rows[0].keys())
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)


TEXT_FIELDS = {'image', 'key', 'gt_name', 'benchmark', 'method', 'run_id', 'competitor', 'metric', 'variant', 'level', 'family'}


def read_rows(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        rows = list(csv.DictReader(f))
    for r in rows:
        for k, v in r.items():
            if k in TEXT_FIELDS:
                continue
            try:
                r[k] = float(v)
            except (TypeError, ValueError):
                pass
    return rows


# ==========================================================================
# Inference protocols (Config.INFERENCE_PROTOCOL)
# ==========================================================================
def parse_protocol(name):
    """'full' | 'tiled<P>' with optional '+x8' -> (mode, tile, self_ensemble)."""
    base, _, ens = name.partition('+')
    if base == 'full':
        return 'full', None, ens == 'x8'
    if base.startswith('tiled'):
        return 'tiled', int(base[5:] or 256), ens == 'x8'
    raise ValueError(f'unknown inference protocol {name!r}')


def sin2_window(P, device):
    """w(n) = sin^2(pi (n + 1/2) / P): w(n) + w(n + P/2) = 1, i.e. overlap-add with hop P/2 is an exact
    partition of unity (Princen-Bradley condition) in each axis; the 2-D window is the outer product.
    It also vanishes at the tile borders, where the SRB places its bias impulse and the cyclic shift wraps."""
    n = torch.arange(P, device=device, dtype=torch.float32)
    w = torch.sin(math.pi * (n + 0.5) / P) ** 2
    return w[:, None] * w[None, :]


def _full_forward(model, x, x_prompt=None):
    override = None
    if x_prompt is not None and getattr(model, 'use_prompt', False):
        override = model.prompt_learner(model.check_image_size(x_prompt)[0])
    return model(x, prompt_override=override)


def tiled_forward(model, x, P, batch=32, x_prompt=None):
    """Train-consistent inference: overlapping P x P tiles (hop P/2, sin^2 overlap-add).  The image is
    reflect-padded by P/2 (and up to a multiple of P/2) so that every tile is a full P x P input and every
    pixel is covered by exactly 2 x 2 tiles whose weights sum to 1.  Images not larger than P go through
    the model directly (that already is the training regime).  Returns (output, mean descriptor)."""
    _, C, H, W = x.shape
    if H <= P and W <= P:
        return _full_forward(model, x, x_prompt)
    h = P // 2
    Hp, Wp = math.ceil((H + 2 * h) / h) * h, math.ceil((W + 2 * h) / h) * h
    pads = (h, Wp - W - h, h, Hp - H - h)
    mode = 'reflect' if min(H, W) > max(pads) else 'replicate'
    xp = F.pad(x, pads, mode=mode)
    xq = F.pad(x_prompt, pads, mode=mode) if x_prompt is not None else None
    win = sin2_window(P, x.device)
    acc = torch.zeros((1, C, Hp, Wp), device=x.device, dtype=torch.float32)
    pos = [(i, j) for i in range(0, Hp - P + 1, h) for j in range(0, Wp - P + 1, h)]
    prompts = []
    for k in range(0, len(pos), batch):
        chunk = pos[k:k + batch]
        tiles = torch.cat([xp[:, :, i:i + P, j:j + P] for i, j in chunk])
        tq = torch.cat([xq[:, :, i:i + P, j:j + P] for i, j in chunk]) if xq is not None else None
        y, p = _full_forward(model, tiles, tq)
        for t, (i, j) in enumerate(chunk):
            acc[:, :, i:i + P, j:j + P] += y[t:t + 1].float() * win
        if p is not None:
            prompts.append(p.float())
    out = acc[:, :, h:h + H, h:h + W].clamp(0, 1)
    return out, (torch.cat(prompts).mean(0, keepdim=True) if prompts else None)


_DIHEDRAL = [(k, f) for f in (False, True) for k in range(4)]


def _d4(t, k, f):
    t = torch.flip(t, dims=[3]) if f else t
    return torch.rot90(t, k, dims=[2, 3])


def _d4_inv(t, k, f):
    t = torch.rot90(t, -k, dims=[2, 3])
    return torch.flip(t, dims=[3]) if f else t


@torch.no_grad()
def run_model(model, x, protocol='full', x_prompt=None, tile_batch=32):
    """Restored image and descriptor of `x` under an inference protocol (weights untouched)."""
    mode, P, ens = parse_protocol(protocol)
    fwd = (lambda a, q: tiled_forward(model, a, P, tile_batch, q)) if mode == 'tiled' else (lambda a, q: _full_forward(model, a, q))
    if not ens:
        return fwd(x, x_prompt)
    outs, ps = [], []
    for k, f in _DIHEDRAL:
        y, p = fwd(_d4(x, k, f), _d4(x_prompt, k, f) if x_prompt is not None else None)
        outs.append(_d4_inv(y.float(), k, f))
        if p is not None:
            ps.append(p.float())
    return torch.stack(outs).mean(0), (torch.stack(ps).mean(0) if ps else None)


# ==========================================================================
# Registration before scoring (benchmarks with align='ecc', i.e. Koehler)
# ==========================================================================
def ecc_align(pred, gt):
    """RealBlur evaluation protocol (Rim et al., ECCV 2020): least-squares intensity scaling of `pred`
    to `gt`, ECC homography (OpenCV, gaussFiltSize 5) of `pred` onto `gt`, then both images restricted
    to the region covered by the warped image (here: its largest centred inner rectangle, so that every
    metric of the single metric function can be applied).  Tensors 1x3xHxW in [0, 1]."""
    import cv2
    z = pred[0].permute(1, 2, 0).float().cpu().numpy().astype(np.float32)
    x = gt[0].permute(1, 2, 0).float().cpu().numpy().astype(np.float32)
    zs = np.clip(float((x * z).sum() / max((z * z).sum(), 1e-8)) * z, 0, 1).astype(np.float32)
    warp = np.eye(3, dtype=np.float32)
    ok = True
    try:
        _, warp = cv2.findTransformECC(cv2.cvtColor(x, cv2.COLOR_RGB2GRAY), cv2.cvtColor(zs, cv2.COLOR_RGB2GRAY), warp,
                                       cv2.MOTION_HOMOGRAPHY, (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 100, 0), None, 5)
    except cv2.error:
        ok, warp = False, np.eye(3, dtype=np.float32)
    H, W = x.shape[:2]
    zr = cv2.warpPerspective(zs, warp, (W, H), flags=cv2.INTER_CUBIC + cv2.WARP_INVERSE_MAP, borderMode=cv2.BORDER_REFLECT)
    cr = cv2.warpPerspective(np.ones((H, W), np.float32), warp, (W, H), flags=cv2.INTER_NEAREST + cv2.WARP_INVERSE_MAP,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    t, b, l, r = 0, H, 0, W   # shrink a centred rectangle until it lies inside the valid region
    while t < b - 16 and l < r - 16 and cr[t:b, l:r].min() < 1:
        t, b, l, r = t + 1, b - 1, l + 1, r - 1
    to_t = lambda a: torch.from_numpy(np.ascontiguousarray(np.clip(a[t:b, l:r], 0, 1))).permute(2, 0, 1).unsqueeze(0)
    return to_t(zr), to_t(x), dict(ecc_converged=ok, crop=(t, H - b, l, W - r))


def _source_desc(cfg, key):
    for attr in ('KAGGLE_SOURCES', 'HF_ZIP_SOURCES', 'HF_PARQUET_SOURCES', 'GDOWN_SOURCES'):
        d = getattr(cfg, attr, None) or {}
        if key in d:
            return [attr, d[key]]
    return [key]


def eval_signature(cfg, bench, protocol, extra=None):
    """Everything a stored result depends on: benchmark definition and its data sources, inference
    protocol, registration, metric protocol.  A stored summary with another signature is recomputed."""
    b = cfg.benchmarks()
    spec = b[bench]
    srcs = sorted(set(spec_sources(spec, b)))
    blob = [bench, spec, b.get(spec.get('base')), [_source_desc(cfg, k) for k in srcs], protocol, cfg.SYNTHETIC_SEED,
            METRIC_PROTOCOL_VERSION, extra]
    return hashlib.md5(json.dumps(blob, sort_keys=True, default=str).encode()).hexdigest()


METRIC_PROTOCOL_VERSION = 2   # 2: per-benchmark registration (Koehler), inference protocols


def _centre_crop(t, max_hw):
    H, W = t.shape[-2:]
    mh, mw = max_hw if H <= W else max_hw[::-1]
    h, w = min(H, mh), min(W, mw)
    t0, l0 = (H - h) // 2, (W - w) // 2
    return t[..., t0:t0 + h, l0:l0 + w]


@torch.no_grad()
def select_inference_protocol(cfg, model, val_items, device, out_dir):
    """Chooses the inference protocol on the held-out validation pairs of the training pool at full
    resolution (centre crop to INFERENCE_SELECT_MAX_SIZE), never on a test set.  Criterion: mean over
    categories of the per-category mean PSNR; 'full' is kept unless another protocol gains at least
    INFERENCE_SELECT_MIN_GAIN dB.  Writes selection.json and a per-category table."""
    suite = MetricSuite(device, ('psnr', 'ssim'))
    per = {c: {} for c in cfg.INFERENCE_CANDIDATES}
    sizes = []
    for it in tqdm(val_items, desc='inference protocol selection', leave=False):
        x = _centre_crop(TF.to_tensor(Image.open(it['input']).convert('RGB')), cfg.INFERENCE_SELECT_MAX_SIZE).unsqueeze(0)
        g = _centre_crop(TF.to_tensor(Image.open(it['gt']).convert('RGB')), cfg.INFERENCE_SELECT_MAX_SIZE).unsqueeze(0)
        if x.shape != g.shape:
            continue
        sizes.append(x.shape[-2] * x.shape[-1])
        x = x.to(device)
        for c in cfg.INFERENCE_CANDIDATES:
            y, _ = run_model(model, x, c, tile_batch=cfg.TILE_BATCH)
            m = suite(y, g)
            per[c].setdefault(it['category'], []).append((m['psnr'][0], m['ssim'][0]))
    rows, score = [], {}
    cats = sorted({k for c in per for k in per[c]})
    for c in cfg.INFERENCE_CANDIDATES:
        row = dict(protocol=c)
        for k in cats:
            row[k] = float(np.mean([v[0] for v in per[c].get(k, [])])) if per[c].get(k) else None
        score[c] = float(np.mean([row[k] for k in cats if row[k] is not None])) if cats else float('nan')
        row['mean_over_categories_psnr'] = score[c]
        row['mean_ssim'] = float(np.mean([v[1] for k in cats for v in per[c].get(k, [])])) if cats else None
        rows.append(row)
    best = max(score, key=lambda c: score[c])
    chosen = 'full' if ('full' in score and score[best] - score['full'] < cfg.INFERENCE_SELECT_MIN_GAIN) else best
    os.makedirs(out_dir, exist_ok=True)
    write_rows(rows, os.path.join(out_dir, 'validation_per_category.csv'))
    sel = dict(selected=chosen, scores=score, criterion='mean over categories of the mean PSNR on the held-out validation '
               'pairs (full resolution, centre crop to %dx%d), never a test set' % tuple(cfg.INFERENCE_SELECT_MAX_SIZE),
               n_images=len(sizes), median_pixels=float(np.median(sizes)) if sizes else None, candidates=cfg.INFERENCE_CANDIDATES)
    write_json(sel, os.path.join(out_dir, 'selection.json'))
    print('[protocol] validation PSNR: ' + ', '.join(f'{c} {v:.3f}' for c, v in score.items()) + f'  -> {chosen}')
    return sel


class Evaluator:
    def __init__(self, cfg, roots, device=None, metric_names=('psnr', 'ssim', 'lpips', 'dists', 'psnr_y', 'ssim_y')):
        self.cfg, self.roots = cfg, roots
        self.device = torch.device(device or (cfg.DEVICE if torch.cuda.is_available() else 'cpu'))
        self.metric_names = tuple(metric_names)
        self.metrics = MetricSuite(self.device, self.metric_names)
        self._light = None
        # inference protocol of every evaluation (main.Experiment sets the selected one when 'auto')
        self.protocol = 'full' if cfg.INFERENCE_PROTOCOL == 'auto' else cfg.INFERENCE_PROTOCOL

    @property
    def light(self):  # PSNR/SSIM only, for inputs
        if self._light is None:
            self._light = MetricSuite(self.device, ('psnr', 'ssim'))
        return self._light

    # ------------------------------------------------------------- models
    def load_run_model(self, run):
        model = build_model(self.cfg, run)
        path = os.path.join(self.cfg.ckpt_dir(run['run_id']), 'final.pth')
        if not os.path.exists(path):
            raise FileNotFoundError(f'{run["run_id"]}: no final.pth (training not complete)')
        load_weights(model, path)
        return model.to(self.device).eval()

    def bench_dir(self, run_id, bench, tag=''):
        return os.path.join(self.cfg.run_dir(run_id), 'eval', bench + tag)

    def signature(self, bench, tag=''):
        return eval_signature(self.cfg, bench, self.protocol, tag)

    def is_done(self, run_id, bench, tag=''):
        """A stored result counts only if it was computed with the current data, protocol and metrics."""
        summ = read_json(os.path.join(self.bench_dir(run_id, bench, tag), 'summary.json'))
        return bool(summ) and summ.get('eval_signature') == self.signature(bench, tag)

    def prepare(self, bench, out, gt):
        """Pair actually scored: registered to the reference for benchmarks with align='ecc'."""
        if self.cfg.benchmarks()[bench].get('align') == 'ecc':
            a, g, info = ecc_align(out, gt)
            return a, g, info
        return out, gt, None

    # ----------------------------------------------------------- evaluate
    @torch.no_grad()
    def evaluate(self, model, run_id, bench, prompt_mode=None, save_images=False, collect_prompts=False,
                 max_images=None, metric_names=None, force=False, protocol=None):
        """prompt_mode=None: normal inference.  prompt_mode='first': for mixed benchmarks the
        descriptor p is replaced by the descriptor of the singly-degraded image (control)."""
        tag = ('' if prompt_mode is None else f'__prompt_{prompt_mode}') + \
              ('' if protocol in (None, self.protocol) else f'__protocol_{protocol}')
        if protocol in (None, self.protocol):
            return self._evaluate(model, run_id, bench, tag, prompt_mode, save_images, collect_prompts, max_images, metric_names, force)
        saved, self.protocol = self.protocol, protocol   # evaluation under another protocol (reported only)
        try:
            return self._evaluate(model, run_id, bench, tag, prompt_mode, False, collect_prompts, max_images, metric_names, force)
        finally:
            self.protocol = saved

    def _evaluate(self, model, run_id, bench, tag, prompt_mode, save_images, collect_prompts, max_images, metric_names, force):
        out_dir = self.bench_dir(run_id, bench, tag)
        if self.is_done(run_id, bench, tag) and not force and not collect_prompts:
            return read_json(os.path.join(out_dir, 'summary.json'))
        names = tuple(metric_names or self.metric_names)
        suite = self.metrics if set(names) <= set(self.metric_names) else MetricSuite(self.device, names)
        ds = BenchmarkDataset(self.cfg, self.roots, bench, max_images=max_images)
        loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=min(6, self.cfg.NUM_WORKERS), collate_fn=collate_single)
        rows, prompts, n_resized, n_unaligned = [], [], 0, 0
        amp = bool(self.cfg.EVAL_AMP and self.device.type == 'cuda')
        torch.backends.cudnn.benchmark = False   # test images have many different sizes
        for item in tqdm(loader, desc=f'{run_id} | {bench}{tag}', leave=False):
            x = item['input'].unsqueeze(0).to(self.device)
            gt = item['gt'].unsqueeze(0)
            xf = None
            if prompt_mode == 'first' and 'first' in item and getattr(model, 'use_prompt', False):
                xf = item['first'].unsqueeze(0).to(self.device)
            if self.device.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            with torch.autocast(device_type=self.device.type, enabled=amp):
                out, p = run_model(model, x, self.protocol, x_prompt=xf, tile_batch=self.cfg.TILE_BATCH)
            if self.device.type == 'cuda':
                torch.cuda.synchronize()
            ms = (time.time() - t0) * 1000
            out_s, gt_s, info = self.prepare(bench, out.float(), gt.to(out.device))
            in_s, gt_i, _ = self.prepare(bench, x.float(), gt.to(x.device))
            m = suite(out_s, gt_s)
            mi = self.light(in_s, gt_i)
            if info is not None:
                n_unaligned += int(not info['ecc_converged'])
            n_resized += int(bool(item.get('gt_resized')))
            row = dict(image=item['name'], key=item['key'])
            for k in names:
                row[k] = m[k][0]
            row.update(input_psnr=mi['psnr'][0], input_ssim=mi['ssim'][0], H=x.shape[2], W=x.shape[3], ms=round(ms, 2))
            rows.append(row)
            if collect_prompts and p is not None:
                prompts.append(p.float().cpu().numpy()[0])
            if save_images:
                save_png(out[0], os.path.join(out_dir, 'restored', os.path.splitext(item['name'])[0] + '.png'))
        write_rows(rows, os.path.join(out_dir, 'per_image.csv'))
        summ = dict(run_id=run_id, benchmark=bench, prompt_mode=prompt_mode, n_images=len(rows), n_gt_resized=n_resized,
                    inference_protocol=self.protocol, eval_signature=self.signature(bench, tag),
                    registration=self.cfg.benchmarks()[bench].get('align'), n_ecc_not_converged=n_unaligned,
                    dataset_report=ds.report)
        for k in list(names) + ['input_psnr', 'input_ssim']:
            v = [r[k] for r in rows]
            summ[k] = float(np.mean(v)) if v else None
            summ[k + '_std'] = float(np.std(v)) if v else None
        summ['mean_ms'] = float(np.mean([r['ms'] for r in rows[1:]])) if len(rows) > 1 else None
        write_json(summ, os.path.join(out_dir, 'summary.json'))
        if collect_prompts and prompts:
            np.save(os.path.join(out_dir, 'prompts.npy'), np.stack(prompts))
        rj = os.path.join(self.cfg.run_dir(run_id), 'run.json')
        rec = read_json(rj, {})
        rec.setdefault('eval', {})[bench + tag] = {k: summ[k] for k in summ if k not in ('dataset_report',)}
        write_json(rec, rj)
        print(f'[eval] {run_id:22s} {bench + tag:28s} n={len(rows):5d}  PSNR {summ["psnr"]:.2f}  SSIM {summ["ssim"]:.3f}'
              f'  (input {summ["input_psnr"]:.2f}/{summ["input_ssim"]:.3f})  [{self.protocol}'
              f'{", registered" if summ["registration"] else ""}]')
        return summ

    # --------------------------------------------------------- competitors
    @torch.no_grad()
    def score_folder(self, bench, method, folder, out_csv):
        """Score restored images of another method with the same metric function.
        Files are matched by input stem, ground-truth stem or pairing key."""
        sig = eval_signature(self.cfg, bench, 'competitor', [method, folder])
        if os.path.exists(out_csv) and read_json(out_csv.replace('.csv', '.json'), {}).get('eval_signature') == sig:
            return read_rows(out_csv)
        if not os.path.isdir(folder):
            print(f'[competitor] {method} on {bench}: folder {folder} missing -> skipped')
            return []
        files = {}
        for f in glob.glob(os.path.join(folder, '**', '*'), recursive=True):
            if f.lower().endswith(IMG_EXT):
                files.setdefault(os.path.splitext(os.path.basename(f))[0].lower(), f)
        ds = BenchmarkDataset(self.cfg, self.roots, bench)
        rows, missing, shape_mismatch = [], 0, 0
        for i in tqdm(range(len(ds)), desc=f'{method} | {bench}', leave=False):
            it = ds[i]
            cands = [os.path.splitext(it['name'])[0].lower(), os.path.splitext(it['gt_name'])[0].lower(), it['key'].lower()]
            f = next((files[c] for c in cands if c in files), None)
            if f is None:
                missing += 1
                continue
            r = TF.to_tensor(Image.open(f).convert('RGB'))
            if r.shape != it['gt'].shape:
                shape_mismatch += 1
                continue
            r_s, g_s, _ = self.prepare(bench, r.unsqueeze(0), it['gt'].unsqueeze(0))
            m = self.metrics(r_s, g_s)
            rows.append(dict(image=it['name'], key=it['key'], **{k: m[k][0] for k in self.metric_names}))
        write_rows(rows, out_csv)
        write_json(dict(method=method, benchmark=bench, folder=folder, n_scored=len(rows), n_missing=missing,
                        n_shape_mismatch=shape_mismatch, eval_signature=sig), out_csv.replace('.csv', '.json'))
        print(f'[competitor] {method:14s} {bench:10s} scored {len(rows)} (missing {missing}, shape mismatch {shape_mismatch})')
        return rows


# ==========================================================================
# Explainability & interpretability suite (figures of the paper), from the
# submitted notebook; now takes an already loaded model, skips modules that an
# ablated model does not have, and runs on a handful of images per benchmark.
# ==========================================================================

class KASIRNetInferencer:
    def __init__(self, model, output_root, metrics, explain_max_size=512, enable_occlusion=False):
        self.device = next(model.parameters()).device
        self.model = model.eval()
        self.metrics_calc = metrics
        self.output_root = output_root
        self.explain_max_size = explain_max_size
        self.enable_occlusion = enable_occlusion

        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32)
        self.sobel_x = sobel_x.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(self.device)
        self.sobel_y = sobel_y.view(1, 1, 3, 3).repeat(3, 1, 1, 1).to(self.device)

        self._acts = {}
        self._grads = {}
        self._hook_handles = []

        self.global_prompts = []
        self.global_metrics = {}

        os.makedirs(self.output_root, exist_ok=True)

    # ------------------------------------------------------------------
    # Hook Configuration
    # ------------------------------------------------------------------
    def _get_stage_modules(self):
        names = [('head', 'head_attn'), ('encoder1', 'encoder1'), ('encoder2', 'encoder2'), ('encoder3', 'encoder3'),
                 ('encoder4', 'encoder4'), ('bottleneck', 'bottleneck'), ('decoder4', 'decoder4'), ('decoder3', 'decoder3'),
                 ('decoder2', 'decoder2'), ('decoder1', 'decoder1')]
        return {k: getattr(self.model, a) for k, a in names if hasattr(self.model, a)}

    def _get_srb_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SpectralResonanceBlock)}

    def _get_sim_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SemanticInjection)}

    def _get_sft_modules(self):
        return {name: mod for name, mod in self.model.named_modules() if isinstance(mod, SFTLayer)}

    def _register_hooks(self, capture_grads=False):
        self._remove_hooks()
        self._acts, self._grads = {}, {}

        def make_fwd_hook(key):
            def hook(module, inp, out):
                t = out[0] if isinstance(out, (tuple, list)) else out
                if torch.is_tensor(t):
                    self._acts[key] = t
                    if capture_grads and t.requires_grad:
                        def grad_hook(grad, k=key):
                            self._grads[k] = grad.detach()
                        t.register_hook(grad_hook)
            return hook

        for name, module in self._get_stage_modules().items():
            self._hook_handles.append(module.register_forward_hook(make_fwd_hook(f'stage/{name}')))
        for name, module in self._get_srb_modules().items():
            self._hook_handles.append(module.resonance_gate.register_forward_hook(make_fwd_hook(f'gate/{name}')))
        for name, module in self._get_sim_modules().items():
            self._hook_handles.append(module.spatial_attn.register_forward_hook(make_fwd_hook(f'sim/{name}')))
            self._hook_handles.append(module.channel_attn.register_forward_hook(make_fwd_hook(f'simch/{name}')))
        for name, module in self._get_sft_modules().items():
            self._hook_handles.append(module.sft_scale.register_forward_hook(make_fwd_hook(f'sft_scale/{name}')))
            self._hook_handles.append(module.sft_shift.register_forward_hook(make_fwd_hook(f'sft_shift/{name}')))

    def _remove_hooks(self):
        for h in self._hook_handles: h.remove()
        self._hook_handles = []

    @torch.no_grad()
    def restore(self, input_tensor):
        input_tensor = input_tensor.to(self.device).unsqueeze(0)
        restored, prompt_emb = self.model(input_tensor)
        if prompt_emb is None:
            prompt_emb = torch.zeros(1, 256, device=restored.device)
        return torch.clamp(restored, 0, 1).squeeze(0).cpu(), prompt_emb.float().squeeze(0).cpu()

    def _resize_for_explain(self, tensor):
        _, H, W = tensor.shape
        m = max(H, W)
        if m <= self.explain_max_size: return tensor
        scale = self.explain_max_size / m
        new_h, new_w = max(16, int(H * scale) // 16 * 16), max(16, int(W * scale) // 16 * 16)
        return torch.nn.functional.interpolate(tensor.unsqueeze(0), size=(new_h, new_w), mode='bilinear', align_corners=False).squeeze(0)

    # ------------------------------------------------------------------
    # Utility Savers for Individual Image Extractions
    # ------------------------------------------------------------------
    @staticmethod
    def _to_np(t): return t.detach().permute(1, 2, 0).cpu().numpy().clip(0, 1)

    @staticmethod
    def _normalize_map(m): return (m - m.min()) / (m.max() - m.min() + 1e-8)

    def _overlay_heatmap(self, image_np, heat_np, alpha=0.5, cmap='jet'):
        heat = self._normalize_map(heat_np)
        # Fixed DeprecationWarning: Using plt.get_cmap instead of cm.get_cmap
        heat_rgb = plt.get_cmap(cmap)(heat)[..., :3]
        return (1 - alpha) * image_np + alpha * heat_rgb

    def _save_single_img(self, img_np, save_path, cmap=None):
        if img_np.ndim == 2:
            plt.imsave(save_path, img_np, cmap=cmap)
        else:
            plt.imsave(save_path, np.clip(img_np, 0, 1))

    # ------------------------------------------------------------------
    # V1. Comparison
    # ------------------------------------------------------------------
    def vis_comparison(self, input_t, restored_t, gt_t, metrics, save_path, task_name):
        panels = [self._to_np(input_t), self._to_np(restored_t)]
        titles = [f"Input ({task_name})", f"Restored (PSNR: {metrics.get('psnr','N/A')})"]
        if gt_t is not None:
            panels.append(self._to_np(gt_t))
            titles.append("Ground Truth")
            err = np.abs(self._to_np(restored_t) - self._to_np(gt_t)).mean(axis=2)
            panels.append(err)
            titles.append(f"|Error|")

        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
        for ax, img, title in zip(np.atleast_1d(axes), panels, titles):
            if img.ndim == 2:
                im = ax.imshow(img, cmap='inferno')
                plt.colorbar(im, ax=ax, fraction=0.046)
            else:
                ax.imshow(img)
            ax.set_title(title, fontsize=11)
            ax.axis('off')
        fig.tight_layout()
        plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(panels[0], os.path.join(base_dir, f"{stem}_input.png"))
        self._save_single_img(panels[1], os.path.join(base_dir, f"{stem}_restored.png"))
        if gt_t is not None:
            self._save_single_img(panels[2], os.path.join(base_dir, f"{stem}_gt.png"))
            self._save_single_img(panels[3], os.path.join(base_dir, f"{stem}_errormap.png"), cmap='inferno')

    # ------------------------------------------------------------------
    # V2. Error Maps
    # ------------------------------------------------------------------
    def vis_error_maps(self, restored_t, gt_t, save_path):
        if gt_t is None: return
        r, g = self._to_np(restored_t), self._to_np(gt_t)
        err = np.abs(r - g)

        with torch.no_grad():
            rt, gt_tnsr = restored_t.unsqueeze(0).to(self.device), gt_t.unsqueeze(0).to(self.device)
            rl, gl = rt.mean(dim=1, keepdim=True), gt_tnsr.mean(dim=1, keepdim=True)
            k, C1, C2 = 11, 0.01 ** 2, 0.03 ** 2
            mu_r = torch.nn.functional.avg_pool2d(rl, k, 1, k // 2)
            mu_g = torch.nn.functional.avg_pool2d(gl, k, 1, k // 2)
            s_r = torch.nn.functional.avg_pool2d(rl * rl, k, 1, k // 2) - mu_r ** 2
            s_g = torch.nn.functional.avg_pool2d(gl * gl, k, 1, k // 2) - mu_g ** 2
            s_rg = torch.nn.functional.avg_pool2d(rl * gl, k, 1, k // 2) - mu_r * mu_g
            ssim_map = ((2 * mu_r * mu_g + C1) * (2 * s_rg + C2)) / ((mu_r ** 2 + mu_g ** 2 + C1) * (s_r + s_g + C2))
            ssim_map = ssim_map.squeeze().cpu().numpy()

        maps = [err.mean(axis=2), err[..., 0], err[..., 1], err[..., 2], 1 - ssim_map]
        titles = ["Mean |Error|", "Error R", "Error G", "Error B", "1 - SSIM map"]
        cmaps = ['inferno', 'Reds', 'Greens', 'Blues', 'magma']

        fig, axes = plt.subplots(1, 5, figsize=(25, 5))
        for ax, m, t, cmp_ in zip(axes, maps, titles, cmaps):
            im = ax.imshow(m, cmap=cmp_)
            ax.set_title(t, fontsize=10); ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)
        plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        for i, suffix in enumerate(['mean_err', 'R_err', 'G_err', 'B_err', 'inv_ssim']):
            self._save_single_img(maps[i], os.path.join(base_dir, f"{stem}_{suffix}.png"), cmap=cmaps[i])

    # ------------------------------------------------------------------
    # V3. Grad-CAM
    # ------------------------------------------------------------------
    def vis_gradcam(self, input_t, gt_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        gt_small = None
        if gt_t is not None:
            gt_small = torch.nn.functional.interpolate(gt_t.unsqueeze(0).to(self.device), size=small.shape[2:], mode='bilinear', align_corners=False)

        self._register_hooks(capture_grads=True)
        self.model.zero_grad()

        with torch.enable_grad():
            restored, _ = self.model(small)
            if gt_small is not None:
                scalar = torch.nn.functional.l1_loss(restored, gt_small)
            else:
                scalar = restored.mean()
            scalar.backward()

        targets = ['stage/bottleneck', 'stage/decoder4', 'stage/decoder3', 'stage/decoder2', 'stage/decoder1']
        input_np = self._to_np(small.squeeze(0))
        cams, names = [], []

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for key in targets:
            if key not in self._acts or key not in self._grads: continue
            act, grad = self._acts[key], self._grads[key]
            weights = grad.mean(dim=(2, 3), keepdim=True)
            cam = torch.relu((weights * act).sum(dim=1)).squeeze(0)
            cam = torch.nn.functional.interpolate(cam[None, None], size=small.shape[2:], mode='bilinear', align_corners=False).squeeze().detach().cpu().numpy()
            cams.append(cam)

            name = key.split('/')[-1]
            names.append(name)

            self._save_single_img(self._normalize_map(cam), os.path.join(base_dir, f"{stem}_{name}_gradcam.png"), cmap='jet')
            self._save_single_img(self._overlay_heatmap(input_np, cam), os.path.join(base_dir, f"{stem}_{name}_overlay.png"))

        self._remove_hooks()
        self.model.zero_grad()

        if not cams: return
        n = len(cams) + 1
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        axes[0, 0].imshow(input_np); axes[0, 0].set_title("Input"); axes[0, 0].axis('off')
        axes[1, 0].axis('off')
        for i, (cam, name) in enumerate(zip(cams, names)):
            axes[0, i + 1].imshow(self._normalize_map(cam), cmap='jet'); axes[0, i + 1].set_title(f"Grad-CAM: {name}", fontsize=10); axes[0, i + 1].axis('off')
            axes[1, i + 1].imshow(self._overlay_heatmap(input_np, cam)); axes[1, i + 1].set_title(f"Overlay: {name}", fontsize=10); axes[1, i + 1].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # V4. Feature Maps
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_feature_maps(self, input_t, save_path, topk=8):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        stage_keys = sorted([k for k in self._acts if k.startswith('stage/')])
        self._remove_hooks()

        n_stages = len(stage_keys)
        fig, axes = plt.subplots(n_stages, topk + 1, figsize=(2.2 * (topk + 1), 2.2 * n_stages))
        if n_stages == 1: axes = axes[None, :]
        input_np = self._to_np(small.squeeze(0))

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for r, key in enumerate(stage_keys):
            feat = self._acts[key].squeeze(0)
            energy = feat.abs().mean(dim=(1, 2))
            idx = torch.argsort(energy, descending=True)[:topk]

            stage_name = key.split('/')[-1]
            axes[r, 0].imshow(input_np); axes[r, 0].set_ylabel(stage_name, fontsize=10, fontweight='bold'); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])

            for c, ci in enumerate(idx):
                fmap = feat[ci].cpu().numpy()
                axes[r, c + 1].imshow(self._normalize_map(fmap), cmap='viridis')
                axes[r, c + 1].set_title(f"ch{ci.item()}", fontsize=7); axes[r, c + 1].axis('off')

                self._save_single_img(self._normalize_map(fmap), os.path.join(base_dir, f"{stem}_{stage_name}_ch{ci.item()}.png"), cmap='viridis')

        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V5. Spectral Analysis
    # ------------------------------------------------------------------
    def vis_spectral_analysis(self, input_t, restored_t, gt_t, save_path):
        def log_spectrum(t):
            gray = t.mean(dim=0)
            f = torch.fft.fftshift(torch.fft.fft2(gray))
            return torch.log1p(torch.abs(f)).cpu().numpy()

        def radial_profile(spec):
            h, w = spec.shape
            cy, cx = h // 2, w // 2
            y, x = np.indices((h, w))
            r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
            tbin = np.bincount(r.ravel(), spec.ravel())
            nr = np.bincount(r.ravel())
            return tbin / np.maximum(nr, 1)

        tensors = {'Input': input_t, 'Restored': restored_t}
        if gt_t is not None: tensors['Ground Truth'] = gt_t

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        profiles = {}

        n = len(tensors)
        fig = plt.figure(figsize=(5 * n + 6, 5))
        gs = fig.add_gridspec(1, n + 1, width_ratios=[1] * n + [1.4])

        for i, (name, t) in enumerate(tensors.items()):
            spec = log_spectrum(t)
            profiles[name] = radial_profile(spec)
            ax = fig.add_subplot(gs[0, i])
            im = ax.imshow(spec, cmap='magma')
            ax.set_title(f"log|FFT| — {name}", fontsize=11); ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046)

            self._save_single_img(self._normalize_map(spec), os.path.join(base_dir, f"{stem}_fft_{name.replace(' ', '_')}.png"), cmap='magma')

        ax_p = fig.add_subplot(gs[0, n])
        colors = {'Input': 'tab:red', 'Restored': 'tab:blue', 'Ground Truth': 'tab:green'}
        for name, prof in profiles.items():
            ax_p.plot(prof[:min(len(prof), 256)], label=name, color=colors.get(name), lw=1.5)
        ax_p.set_yscale('log'); ax_p.legend(); ax_p.grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        fig_prof, ax_prof = plt.subplots(figsize=(6, 4))
        for name, prof in profiles.items():
            ax_prof.plot(prof[:min(len(prof), 256)], label=name, color=colors.get(name), lw=1.5)
        ax_prof.set_yscale('log'); ax_prof.legend(); ax_prof.grid(alpha=0.3)
        fig_prof.tight_layout()
        fig_prof.savefig(os.path.join(base_dir, f"{stem}_radial_profile.png"), dpi=130); plt.close(fig_prof)

    # ------------------------------------------------------------------
    # V6. Resonance Gates
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_resonance_gates(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        gate_keys = sorted([k for k in self._acts if k.startswith('gate/')])
        gates = {k.replace('gate/', ''): self._acts[k].squeeze().cpu().numpy() for k in gate_keys}
        self._remove_hooks()
        if not gates: return

        n = len(gates)
        cols = 3
        rows = int(np.ceil(n / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 3.2 * rows))
        axes = np.atleast_2d(axes)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, g) in enumerate(gates.items()):
            ax = axes[i // cols, i % cols]
            ax.bar(np.arange(len(g)), np.sort(g)[::-1], color='steelblue', width=1.0)
            ax.axhline(g.mean(), color='red', ls='--', lw=1)
            ax.set_title(name.replace('.', ' → '), fontsize=8); ax.set_ylim(0, 1)

            fig_bar, ax_bar = plt.subplots(figsize=(5,3))
            ax_bar.bar(np.arange(len(g)), np.sort(g)[::-1], color='steelblue', width=1.0)
            ax_bar.axhline(g.mean(), color='red', ls='--', lw=1)
            ax_bar.set_ylim(0, 1); ax_bar.set_title(name)
            fig_bar.tight_layout()
            fig_bar.savefig(os.path.join(base_dir, f"{stem}_gate_{name.replace('.', '_')}.png"), dpi=110); plt.close(fig_bar)

        for j in range(n, rows * cols): axes[j // cols, j % cols].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V7. Semantic Attention
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_semantic_attention(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)
        sim_keys = sorted([k for k in self._acts if k.startswith('sim/')])
        maps = {}
        for k in sim_keys:
            m = self._acts[k]
            m = torch.nn.functional.interpolate(m, size=small.shape[2:], mode='bilinear', align_corners=False)
            maps[k.replace('sim/', '')] = m.squeeze().cpu().numpy()
        self._remove_hooks()
        if not maps: return

        input_np = self._to_np(small.squeeze(0))
        n = len(maps)
        cols = 4
        rows = int(np.ceil((n + 1) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
        axes = np.atleast_2d(axes)
        axes[0, 0].imshow(input_np); axes[0, 0].set_title("Input"); axes[0, 0].axis('off')

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, m) in enumerate(maps.items(), start=1):
            ax = axes[i // cols, i % cols]
            overlay = self._overlay_heatmap(input_np, m)
            ax.imshow(overlay); ax.set_title(f"{name}", fontsize=8); ax.axis('off')

            self._save_single_img(overlay, os.path.join(base_dir, f"{stem}_sim_{name}.png"))

        for j in range(n + 1, rows * cols): axes[j // cols, j % cols].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=110); plt.close(fig)

    # ------------------------------------------------------------------
    # V8. Edges
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_edges(self, input_t, restored_t, gt_t, save_path):
        def edges(t):
            t = t.unsqueeze(0).to(self.device)
            ex = torch.nn.functional.conv2d(t, self.sobel_x, padding=1, groups=3)
            ey = torch.nn.functional.conv2d(t, self.sobel_y, padding=1, groups=3)
            return torch.sqrt(ex ** 2 + ey ** 2 + 1e-6).mean(dim=1).squeeze(0).cpu().numpy()

        e_in, e_re = edges(input_t), edges(restored_t)
        panels = [e_in, e_re]
        titles = [f"Input edges", f"Restored edges"]
        if gt_t is not None:
            e_gt = edges(gt_t)
            panels += [e_gt, np.abs(e_re - e_gt)]
            titles += [f"GT edges", f"|Restored - GT| edge error"]

        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
        for ax, p, t in zip(np.atleast_1d(axes), panels, titles):
            im = ax.imshow(p, cmap='gray' if 'error' not in t else 'inferno')
            ax.set_title(t, fontsize=10); ax.axis('off'); plt.colorbar(im, ax=ax, fraction=0.046)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        keys = ['input', 'restored', 'gt', 'error']
        for i, p in enumerate(panels):
            cmap_arg = 'inferno' if keys[i] == 'error' else 'gray'
            self._save_single_img(p, os.path.join(base_dir, f"{stem}_edge_{keys[i]}.png"), cmap=cmap_arg)

    # ------------------------------------------------------------------
    # V9. Histograms
    # ------------------------------------------------------------------
    def vis_histograms(self, input_t, restored_t, gt_t, save_path):
        tensors = {'Input': input_t, 'Restored': restored_t}
        if gt_t is not None: tensors['Ground Truth'] = gt_t
        colors3 = ['red', 'green', 'blue']

        fig, axes = plt.subplots(1, len(tensors) + 1, figsize=(5.5 * (len(tensors) + 1), 4.5))

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem

        for i, (name, t) in enumerate(tensors.items()):
            arr = self._to_np(t)

            fig_h, ax_h = plt.subplots()
            for c in range(3):
                axes[i].hist(arr[..., c].ravel(), bins=64, color=colors3[c], alpha=0.45, density=True)
                ax_h.hist(arr[..., c].ravel(), bins=64, color=colors3[c], alpha=0.45, density=True)
            axes[i].set_title(f"RGB histogram — {name}", fontsize=11); axes[i].grid(alpha=0.3)

            fig_h.tight_layout()
            fig_h.savefig(os.path.join(base_dir, f"{stem}_hist_{name.replace(' ', '_')}.png"), dpi=130)
            plt.close(fig_h)

        ls = {'Input': 'tab:red', 'Restored': 'tab:blue', 'Ground Truth': 'tab:green'}
        for name, t in tensors.items():
            lum = self._to_np(t).mean(axis=2).ravel()
            xs = np.sort(lum)
            axes[-1].plot(xs, np.linspace(0, 1, len(xs)), label=name, color=ls.get(name), lw=1.5)
        axes[-1].legend(); axes[-1].grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # V10. Prompts
    # ------------------------------------------------------------------
    def vis_prompt(self, prompt_emb, save_path, task_name):
        p = prompt_emb.numpy()
        fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
        axes[0].bar(np.arange(len(p)), p, color='skyblue', width=1.0)
        axes[0].set_title(f"Degradation Prompt ({len(p)}D)", fontsize=11); axes[0].grid(alpha=0.3, axis='y')

        top = np.argsort(np.abs(p))[::-1][:20]
        axes[1].barh(np.arange(len(top)), p[top], color=['tab:blue' if v >= 0 else 'tab:red' for v in p[top]])
        axes[1].set_yticks(np.arange(len(top))); axes[1].set_yticklabels([f"dim {d}" for d in top], fontsize=7)
        axes[1].invert_yaxis(); axes[1].grid(alpha=0.3, axis='x')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        fig_b, ax_b = plt.subplots(figsize=(8,3))
        ax_b.bar(np.arange(len(p)), p, color='skyblue', width=1.0)
        fig_b.tight_layout(); fig_b.savefig(os.path.join(base_dir, f"{stem}_prompt_bar.png"), dpi=130); plt.close(fig_b)

    # ------------------------------------------------------------------
    # V11. Zoom Crops
    # ------------------------------------------------------------------
    def vis_zoom_crops(self, input_t, restored_t, gt_t, save_path, crop=96):
        if gt_t is None: return
        err = np.abs(self._to_np(restored_t) - self._to_np(gt_t)).mean(axis=2)
        H, W = err.shape
        crop = max(2, min(crop, H, W))
        eh = torch.nn.functional.avg_pool2d(torch.from_numpy(err)[None, None].float(), crop, stride=crop // 2)
        idx = torch.argmax(eh.flatten()).item()
        gw = eh.shape[-1]
        by, bx = idx // gw, idx % gw
        top = min(by * (crop // 2), max(0, H - crop))
        left = min(bx * (crop // 2), max(0, W - crop))

        def cut(t): return self._to_np(t)[top:top + crop, left:left + crop]

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        for ax, img, title in zip(axes, [cut(input_t), cut(restored_t), cut(gt_t), err[top:top + crop, left:left + crop]], ["Input crop", "Restored crop", "GT crop", "Error crop"]):
            if img.ndim == 2: im = ax.imshow(img, cmap='inferno'); plt.colorbar(im, ax=ax, fraction=0.046)
            else: ax.imshow(img)
            ax.set_title(title, fontsize=11); ax.axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(cut(input_t), os.path.join(base_dir, f"{stem}_zoom_input.png"))
        self._save_single_img(cut(restored_t), os.path.join(base_dir, f"{stem}_zoom_restored.png"))
        self._save_single_img(cut(gt_t), os.path.join(base_dir, f"{stem}_zoom_gt.png"))
        self._save_single_img(err[top:top + crop, left:left + crop], os.path.join(base_dir, f"{stem}_zoom_error.png"), cmap='inferno')

    # ------------------------------------------------------------------
    # V12. Occlusion Maps
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_occlusion(self, input_t, gt_t, save_path, patch=64, stride=64):
        if gt_t is None: return
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        gt_small = torch.nn.functional.interpolate(gt_t.unsqueeze(0).to(self.device), size=small.shape[2:], mode='bilinear', align_corners=False)
        base, _ = self.model(small)
        base_err = torch.nn.functional.l1_loss(base, gt_small).item()

        _, _, H, W = small.shape
        heat = np.zeros(((H - 1) // stride + 1, (W - 1) // stride + 1))
        mean_val = small.mean()
        for iy, y in enumerate(range(0, H, stride)):
            for ix, x in enumerate(range(0, W, stride)):
                occluded = small.clone()
                occluded[:, :, y:y + patch, x:x + patch] = mean_val
                out, _ = self.model(occluded)
                heat[iy, ix] = torch.nn.functional.l1_loss(out, gt_small).item() - base_err

        heat_full = torch.nn.functional.interpolate(torch.from_numpy(heat)[None, None].float(), size=(H, W), mode='bilinear', align_corners=False).squeeze().numpy()
        input_np = self._to_np(small.squeeze(0))

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(input_np); axes[0].set_title("Input"); axes[0].axis('off')
        im = axes[1].imshow(heat_full, cmap='jet')
        axes[1].set_title(f"Occlusion ΔL1"); axes[1].axis('off'); plt.colorbar(im, ax=axes[1], fraction=0.046)
        axes[2].imshow(self._overlay_heatmap(input_np, heat_full)); axes[2].set_title("Overlay"); axes[2].axis('off')
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        self._save_single_img(heat_full, os.path.join(base_dir, f"{stem}_occlusion_heatmap.png"), cmap='jet')
        self._save_single_img(self._overlay_heatmap(input_np, heat_full), os.path.join(base_dir, f"{stem}_occlusion_overlay.png"))

    # ------------------------------------------------------------------
    # V13. SFT Layer Modulation Output
    # ------------------------------------------------------------------
    @torch.no_grad()
    def vis_sft_modulation(self, input_t, save_path):
        small = self._resize_for_explain(input_t).to(self.device).unsqueeze(0)
        self._register_hooks(capture_grads=False)
        _ = self.model(small)

        sft_scale_keys = sorted([k for k in self._acts if k.startswith('sft_scale/')])
        sft_shift_keys = sorted([k for k in self._acts if k.startswith('sft_shift/')])

        scales, shifts = {}, {}
        for k_sc, k_sh in zip(sft_scale_keys, sft_shift_keys):
            name = k_sc.replace('sft_scale/', '')
            scales[name] = self._acts[k_sc].mean(dim=1).squeeze().cpu().numpy()
            shifts[name] = self._acts[k_sh].mean(dim=1).squeeze().cpu().numpy()

        self._remove_hooks()
        if not scales: return

        n = len(scales)
        fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
        if n == 1: axes = axes[:, None]
        for i, name in enumerate(scales.keys()):
            sc_map, sh_map = scales[name], shifts[name]
            im1 = axes[0, i].imshow(sc_map, cmap='coolwarm'); axes[0, i].set_title(f"SFT Scale: {name}", fontsize=10); axes[0, i].axis('off')
            plt.colorbar(im1, ax=axes[0, i], fraction=0.046)
            im2 = axes[1, i].imshow(sh_map, cmap='viridis'); axes[1, i].set_title(f"SFT Shift: {name}", fontsize=10); axes[1, i].axis('off')
            plt.colorbar(im2, ax=axes[1, i], fraction=0.046)

        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

        base_dir, stem = os.path.dirname(save_path), Path(save_path).stem
        for name in scales.keys():
            self._save_single_img(scales[name], os.path.join(base_dir, f"{stem}_{name}_sft_scale.png"), cmap='coolwarm')
            self._save_single_img(shifts[name], os.path.join(base_dir, f"{stem}_{name}_sft_shift.png"), cmap='viridis')

    # ------------------------------------------------------------------
    # Global Reports
    # ------------------------------------------------------------------
    def vis_prompt_projection(self, prompts, labels, save_path, title):
        if len(prompts) < 3: return
        X = np.stack(prompts)
        method = "PCA"
        try:
            if len(prompts) >= 8:
                from sklearn.manifold import TSNE
                perp = max(2, min(30, len(prompts) - 1))
                X2 = TSNE(n_components=2, perplexity=perp, init='pca', learning_rate='auto', random_state=42).fit_transform(X)
                method = f"t-SNE (perplexity={perp})"
            else:
                raise ValueError("too few samples for t-SNE")
        except Exception:
            from sklearn.decomposition import PCA
            X2 = PCA(n_components=2, random_state=42).fit_transform(X)
            method = "PCA"

        uniq = sorted(set(labels))
        # Fixed DeprecationWarning: Using plt.get_cmap instead of cm.get_cmap
        cmap = plt.get_cmap('tab10', max(len(uniq), 1))
        fig, ax = plt.subplots(figsize=(8, 7))
        for i, u in enumerate(uniq):
            mask = np.array([l == u for l in labels])
            ax.scatter(X2[mask, 0], X2[mask, 1], s=40, alpha=0.8, color=cmap(i), label=u, edgecolors='k', linewidths=0.3)
        ax.set_title(f"{title}\n{method} projection of Degradation Prompts", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    def vis_prompt_heatmap(self, prompts, names, save_path, title):
        if not prompts: return
        X = np.stack(prompts)
        fig, ax = plt.subplots(figsize=(14, max(3, 0.28 * len(prompts))))
        im = ax.imshow(X, aspect='auto', cmap='coolwarm', vmin=-np.abs(X).max(), vmax=np.abs(X).max())
        ax.set_yticks(np.arange(len(names))); ax.set_yticklabels(names, fontsize=6); ax.set_xlabel("prompt dimension")
        ax.set_title(f"{title} — Prompt heatmap (rows=images)", fontsize=12, fontweight='bold')
        plt.colorbar(im, ax=ax, fraction=0.025)
        fig.tight_layout(); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    def vis_metrics_summary(self, metric_rows, save_path, title):
        if not metric_rows: return
        keys = ['psnr', 'ssim', 'lpips']
        fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
        for ax, k in zip(axes, keys):
            vals = [float(m[k]) for m in metric_rows if m.get(k, 'N/A') != 'N/A']
            if not vals: ax.axis('off'); continue
            ax.plot(vals, 'o-', ms=4, lw=1, color='steelblue')
            ax.axhline(np.mean(vals), color='red', ls='--', label=f"mean={np.mean(vals):.4f} ± {np.std(vals):.4f}")
            ax.set_title(k.upper(), fontsize=11); ax.set_xlabel("image index"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        fig.suptitle(f"{title} — Per-image metrics", fontsize=13, fontweight='bold')
        fig.tight_layout(rect=[0, 0, 1, 0.9]); plt.savefig(save_path, bbox_inches='tight', dpi=130); plt.close(fig)

    # ------------------------------------------------------------------
    # Core Image Explainer Routine
    # ------------------------------------------------------------------
    def explain_image(self, input_tensor, gt_tensor, stem, dataset_dirs, task_name):

        restored_tensor, prompt_emb = self.restore(input_tensor)

        metrics = {'psnr': 'N/A', 'ssim': 'N/A', 'lpips': 'N/A'}
        if gt_tensor is not None:
            with torch.no_grad():
                r, g = restored_tensor.unsqueeze(0).to(self.device), gt_tensor.unsqueeze(0).to(self.device)
                m = self.metrics_calc(r, g)
                for k in ['psnr', 'ssim', 'lpips']:
                    if k in m:
                        metrics[k] = f"{m[k][0]:.4f}"

        try:
            self.vis_comparison(input_tensor, restored_tensor, gt_tensor, metrics, os.path.join(dataset_dirs['01_comparison'], f"{stem}.png"), task_name)
            self.vis_error_maps(restored_tensor, gt_tensor, os.path.join(dataset_dirs['02_error_maps'], f"{stem}.png"))
            self.vis_gradcam(input_tensor, gt_tensor, os.path.join(dataset_dirs['03_gradcam'], f"{stem}.png"))
            self.vis_feature_maps(input_tensor, os.path.join(dataset_dirs['04_feature_maps'], f"{stem}.png"))
            self.vis_spectral_analysis(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['05_spectral_analysis'], f"{stem}.png"))
            self.vis_resonance_gates(input_tensor, os.path.join(dataset_dirs['06_resonance_gates'], f"{stem}.png"))
            self.vis_semantic_attention(input_tensor, os.path.join(dataset_dirs['07_semantic_attention'], f"{stem}.png"))
            self.vis_edges(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['08_edge_analysis'], f"{stem}.png"))
            self.vis_histograms(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['09_histograms'], f"{stem}.png"))
            self.vis_prompt(prompt_emb, os.path.join(dataset_dirs['10_prompt_analysis'], f"{stem}.png"), task_name)
            self.vis_zoom_crops(input_tensor, restored_tensor, gt_tensor, os.path.join(dataset_dirs['11_zoom_crops'], f"{stem}.png"))
            self.vis_sft_modulation(input_tensor, os.path.join(dataset_dirs['13_sft_modulation'], f"{stem}.png"))
            if self.enable_occlusion:
                self.vis_occlusion(input_tensor, gt_tensor, os.path.join(dataset_dirs['12_occlusion'], f"{stem}.png"))
        except Exception as e:
            print(f"  [warn] visualization failed for {stem}: {e}")
        finally:
            self._remove_hooks()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

        return metrics, prompt_emb.numpy()

    def run_dataset_inference(self, dataset_name, items):
        """items: list of dicts with 'input', 'gt' (tensors) and 'name'."""
        dataset_root = os.path.join(self.output_root, dataset_name)
        sub_names = ['01_comparison', '02_error_maps', '03_gradcam', '04_feature_maps',
                     '05_spectral_analysis', '06_resonance_gates', '07_semantic_attention',
                     '08_edge_analysis', '09_histograms', '10_prompt_analysis',
                     '11_zoom_crops', '13_sft_modulation']
        if self.enable_occlusion: sub_names.append('12_occlusion')
        dataset_dirs = {}
        for s in sub_names:
            d = os.path.join(dataset_root, s)
            os.makedirs(d, exist_ok=True)
            dataset_dirs[s] = d
        metric_rows, prompts, names = [], [], []
        for it in tqdm(items, desc=f"Explaining [{dataset_name}]", leave=False):
            stem = Path(it['name']).stem
            try:
                metrics, prompt = self.explain_image(it['input'], it['gt'], stem, dataset_dirs, dataset_name)
                metric_rows.append({'image': it['name'], **metrics})
                prompts.append(prompt)
                names.append(stem)
                self.global_prompts.append((dataset_name, prompt))
            except Exception as e:
                print(f"  [warn] explain failed for {stem}: {e}")

        if metric_rows:
            csv_path = os.path.join(dataset_root, 'metrics.csv')
            with open(csv_path, 'w', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=['image', 'psnr', 'ssim', 'lpips'])
                writer.writeheader(); writer.writerows(metric_rows)

        self.vis_metrics_summary(metric_rows, os.path.join(dataset_root, 'metrics_summary.png'), dataset_name)
        self.vis_prompt_projection(prompts, [dataset_name] * len(prompts), os.path.join(dataset_root, 'prompt_projection.png'), dataset_name)
        self.vis_prompt_heatmap(prompts, names, os.path.join(dataset_root, 'prompt_heatmap.png'), dataset_name)

        self.global_metrics[dataset_name] = metric_rows
        return metric_rows

    def finalize_global_reports(self):
        if len(self.global_prompts) >= 3:
            labels = [d for d, _ in self.global_prompts]
            prompts = [p for _, p in self.global_prompts]
            self.vis_prompt_projection(prompts, labels, os.path.join(self.output_root, 'all_datasets_prompt_projection.png'), "All datasets")

        if self.global_metrics:
            keys = ['psnr', 'ssim', 'lpips']
            names = list(self.global_metrics.keys())
            fig, axes = plt.subplots(1, 3, figsize=(6 * 3, 5))
            for ax, k in zip(axes, keys):
                means, stds = [], []
                for n in names:
                    vals = [float(m[k]) for m in self.global_metrics[n] if m.get(k, 'N/A') != 'N/A']
                    means.append(np.mean(vals) if vals else 0)
                    stds.append(np.std(vals) if vals else 0)
                ax.bar(names, means, yerr=stds, capsize=4, color='steelblue', alpha=0.85)
                ax.set_title(k.upper(), fontsize=12); ax.tick_params(axis='x', rotation=30); ax.grid(alpha=0.3, axis='y')
                for x, m in zip(range(len(names)), means):
                    ax.text(x, m, f"{m:.3f}", ha='center', va='bottom', fontsize=8)
            fig.suptitle("Cross-Dataset Performance Comparison", fontsize=14, fontweight='bold')
            fig.tight_layout(rect=[0, 0, 1, 0.92])
            path = os.path.join(self.output_root, 'all_datasets_metrics_comparison.png')
            plt.savefig(path, bbox_inches='tight', dpi=130); plt.close(fig)
            print(f"Global comparison saved: {path}")

In [ ]:
# ===== Analysis.py (identical to Analysis.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - analyses requested by the reviewers (TODO items of the revision)
# --------------------------------------------------------------------------
#   step0_audit            TODO-01/03/04/05/13/15 facts (metric, SSIM-loss gradient, gate
#                          initialisation, submitted checkpoint, data pairing / leakage)
#   cost_report            TODO-13  parameters, MACs, peak memory, latency (FP32/FP16)
#   significance_report    TODO-12C bootstrap 95 % CI, paired Wilcoxon (+ Holm) vs competitors
#                          TODO-14  LPIPS / DISTS and "Divergent" percentage
#   prompt_probe_report    TODO-16  logistic-regression probe on p (5-fold CV) + mixed sets
#   severity_report        TODO-09A SIM vs severity  (+ frozen-feature stability per level)
#   spectral_report        R3-2/R5-2 amplitude-vs-phase attribution, SRB phase deviation, gates
#   qualitative_figures    TODO-06  supplementary Figs. S1 (Rain100L) and S2 (Snow100K-L)
#   ablation_tables        TODO-09B/12A/12B/15 + Tables 6/7 + SCWAM design
#   main_tables            Tables 4/5 (8 benchmarks), Rain100H (TODO-02), zero-shot (TODO-17)
#   section41_report       Section 4.1 reproducibility facts collected from run.json files
# Every function writes CSV (machine-readable) and a LaTeX-ready .tex fragment
# (PSNR two decimals, SSIM three decimals) under OUTPUT_ROOT/tables or /analysis.
# ==========================================================================
import os
import json
import math
import time
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt



# ==========================================
# 0. small helpers
# ==========================================

def fmt(v, d=2):
    return '--' if v is None or (isinstance(v, float) and math.isnan(v)) else f'{v:.{d}f}'


def write_tex(rows, cols, path, digits):
    """cols: list of keys; digits: {key: decimals}.  One LaTeX row per dict."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write('% ' + ' & '.join(cols) + '\n')
        for r in rows:
            cells = []
            for c in cols:
                v = r.get(c)
                cells.append(fmt(v, digits[c]) if (c in digits and isinstance(v, (int, float))) else str(v if v is not None else '--'))
            f.write(' & '.join(cells) + ' \\\\\n')


def save_table(rows, name, cfg, cols=None, digits=None):
    if not rows:
        print(f'[table] {name}: nothing to write yet')
        return
    os.makedirs(cfg.tables_dir, exist_ok=True)
    write_rows(rows, os.path.join(cfg.tables_dir, name + '.csv'))
    cols = cols or list(rows[0].keys())
    digits = digits or {c: (3 if 'ssim' in c.lower() or 'lpips' in c.lower() or 'dists' in c.lower() else 2) for c in cols}
    write_tex(rows, cols, os.path.join(cfg.tables_dir, name + '.tex'), digits)
    print(f'[table] {name}: {len(rows)} rows -> {cfg.tables_dir}')


def per_image(cfg, run_id, bench, tag=''):
    return read_rows(os.path.join(cfg.run_dir(run_id), 'eval', bench + tag, 'per_image.csv'))


def run_eval(cfg, run_id, bench, key='psnr'):
    rec = read_json(os.path.join(cfg.run_dir(run_id), 'run.json'), {})
    return rec.get('eval', {}).get(bench, {}).get(key)


# ==========================================
# 1. statistics
# ==========================================

def bootstrap_mean_ci(x, n=10000, seed=7, alpha=0.05):
    x = np.asarray(x, np.float64)
    if len(x) == 0:
        return (float('nan'),) * 3
    rng = np.random.default_rng(seed)
    means = []
    chunk = max(1, min(n, 20_000_000 // max(1, len(x))))
    done = 0
    while done < n:
        k = min(chunk, n - done)
        idx = rng.integers(0, len(x), size=(k, len(x)))
        means.append(x[idx].mean(1))
        done += k
    means = np.concatenate(means)
    return float(x.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


def paired_test(a, b, n_boot=10000, seed=7):
    """Two-sided Wilcoxon signed-rank on paired per-image values a (ours) vs b."""
    from scipy.stats import wilcoxon
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    d = a - b
    res = dict(n=len(d), mean_a=float(a.mean()), mean_b=float(b.mean()), mean_diff=float(d.mean()),
               median_diff=float(np.median(d)), win_rate=float((d > 0).mean()))
    m, lo, hi = bootstrap_mean_ci(d, n_boot, seed)
    res.update(diff_ci_low=lo, diff_ci_high=hi)
    if len(d) >= 6 and np.any(d != 0):
        w = wilcoxon(a, b, zero_method='wilcox', alternative='two-sided')
        res.update(wilcoxon_W=float(w.statistic), p_value=float(w.pvalue))
        nz = d[d != 0]
        r = np.argsort(np.argsort(np.abs(nz))) + 1.0
        res['rank_biserial'] = float((r[nz > 0].sum() - r[nz < 0].sum()) / r.sum())
    else:
        res.update(wilcoxon_W=None, p_value=None, rank_biserial=None)
    return res


def holm(pvals):
    idx = [i for i, p in enumerate(pvals) if p is not None]
    order = sorted(idx, key=lambda i: pvals[i])
    adj, running = [None] * len(pvals), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(order) - rank) * pvals[i]))
        adj[i] = running
    return adj


def join_by_key(rows_a, rows_b):
    mb = {r['key'] if isinstance(r['key'], str) else str(r['key']): r for r in rows_b}
    out = []
    for r in rows_a:
        k = r['key'] if isinstance(r['key'], str) else str(r['key'])
        if k in mb:
            out.append((r, mb[k]))
    return out


# ==========================================
# 2. Step 0 - facts (no training)
# ==========================================

SUBMITTED_CODE_FACTS = {   # read from the submitted notebook/config (commit 1d4282b)
    'loss_weights_used_by_notebook': 'pix 5.0, fft 1.0 (HYPERPARAMS fft_weight), perc 1.0, ssim 1.0, id 1.0, edge 1.0 (constructor default, '
                                     'edge_weight not passed) -- the manuscript states fft 0.5 and edge 0.5',
    'ssim_loss': 'pyiqa.create_metric("ssim") without as_loss=True: pyiqa evaluates it under torch.set_grad_enabled(False), '
                 'so the SSIM term added a constant and no gradient (see check below)',
    'frozen_networks': 'models.vgg19(weights=None) in the SIM extractor and in the perceptual loss, models.resnet50(weights=None) '
                       'in the identity loss: no ImageNet weights were loaded by the released code; the identity-loss ResNet-50 '
                       'was left in train mode (BatchNorm batch statistics)',
    'optimisation': 'HYPERPARAMS batch_size 64, epochs 3000, Adam lr 1e-4, CosineAnnealingLR(T_max=3000) stepped once per epoch, '
                    'clip_grad_norm 1.0, no autocast/GradScaler (FP32 training)',
    'model_selection': 'best.pth = highest mean PSNR on centre 256x256 crops of GoPro test + DeRaindrop test_b + RESIDE-6K test '
                       '(test images used for checkpoint selection)',
    'inference': 'full-resolution forward, pyiqa psnr/ssim/lpips per image, but at most the first 256 sorted images of each test '
                 'folder (max_images=256); ground truth resized to the input size when they differ',
    'data_rainy': 'JRDR rain_data_train_Heavy (= RainTrainH): input=norain/ (clean) and gt=rain/X2/ (rainy) -> direction swapped; '
                  'pairs formed by zipping two independently sorted lists -> scene ids differ in all 1800 pairs',
    'data_raining': 'adeshpal/image-deraining Dataset/input (rain-001..080) = images 1-80 of the Rain100L test set',
    'data_kohler': 'training category "blurred" = kohlerdeblur/kohler/blur|sharp (all 48 pairs); Config.py TEST_DATA_CONFIG '
                   'kohler_blur = the same folders; the notebook test entry is empty',
    'data_snow_sidd': 'DATA_CONFIG snow has empty paths (no snow training data); no SIDD path in any configuration; the snow '
                      'test folder came from a private gdown archive (/content/input, /content/gt)',
    'data_hide': 'DATA_CONFIG hide = DBlur/HIDE/train, a folder that does not exist in the DBlur collection (HIDE is test-only '
                 'there, 2025 pairs); the loader silently skipped empty folders, so no HIDE image was trained on and the '
                 'submitted pool had 12 non-empty categories (hide and snow empty)',
    'srb_gate': 'resonance_gate ends with Sigmoid and is not zero-initialised (G ~ 0.5 at initialisation, see check below)',
}


def step0_audit(cfg, roots=None, device='cpu'):
    out_dir = cfg.analysis_dir('step0_facts')
    os.makedirs(out_dir, exist_ok=True)
    facts = dict(submitted_code=SUBMITTED_CODE_FACTS, metric_protocol=PROTOCOL, environment=environment_info())
    torch.manual_seed(0)
    # --- metric introspection
    try:
        import pyiqa
        s = pyiqa.create_metric('ssim', device='cpu')
        p = pyiqa.create_metric('psnr', device='cpu')
        facts['pyiqa_ssim_options'] = dict(test_y_channel=s.net.test_y_channel, color_space=s.net.color_space,
                                           downsample=s.net.downsample, crop_border=s.net.crop_border)
        facts['pyiqa_psnr_options'] = dict(test_y_channel=p.net.test_y_channel, crop_border=p.net.crop_border)
        # --- gradient of the submitted SSIM loss
        pred = torch.rand(2, 3, 64, 64, requires_grad=True)
        gt = torch.rand(2, 3, 64, 64)
        l_sub = 1.0 - s(pred, gt).mean()
        l_new = 1.0 - ssim_y(pred, gt).mean()
        l_new.backward()
        facts['ssim_loss_gradient'] = dict(submitted_requires_grad=bool(l_sub.requires_grad),
                                           revised_grad_norm=float(pred.grad.norm()),
                                           revised_vs_pyiqa_ssim_abs_diff=float((ssim_y(pred.detach(), gt) - s(pred.detach(), gt)).abs().max()))
    except Exception as e:
        facts['pyiqa_check_error'] = str(e)
    # --- gate initialisation
    srb = SpectralResonanceBlock(64)
    g = srb.trace(torch.randn(4, 64, 32, 32))['gate']
    facts['srb_gate_at_initialisation'] = dict(mean=float(g.mean()), std=float(g.std()), min=float(g.min()), max=float(g.max()))
    # --- parameter accounting of the paper architecture (Table 10)
    m = KASIRNet(extractor_pretrained=False)
    facts['parameters_M'] = component_parameter_table(m)
    # --- submitted checkpoints
    for tag, path, flags in (('submitted_full', cfg.LEGACY_CHECKPOINT, {}), ('submitted_noSIM', cfg.LEGACY_NOSIM_CHECKPOINT, None)):
        if not path or not os.path.exists(path):
            facts[f'{tag}_checkpoint'] = f'not found at {path!r}'
            continue
        ck = torch.load(path, map_location='cpu', weights_only=False)
        sd = ck.get('model_state_dict', ck)
        info = {k: ck[k] for k in ('epoch', 'best_psnr', 'current_psnr', 'iteration') if isinstance(ck, dict) and k in ck}
        info['n_tensors'] = len(sd)
        info['n_parameters_M'] = sum(v.numel() for v in sd.values() if v.dtype.is_floating_point) / 1e6
        try:
            mm = KASIRNet(extractor_pretrained=False, **({} if flags is not None else dict(use_sim=not any(k.startswith('vgg_extractor') for k in sd))))
            mm.load_state_dict(sd, strict=True)
            info['loads_strictly_into_current_architecture'] = True
            info['flags'] = mm.flags
        except Exception as e:
            info['loads_strictly_into_current_architecture'] = False
            info['load_error'] = str(e)[:400]
        if any(k.startswith('vgg_extractor') for k in sd):
            ref = SEMANTIC_EXTRACTORS['vgg19'](pretrained=True).state_dict()
            diffs = [float((sd['vgg_extractor.' + k].float() - v.float()).abs().max()) for k, v in ref.items()
                     if 'vgg_extractor.' + k in sd and v.dtype.is_floating_point and k not in ('mean', 'std')]
            info['vgg_max_abs_diff_to_imagenet'] = max(diffs) if diffs else None
            info['vgg_is_imagenet'] = bool(diffs and max(diffs) < 1e-5)
        facts[f'{tag}_checkpoint'] = info
    # --- data facts (if the sources are available)
    if roots is not None:
        facts['data'] = _data_facts(cfg, roots)
    write_json(facts, os.path.join(out_dir, 'facts.json'))
    with open(os.path.join(out_dir, 'facts.md'), 'w') as f:
        f.write('# Step 0 facts (generated)\n\n')
        for k, v in facts.items():
            f.write(f'## {k}\n```\n{json.dumps(v, indent=2, default=str)}\n```\n\n')
    print(f'[step0] facts written to {out_dir}')
    return facts


def _data_facts(cfg, roots):
    out = {}
    try:
        d = resolve_path(roots, ('jrdr', 'JRDR/rain_data_train_Heavy'))
        A, B = list_images(os.path.join(d, 'norain')), list_images(os.path.join(d, 'rain', 'X2'))

        def hf(f):
            x = np.asarray(Image.open(f).convert('L'), np.float32) / 255.
            return float(np.abs(np.diff(x, axis=1)).mean() + np.abs(np.diff(x, axis=0)).mean())
        import re
        key = lambda f: int(re.findall(r'\d+', os.path.basename(f))[0])
        out['jrdr_heavy'] = dict(hf_energy_norain=float(np.mean([hf(f) for f in A[:50]])),
                                 hf_energy_rain_X2=float(np.mean([hf(f) for f in B[:50]])),
                                 sorted_zip_pairs_with_different_scene=int(sum(key(a) != key(b) for a, b in zip(A, B))),
                                 n_pairs=len(A))
    except Exception as e:
        out['jrdr_heavy'] = f'unavailable: {e}'
    man = read_json(os.path.join(cfg.manifest_dir, 'train_manifest.json'))
    if man:
        out['leakage_audit_of_revised_pool'] = man.get('audit')
        out['revised_pool'] = {k: dict(n_train=v.get('n_train'), n_val=v.get('n_val'), removed=v.get('n_removed_overlap'))
                               for k, v in man['categories'].items()}
    if 'legacy_raining' in roots:
        try:
            aud = LeakageAuditor(os.path.join(cfg.manifest_dir, 'thumb_cache'), cfg.OVERLAP_THRESHOLD)
            tr = list_images(resolve_path(roots, ('legacy_raining', 'Dataset/target')))
            te = [g for _, g, _ in pair_files(resolve_path(roots, ('rain100l', '.')), resolve_path(roots, ('rain100l', '.')),
                                              ('regex', r'^rain-(\d+)$', r'^norain-(\d+)$'))[0]]
            dup, _ = aud.duplicates(tr, te, 'legacy_raining', 'rain100l')
            out['submitted_raining_vs_rain100l_test'] = dict(n_train=len(tr), n_duplicates=int(dup.sum()))
        except Exception as e:
            out['submitted_raining_vs_rain100l_test'] = f'unavailable: {e}'
    return out


# ==========================================
# 3. cost (TODO-13)
# ==========================================

def count_macs(model, size, device):
    from torch.utils.flop_counter import FlopCounterMode
    WindowAttention.use_fused = False   # explicit matmuls are counted by FlopCounterMode
    try:
        x = torch.rand(1, 3, *size, device=device)
        fc = FlopCounterMode(display=False, depth=None)
        with torch.no_grad(), fc:
            model(x)
        total = fc.get_total_flops() / 2 / 1e9
        per = fc.get_flop_counts()
        ext = 0.0
        for k, v in per.items():
            if k.endswith('vgg_extractor') or k.endswith('semantic_extractor'):
                ext = sum(v.values()) / 2 / 1e9
        return dict(total_GMACs=total, extractor_GMACs=ext, restoration_GMACs=total - ext,
                    note='FFT operations are not counted by torch.utils.flop_counter')
    finally:
        WindowAttention.use_fused = True


def profile_latency(model, size, device, dtype, warmup, repeats):
    x = torch.rand(1, 3, *size, device=device)
    amp = dtype == 'fp16' and device.type == 'cuda'
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
        for _ in range(warmup):
            model(x)
        if device.type == 'cuda':
            torch.cuda.synchronize()
            st, en = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
            st.record()
            for _ in range(repeats):
                model(x)
            en.record()
            torch.cuda.synchronize()
            ms = st.elapsed_time(en) / repeats
            mem = torch.cuda.max_memory_allocated() / 2 ** 30
        else:
            t0 = time.time()
            for _ in range(repeats):
                model(x)
            ms, mem = (time.time() - t0) * 1000 / repeats, None
    return dict(size=f'{size[0]}x{size[1]}', precision=dtype, latency_ms=ms, peak_memory_GB=mem)


def cost_report(cfg, run=None, device=None):
    device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    run = run or cfg.full_run()
    out_dir = cfg.analysis_dir('cost')
    model = build_model(cfg, run)
    ck = os.path.join(cfg.ckpt_dir(run['run_id']), 'final.pth')
    if os.path.exists(ck):
        load_weights(model, ck)
    model = model.to(device).eval()
    res = dict(run_id=run['run_id'], parameters_M=component_parameter_table(model), environment=environment_info())
    # loss-only networks (training time only, never loaded for inference)
    vl, il = VGGFeatureExtractor(), IdentityFeature()
    res['loss_networks_M'] = dict(vgg19_perceptual=sum(p.numel() for p in vl.parameters()) / 1e6,
                                  resnet50_identity=sum(p.numel() for p in il.parameters()) / 1e6)
    res['instantiated_at_training_M'] = res['parameters_M']['loaded at inference'] + sum(res['loss_networks_M'].values())
    # SRB vs plain-conv replacement block
    srb_p = sum(p.numel() for m in model.modules() if isinstance(m, SpectralResonanceBlock) for p in m.parameters())
    plain = KASIRNet(extractor_pretrained=False, use_srb=False)
    plain_p = sum(p.numel() for m in plain.modules() if isinstance(m, PlainConvBlock) for p in m.parameters())
    res['srb_vs_plain_conv_M'] = dict(srb_blocks=srb_p / 1e6, plain_conv_blocks=plain_p / 1e6, difference=(srb_p - plain_p) / 1e6)
    res['macs'] = {f'{h}x{w}': count_macs(model, (h, w), device) for (h, w) in [(256, 256), (128, 128)]}
    rows = []
    for size in cfg.PROFILE_SIZES:
        for dt in ('fp32', 'fp16'):
            if dt == 'fp16' and device.type != 'cuda':
                continue
            try:
                rows.append(profile_latency(model, size, device, dt, cfg.PROFILE_WARMUP if device.type == 'cuda' else 1,
                                            cfg.PROFILE_REPEATS if device.type == 'cuda' else 2))
            except torch.cuda.OutOfMemoryError:
                rows.append(dict(size=f'{size[0]}x{size[1]}', precision=dt, latency_ms=None, peak_memory_GB='OOM'))
    res['inference_profile'] = rows
    # peak memory of one training step (batch BATCH_SIZE, patch, AMP, all loss terms)
    if device.type == 'cuda':
        try:
            tm = build_model(cfg, run).to(device).train()
            crit = OmniRestorationLoss(cfg.loss_weights(run), amp=cfg.AMP).to(device)
            opt = torch.optim.Adam([p for p in tm.parameters() if p.requires_grad], lr=cfg.LR)
            torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            x = torch.rand(cfg.BATCH_SIZE, 3, cfg.PATCH_SIZE, cfg.PATCH_SIZE, device=device)
            for _ in range(3):
                opt.zero_grad()
                with torch.autocast('cuda', dtype=torch.float16, enabled=cfg.AMP):
                    y, _ = tm(x)
                l, _ = crit(y, x)
                l.backward()
                opt.step()
            res['training_step_peak_memory_GB'] = torch.cuda.max_memory_allocated() / 2 ** 30
            del tm, crit, opt
        except Exception as e:
            res['training_step_peak_memory_GB'] = f'failed: {e}'
    write_json(res, os.path.join(out_dir, 'cost.json'))
    save_table([dict(size=r['size'], precision=r['precision'], latency_ms=r['latency_ms'], peak_memory_GB=r['peak_memory_GB'])
                for r in rows], 'table10_latency_memory', cfg, digits=dict(latency_ms=1, peak_memory_GB=2))
    return res


# ==========================================
# 4. significance (TODO-12C) and perceptual metrics (TODO-14)
# ==========================================

def significance_report(cfg, evaluator, run_id):
    ci_rows, test_rows, perc_rows = [], [], []
    for bench in cfg.MAIN_BENCHMARKS + ['rain100h']:
        ours = per_image(cfg, run_id, bench)
        if not ours:
            continue
        row = dict(benchmark=bench, n=len(ours))
        for k in ('psnr', 'ssim', 'lpips', 'dists'):
            if k in ours[0]:
                m, lo, hi = bootstrap_mean_ci([r[k] for r in ours], cfg.BOOTSTRAP_RESAMPLES, cfg.BOOTSTRAP_SEED)
                row.update({k: m, f'{k}_ci_low': lo, f'{k}_ci_high': hi})
        ci_rows.append(row)
        perc_rows.append(dict(benchmark=bench, method='KASIRNet', n=len(ours), **{k: row.get(k) for k in ('psnr', 'ssim', 'lpips', 'dists')},
                              divergent_lpips_pct=None, divergent_dists_pct=None))
        for method, folder in cfg.COMPETITOR_OUTPUTS.get(bench, {}).items():
            comp = evaluator.score_folder(bench, method, folder,
                                          os.path.join(cfg.analysis_dir('competitors'), bench, method.replace(' ', '_') + '.csv'))
            pairs = join_by_key(ours, comp)
            if len(pairs) < 2:
                continue
            for k in ('psnr', 'ssim', 'lpips', 'dists'):
                t = paired_test([a[k] for a, _ in pairs], [b[k] for _, b in pairs], cfg.BOOTSTRAP_RESAMPLES, cfg.BOOTSTRAP_SEED)
                test_rows.append(dict(benchmark=bench, competitor=method, metric=k, **t))
            div_l = np.mean([(a['psnr'] > b['psnr']) and (a['lpips'] > b['lpips']) for a, b in pairs]) * 100
            div_d = np.mean([(a['psnr'] > b['psnr']) and (a['dists'] > b['dists']) for a, b in pairs]) * 100
            perc_rows.append(dict(benchmark=bench, method=method, n=len(pairs),
                                  **{k: float(np.mean([b[k] for _, b in pairs])) for k in ('psnr', 'ssim', 'lpips', 'dists')},
                                  divergent_lpips_pct=float(div_l), divergent_dists_pct=float(div_d)))
    adj = holm([r['p_value'] for r in test_rows])
    for r, a in zip(test_rows, adj):
        r['p_holm'] = a
    save_table(ci_rows, 'table9_bootstrap_ci', cfg)
    save_table(test_rows, 'table9_wilcoxon_vs_competitors', cfg,
               digits=dict(n=0, mean_a=3, mean_b=3, mean_diff=3, median_diff=3, win_rate=3, diff_ci_low=3, diff_ci_high=3,
                           wilcoxon_W=1, p_value=4, rank_biserial=3, p_holm=4))
    save_table(perc_rows, 'table11_perceptual', cfg,
               digits=dict(n=0, psnr=2, ssim=3, lpips=3, dists=3, divergent_lpips_pct=1, divergent_dists_pct=1))
    return ci_rows, test_rows, perc_rows


# ==========================================
# 5. prompt probe + mixed degradations (TODO-16)
# ==========================================

@torch.no_grad()
def collect_descriptors(cfg, roots, model, bench, n, device, field='input'):
    ds = BenchmarkDataset(cfg, roots, bench, max_images=n)
    P = []
    for i in range(len(ds)):
        x = ds[i][field].unsqueeze(0).to(device)
        P.append(model.prompt_learner(model.check_image_size(x)[0]).float().cpu().numpy()[0])
    return np.stack(P)


def prompt_probe_report(cfg, roots, evaluator, model, run_id):
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
    from sklearn.metrics import confusion_matrix
    out_dir = cfg.analysis_dir('prompt_probe')
    os.makedirs(out_dir, exist_ok=True)
    dev = evaluator.device
    fams, X, y = [], [], []
    for f in cfg.PROBE_FAMILIES:
        try:
            P = collect_descriptors(cfg, roots, model, cfg.PROBE_FAMILIES[f], cfg.PROBE_IMAGES_PER_FAMILY, dev)
        except (FileNotFoundError, KeyError) as e:
            print(f'[probe] family {f} skipped: {e}')
            continue
        X.append(P); y += [len(fams)] * len(P); fams.append(f)
    X, y = np.concatenate(X), np.array(y)
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=1.0))
    cv = StratifiedKFold(max(2, min(5, int(np.bincount(y).min()))), shuffle=True, random_state=cfg.SEED)
    acc = cross_val_score(clf, X, y, cv=cv)
    pred = cross_val_predict(clf, X, y, cv=cv)
    cm = confusion_matrix(y, pred, labels=list(range(len(fams))))
    clf.fit(X, y)
    probe = dict(families=fams, n_per_family=int(np.bincount(y).min()), cv_accuracy_mean=float(acc.mean()),
                 cv_accuracy_std=float(acc.std()), fold_accuracies=acc.tolist(), confusion_matrix=cm.tolist(),
                 per_class_recall={f: float(cm[i, i] / max(1, cm[i].sum())) for i, f in enumerate(fams)})
    # mixed degradations: probe probabilities + restoration with normal / first-degradation descriptor
    mixed_rows, projections = [], {'single': (X, y)}
    for mb in [b for b in cfg.EXTRA_BENCHMARKS if b.startswith('mix_')]:
        try:
            spec = cfg.benchmarks()[mb]
            Pm = collect_descriptors(cfg, roots, model, mb, cfg.PROBE_IMAGES_PER_FAMILY, dev)
            Pf = collect_descriptors(cfg, roots, model, mb, cfg.PROBE_IMAGES_PER_FAMILY, dev, field='first')
            pr = clf.predict_proba(Pm)
            if not all(f in fams for f in spec['families']):
                continue
            fi = [fams.index(f) for f in spec['families']]
            top2 = np.argsort(-pr, 1)[:, :2]
            norm = evaluator.evaluate(model, run_id, mb, metric_names=('psnr', 'ssim'))
            ctrl = evaluator.evaluate(model, run_id, mb, prompt_mode='first', metric_names=('psnr', 'ssim'))
            cos = float(np.mean(np.sum(Pm * Pf, 1) / (np.linalg.norm(Pm, axis=1) * np.linalg.norm(Pf, axis=1) + 1e-8)))
            mixed_rows.append(dict(benchmark=mb, families='+'.join(spec['families']), n=norm['n_images'],
                                   input_psnr=norm['input_psnr'], input_ssim=norm['input_ssim'],
                                   psnr=norm['psnr'], ssim=norm['ssim'],
                                   psnr_first_descriptor=ctrl['psnr'], ssim_first_descriptor=ctrl['ssim'],
                                   prob_first_family=float(pr[:, fi[0]].mean()), prob_second_family=float(pr[:, fi[1]].mean()),
                                   both_in_top2_pct=float(np.mean([set(fi) <= set(t) for t in top2]) * 100),
                                   cos_mixed_vs_first_descriptor=cos))
            projections[mb] = (Pm, None)
        except Exception as e:
            print(f'[probe] {mb} skipped: {e}')
    write_json(probe, os.path.join(out_dir, 'probe.json'))
    save_table([dict(families=', '.join(fams), n_per_family=probe['n_per_family'], cv_accuracy_mean=probe['cv_accuracy_mean'] * 100,
                     cv_accuracy_std=probe['cv_accuracy_std'] * 100, **{f'recall_{k}': v * 100 for k, v in probe['per_class_recall'].items()})],
               'table13_prompt_probe', cfg, digits={**{'n_per_family': 0, 'cv_accuracy_mean': 1, 'cv_accuracy_std': 1},
                                                    **{f'recall_{k}': 1 for k in fams}})
    save_table(mixed_rows, 'table13_mixed_degradations', cfg,
               digits=dict(n=0, input_psnr=2, input_ssim=3, psnr=2, ssim=3, psnr_first_descriptor=2, ssim_first_descriptor=3,
                           prob_first_family=3, prob_second_family=3, both_in_top2_pct=1, cos_mixed_vs_first_descriptor=3))
    try:
        from sklearn.decomposition import PCA
        pca = PCA(2, random_state=0).fit(X)
        fig, ax = plt.subplots(figsize=(7, 6))
        Z = pca.transform(X)
        for i, f in enumerate(fams):
            ax.scatter(*Z[y == i].T, s=12, alpha=.7, label=f)
        for mb, (Pm, _) in projections.items():
            if mb != 'single':
                ax.scatter(*pca.transform(Pm).T, s=14, marker='x', alpha=.8, label=mb)
        ax.legend(fontsize=7); ax.set_title('Visual descriptor p (PCA): single vs mixed degradations')
        fig.tight_layout(); fig.savefig(os.path.join(out_dir, 'descriptor_pca.png'), dpi=200); plt.close(fig)
    except Exception as e:
        print('[probe] figure skipped:', e)
    return probe, mixed_rows


# ==========================================
# 6. severity (TODO-09A) + frozen-feature stability
# ==========================================

SEVERITY_LEVELS = [('blur', 'GoPro tercile 1 (mild)', 'gopro', 0), ('blur', 'GoPro tercile 2', 'gopro', 1),
                   ('blur', 'GoPro tercile 3 (severe)', 'gopro', 2),
                   ('rain', 'Rain100L (light)', 'rain100l', None), ('rain', 'Rain100H (heavy)', 'rain100h', None),
                   ('snow', 'Snow100K-S', 'snow100k_s', None), ('snow', 'Snow100K-M', 'snow100k_m', None), ('snow', 'Snow100K-L', 'snow100k_l', None),
                   ('noise', 'Gaussian sigma=15', 'cbsd68_g15', None), ('noise', 'Gaussian sigma=25', 'cbsd68_g25', None),
                   ('noise', 'Gaussian sigma=50', 'cbsd68_g50', None)]


def _tercile_keys(rows):
    v = np.array([r['input_psnr'] for r in rows])
    q1, q2 = np.quantile(v, [1 / 3, 2 / 3])
    # higher input PSNR = milder blur -> tercile 0 = mildest
    t = [0 if x >= q2 else (1 if x >= q1 else 2) for x in v]
    return {str(r['key']): ti for r, ti in zip(rows, t)}


def severity_report(cfg, run_ids):
    """run_ids: models to compare (e.g. full model, full_s1, noSIM_s1); they must have been
    evaluated on the severity benchmarks (main() does it)."""
    rows = []
    ref = per_image(cfg, run_ids[0], 'gopro')
    terc = _tercile_keys(ref) if ref else {}
    for fam, label, bench, tq in SEVERITY_LEVELS:
        row = dict(family=fam, level=label)
        for rid in run_ids:
            pr = per_image(cfg, rid, bench)
            if tq is not None:
                pr = [r for r in pr if terc.get(str(r['key'])) == tq]
            if not pr:
                continue
            row['n'] = len(pr)
            row['input_psnr'] = float(np.mean([r['input_psnr'] for r in pr]))
            row[f'psnr_{rid}'] = float(np.mean([r['psnr'] for r in pr]))
            row[f'ssim_{rid}'] = float(np.mean([r['ssim'] for r in pr]))
        if len(run_ids) >= 3 and f'psnr_{run_ids[1]}' in row and f'psnr_{run_ids[2]}' in row:
            row['delta_SIM_psnr'] = row[f'psnr_{run_ids[1]}'] - row[f'psnr_{run_ids[2]}']
        rows.append(row)
    save_table(rows, 'table8_sim_severity', cfg)
    return rows


@torch.no_grad()
def feature_stability_report(cfg, roots, device, n_images=60):
    """Cosine similarity between frozen-extractor features of the degraded input and of the
    clean image (per position, averaged), per scale and per severity level: measures how far the
    'stable semantic prior' assumption holds for each extractor (Reviewer 3, comment 3).
    GoPro levels are terciles of the input PSNR within the sampled images."""
    rows = []
    benches = []
    for fam, label, bench, tq in SEVERITY_LEVELS:
        if bench not in [b for b, _ in benches]:
            benches.append((bench, fam))
    for ename, cls in SEMANTIC_EXTRACTORS.items():
        ext = cls(pretrained=True).to(device).eval()
        for bench, fam in benches:
            try:
                ds = BenchmarkDataset(cfg, roots, bench, max_images=n_images)
            except Exception as e:
                print(f'[stability] {bench} skipped: {e}')
                continue
            sims, pin = [], []
            for i in range(len(ds)):
                it = ds[i]
                x, g = it['input'].unsqueeze(0).to(device), it['gt'].unsqueeze(0).to(device)
                pin.append(float(10 * torch.log10(1 / (((x - g) ** 2).mean() + 1e-8))))
                if max(x.shape[2:]) > 1024:
                    sc = 1024 / max(x.shape[2:])
                    x = F.interpolate(x, scale_factor=sc, mode='bilinear', antialias=True)
                    g = F.interpolate(g, scale_factor=sc, mode='bilinear', antialias=True)
                sims.append([float(F.cosine_similarity(a, b, dim=1).mean()) for a, b in zip(ext(x), ext(g))])
            if not sims:
                continue
            S, P = np.array(sims), np.array(pin)
            if bench == 'gopro':
                q1, q2 = np.quantile(P, [1 / 3, 2 / 3])
                groups = [('GoPro tercile 1 (mild)', P >= q2), ('GoPro tercile 2', (P >= q1) & (P < q2)), ('GoPro tercile 3 (severe)', P < q1)]
            else:
                groups = [(next(l for f, l, b, t in SEVERITY_LEVELS if b == bench), np.ones(len(P), bool))]
            for label, mask in groups:
                s = S[mask]
                row = dict(extractor=ename, family=fam, level=label, n=int(mask.sum()), input_psnr=float(P[mask].mean()))
                for k in range(s.shape[1]):
                    row[f'cos_scale{k + 1}'] = float(s[:, k].mean())
                row['cos_mean'] = float(s.mean())
                rows.append(row)
        del ext
    save_table(rows, 'table8_feature_stability', cfg,
               digits={**{f'cos_scale{k}': 3 for k in range(1, 5)}, 'cos_mean': 3, 'n': 0, 'input_psnr': 2})
    return rows


# ==========================================
# 7. spectral analyses (Reviewer 3 c.2, Reviewer 5 c.2, TODO-05)
# ==========================================

def _wrap(a):
    return torch.atan2(torch.sin(a), torch.cos(a))


@torch.no_grad()
def amplitude_phase_attribution(x, y):
    """Oppenheim-Lim style swap test per image.  Returns PSNR of the degraded input, of the input
    with the clean amplitude restored (degraded phase kept) and with the clean phase restored
    (degraded amplitude kept), plus relative amplitude error and amplitude-weighted phase error."""
    X, Y = torch.fft.fft2(x.double()), torch.fft.fft2(y.double())
    h_amp = torch.fft.ifft2(torch.polar(Y.abs(), X.angle())).real.clamp(0, 1)
    h_pha = torch.fft.ifft2(torch.polar(X.abs(), Y.angle())).real.clamp(0, 1)
    ps = lambda a: float(10 * torch.log10(1 / (((a - y.double()) ** 2).mean() + 1e-12)))
    amp_err = float((X.abs() - Y.abs()).norm() / (Y.abs().norm() + 1e-12))
    ph_err = float((Y.abs() * _wrap(X.angle() - Y.angle()).abs()).sum() / (Y.abs().sum() + 1e-12))
    return dict(psnr_input=ps(x), psnr_fix_amplitude=ps(h_amp), psnr_fix_phase=ps(h_pha),
                rel_amplitude_error=amp_err, weighted_phase_error_rad=ph_err)


@torch.no_grad()
def spectral_report(cfg, roots, model, device, benches=None, n_images=30):
    out_rows, srb_rows = [], []
    benches = benches or (cfg.MAIN_BENCHMARKS + ['rain100h', 'snow100k_s', 'snow100k_m', 'cbsd68_g15', 'cbsd68_g25', 'cbsd68_g50'])
    srbs = {n: m for n, m in model.named_modules() if isinstance(m, SpectralResonanceBlock)}
    captured = {}
    hooks = [m.register_forward_pre_hook(lambda mod, inp, n=n: captured.__setitem__(n, inp[0].detach())) for n, m in srbs.items()]
    try:
        for bench in benches:
            try:
                ds = BenchmarkDataset(cfg, roots, bench, max_images=n_images)
            except Exception as e:
                print(f'[spectral] {bench} skipped: {e}')
                continue
            acc, layer = [], {}
            for i in range(len(ds)):
                it = ds[i]
                acc.append(amplitude_phase_attribution(it['input'], it['gt']))
                if srbs:
                    x = it['input'].unsqueeze(0).to(device)
                    if max(x.shape[2:]) > 768:
                        s = 768 / max(x.shape[2:])
                        x = F.interpolate(x, scale_factor=s, mode='bilinear', antialias=True)
                    captured.clear()
                    model(x)
                    for n, inp in captured.items():
                        t = srbs[n].trace(inp)
                        w = t['F_in'].abs()
                        d_gate = float((w * _wrap(t['F_gated'].angle() - t['F_in'].angle()).abs()).sum() / w.sum())
                        d_mix = float((w * _wrap(t['F_out'].angle() - t['F_in'].angle()).abs()).sum() / w.sum())
                        L = layer.setdefault(n, dict(gate_mean=[], gate_std=[], phase_dev_gate=[], phase_dev_output=[]))
                        L['gate_mean'].append(float(t['gate'].mean())); L['gate_std'].append(float(t['gate'].std()))
                        L['phase_dev_gate'].append(d_gate); L['phase_dev_output'].append(d_mix)
            r = dict(benchmark=bench, n=len(acc), **{k: float(np.mean([a[k] for a in acc])) for k in acc[0]})
            gi, ga, gp = r['psnr_input'], r['psnr_fix_amplitude'], r['psnr_fix_phase']
            r['amplitude_share'] = float(max(0.0, ga - gi) / (max(0.0, ga - gi) + max(0.0, gp - gi) + 1e-9))
            out_rows.append(r)
            for n, L in layer.items():
                srb_rows.append(dict(benchmark=bench, srb=n, **{k: float(np.mean(v)) for k, v in L.items()}))
    finally:
        for h in hooks:
            h.remove()
    save_table(out_rows, 'spectral_amplitude_phase_attribution', cfg,
               digits=dict(n=0, psnr_input=2, psnr_fix_amplitude=2, psnr_fix_phase=2, rel_amplitude_error=3,
                           weighted_phase_error_rad=3, amplitude_share=3))
    save_table(srb_rows, 'spectral_srb_phase_and_gates', cfg,
               digits=dict(gate_mean=3, gate_std=3, phase_dev_gate=6, phase_dev_output=3))
    return out_rows, srb_rows


def protocol_effect_report(cfg, evaluator, model, run_id, benches=None):
    """Test-set effect of the inference protocol (reported only; the protocol itself was chosen on the
    held-out validation pairs): every main benchmark with the selected protocol and with 'full'."""
    benches = benches or cfg.MAIN_BENCHMARKS
    sel, rows = evaluator.protocol, []
    if sel == 'full':
        print('[protocol] the selected protocol is the full-image pass: nothing to compare')
        return rows
    for b in benches:
        try:
            a = evaluator.evaluate(model, run_id, b, metric_names=('psnr', 'ssim'))
            f = evaluator.evaluate(model, run_id, b, metric_names=('psnr', 'ssim'), protocol='full')
        except FileNotFoundError as e:
            print(f'[protocol] {b} skipped: {e}')
            continue
        rows.append(dict(benchmark=b, n=a['n_images'], protocol=sel, psnr=a['psnr'], ssim=a['ssim'],
                         psnr_full_image=f['psnr'], ssim_full_image=f['ssim'], delta_psnr=a['psnr'] - f['psnr'],
                         delta_ssim=a['ssim'] - f['ssim']))
    save_table(rows, 'table_inference_protocol_test', cfg, digits=dict(n=0, psnr=2, psnr_full_image=2, delta_psnr=2,
                                                                       ssim=3, ssim_full_image=3, delta_ssim=3))
    return rows


@torch.no_grad()
def srb_consistency_report(cfg, roots, model, device, benches=None, n_images=10, P=256):
    """Why the inference protocol matters (R3-2 / R5-2).
    (a) Exact spatial form of every SRB.  The real 1x1 conv W = [[A, B], [C, D]] on [Re F; Im F] equals
        alpha F + beta conj(F) with alpha = ((A+D) + i(C-B))/2, beta = ((A-D) + i(C+B))/2, i.e. in image space
        y = a_r x + a_i Qx + b_r Rx + b_i QRx  (R: point reflection x(-n mod N), Q: quadrature transform =
        irFFT2(i rFFT2(.))).  The anti-linear terms (beta) couple every position with the opposite position
        of the WHOLE map: inside a 256x256 training patch, or across a whole 720x1280 test frame.
        Table: energy share of each term per SRB (from the weights only).
    (b) Train/test gap of the resonance gate: input of the gate (mean un-normalised amplitude) and gate
        value on full test images vs on their P x P tiles (the training regime)."""
    srbs = {n: m for n, m in model.named_modules() if isinstance(m, SpectralResonanceBlock)}
    rows_a = []
    for n, m in srbs.items():
        Wt = m.freq_conv.weight.detach().float()[:, :, 0, 0].cpu()
        c = Wt.shape[0] // 2
        A, B, Cm, D = Wt[:c, :c], Wt[:c, c:], Wt[c:, :c], Wt[c:, c:]
        e = dict(identity_mix=((A + D) / 2).pow(2).sum(), quadrature=((Cm - B) / 2).pow(2).sum(),
                 reflection=((A - D) / 2).pow(2).sum(), reflection_quadrature=((Cm + B) / 2).pow(2).sum())
        tot = float(sum(e.values()))
        rows_a.append(dict(srb=n, channels=c, **{f'share_{k}': float(v) / tot for k, v in e.items()},
                           anti_linear_share=float(e['reflection'] + e['reflection_quadrature']) / tot,
                           bias_norm=float(m.freq_conv.bias.detach().norm()) if m.freq_conv.bias is not None else 0.0))
    save_table(rows_a, 'spectral_srb_operator_decomposition', cfg, digits=dict(channels=0))
    benches = benches or cfg.MAIN_BENCHMARKS
    cap = {}
    hooks = [m.register_forward_pre_hook(lambda mod, inp, n=n: cap.__setitem__(n, inp[0].detach())) for n, m in srbs.items()]
    acc = {}
    try:
        for bench in benches:
            try:
                ds = BenchmarkDataset(cfg, roots, bench, max_images=n_images)
            except Exception as e:
                print(f'[srb-consistency] {bench} skipped: {e}')
                continue
            for i in range(len(ds)):
                x = ds[i]['input'].unsqueeze(0).to(device)
                H, W = x.shape[2:]
                if H <= P or W <= P:
                    continue
                x = x[:, :, :min(H, 720) // 16 * 16, :min(W, 1280) // 16 * 16]
                cap.clear(); model(x); full = dict(cap)
                tiles = torch.cat([x[:, :, a:a + P, b:b + P] for a in range(0, x.shape[2] - P + 1, P)
                                   for b in range(0, x.shape[3] - P + 1, P)])
                cap.clear(); model(tiles); tl = dict(cap)
                for n, m in srbs.items():
                    gf, gt_ = m.trace(full[n])['gate'], m.trace(tl[n])['gate']
                    sf = torch.fft.rfft2(full[n].float()).abs().mean().item()
                    st = torch.fft.rfft2(tl[n].float()).abs().mean().item()
                    L = acc.setdefault((bench, n), dict(gate_input_full=[], gate_input_tile=[], gate_full=[], gate_tile=[], gate_abs_diff=[]))
                    L['gate_input_full'].append(sf); L['gate_input_tile'].append(st)
                    L['gate_full'].append(float(gf.mean())); L['gate_tile'].append(float(gt_.mean()))
                    L['gate_abs_diff'].append(float((gf.mean(0) - gt_.mean(0)).abs().mean()))
    finally:
        for h in hooks:
            h.remove()
    rows_b = []
    for (bench, n), L in acc.items():
        r = dict(benchmark=bench, srb=n, n=len(L['gate_full']), **{k: float(np.mean(v)) for k, v in L.items()})
        r['gate_input_ratio_full_over_tile'] = r['gate_input_full'] / max(r['gate_input_tile'], 1e-12)
        rows_b.append(r)
    save_table(rows_b, 'spectral_srb_train_test_gap', cfg, digits=dict(n=0))
    return rows_a, rows_b


# ==========================================
# 8. qualitative figures (TODO-06)
# ==========================================

def _edges(t):
    sx = torch.tensor([[-1., 0, 1], [-2, 0, 2], [-1, 0, 1]]).view(1, 1, 3, 3)
    g = F.pad(t.mean(0, keepdim=True).unsqueeze(0), (1, 1, 1, 1), mode='replicate')
    ex, ey = F.conv2d(g, sx), F.conv2d(g, sx.transpose(2, 3))
    return torch.sqrt(ex ** 2 + ey ** 2)[0, 0].numpy()


def _spectrum(t):
    return torch.log1p(torch.fft.fftshift(torch.fft.fft2(t.mean(0))).abs()).numpy()


@torch.no_grad()
def qualitative_figures(cfg, roots, evaluator, model, run_id):
    out_dir = cfg.analysis_dir('figures')
    os.makedirs(out_dir, exist_ok=True)
    made = []
    for fig_id, bench in (('S1', 'rain100l'), ('S2', 'snow100k_l')):
        rows = per_image(cfg, run_id, bench)
        if not rows:
            print(f'[figures] {bench}: evaluate the run first')
            continue
        comps = {m: f for m, f in cfg.FIGURE_COMPETITORS.get(bench, {}).items() if os.path.isdir(f)}
        gains = np.array([r['psnr'] - r['input_psnr'] for r in rows])
        order = np.argsort(gains)
        picks = [rows[order[int(q * (len(rows) - 1))]] for q in (0.25, 0.5, 0.75)]   # representative, not cherry-picked
        ds = BenchmarkDataset(cfg, roots, bench)
        by_key = {str(ds.items[i][2]): i for i in range(len(ds))}
        cols = ['Input'] + list(comps)[:3] + ['KASIRNet', 'GT']
        fig, axes = plt.subplots(3, len(cols), figsize=(2.6 * len(cols), 2.3 * 3))
        sel = []
        for ri, r in enumerate(picks):
            it = ds[by_key[str(r['key'])]]
            out, _ = model(it['input'].unsqueeze(0).to(evaluator.device))
            out = out[0].float().cpu()
            panels = [('Input', it['input'])]
            for m in list(comps)[:3]:
                f = next((os.path.join(comps[m], fn) for fn in os.listdir(comps[m])
                          if os.path.splitext(fn)[0].lower() in (os.path.splitext(it['name'])[0].lower(), str(it['key']).lower())), None)
                panels.append((m, TF.to_tensor(Image.open(f).convert('RGB')) if f else None))
            panels += [('KASIRNet', out), ('GT', it['gt'])]
            for ci, (title, t) in enumerate(panels):
                ax = axes[ri, ci]
                ax.axis('off')
                if t is None:
                    continue
                ax.imshow(t.permute(1, 2, 0).clamp(0, 1).numpy())
                if title != 'GT':
                    m = evaluator.light(t.unsqueeze(0), it['gt'].unsqueeze(0))
                    ax.set_title(f"{title}\n{m['psnr'][0]:.2f} dB / {m['ssim'][0]:.3f}", fontsize=8)
                else:
                    ax.set_title('GT', fontsize=8)
            sel.append(dict(image=it['name'], psnr_gain=float(r['psnr'] - r['input_psnr'])))
        fig.tight_layout()
        for ext in ('png', 'pdf'):
            fig.savefig(os.path.join(out_dir, f'Fig{fig_id}_{bench}.{ext}'), dpi=300)
        plt.close(fig)
        # image / edge / spectrum rows for the median image
        it = ds[by_key[str(picks[1]['key'])]]
        out, _ = model(it['input'].unsqueeze(0).to(evaluator.device))
        cols2 = [('Input', it['input'])] + [(m, None) for m in list(comps)[:3]] + [('KASIRNet', out[0].float().cpu()), ('GT', it['gt'])]
        for j, (m, _) in enumerate(cols2):
            if m in comps:
                f = next((os.path.join(comps[m], fn) for fn in os.listdir(comps[m])
                          if os.path.splitext(fn)[0].lower() in (os.path.splitext(it['name'])[0].lower(), str(it['key']).lower())), None)
                cols2[j] = (m, TF.to_tensor(Image.open(f).convert('RGB')) if f else None)
        fig, axes = plt.subplots(3, len(cols2), figsize=(2.6 * len(cols2), 2.3 * 3))
        for ci, (title, t) in enumerate(cols2):
            for ri in range(3):
                axes[ri, ci].axis('off')
            if t is None:
                continue
            axes[0, ci].imshow(t.permute(1, 2, 0).clamp(0, 1).numpy()); axes[0, ci].set_title(title, fontsize=8)
            axes[1, ci].imshow(_edges(t), cmap='gray')
            axes[2, ci].imshow(_spectrum(t), cmap='magma')
        axes[1, 0].text(-0.08, 0.5, 'Sobel edges', transform=axes[1, 0].transAxes, rotation=90, va='center', ha='right', fontsize=8)
        axes[2, 0].text(-0.08, 0.5, 'log |FFT|', transform=axes[2, 0].transAxes, rotation=90, va='center', ha='right', fontsize=8)
        fig.tight_layout()
        for ext in ('png', 'pdf'):
            fig.savefig(os.path.join(out_dir, f'Fig{fig_id}_{bench}_image_edge_spectrum.{ext}'), dpi=300)
        plt.close(fig)
        write_json(dict(benchmark=bench, rule='images at the 25th/50th/75th percentile of the PSNR gain of KASIRNet over the input',
                        columns=cols, selected=sel), os.path.join(out_dir, f'Fig{fig_id}_{bench}.json'))
        made.append(fig_id)
    return made


# ==========================================
# 9. ablation tables (reduced budget)
# ==========================================

def _scores(cfg, rid, benches):
    s = {b: run_eval(cfg, rid, b) for b in benches}
    return None if any(v is None for v in s.values()) else s


def ablation_tables(cfg):
    B = cfg.ABLATION_BENCHMARKS
    ref = _scores(cfg, 'full_s1', B)
    tables = {}

    def delta_row(rid, label):
        s = _scores(cfg, rid, B)
        if s is None or ref is None:
            return None
        d = {b: s[b] - ref[b] for b in B}
        row = dict(variant=label, run_id=rid, **{f'psnr_{b}': s[b] for b in B}, **{f'd_{b}': d[b] for b in B})
        row['max_drop'] = min(d.values())
        row['avg_delta'] = float(np.mean(list(d.values())))
        row['ssim_avg'] = float(np.mean([run_eval(cfg, rid, b, 'ssim') for b in B]))
        return row

    # Table 6: components (seed 1)
    t6 = []
    for rid, label in [('full_s1', 'Full KASIRNet'), ('noSIM_s1', 'w/o SIM'), ('noSRB_s1', 'w/o SRB (plain conv)'),
                       ('noSCWAM_s1', 'w/o SCWAM+Meshgrid (local conv)'), ('noPrompt_s1', 'w/o Prompt/SFT'),
                       ('noMeshgrid_s1', 'w/o Meshgrid'), ('noGCB_s1', 'w/o GCB'), ('depth3_s1', 'depth: 3 scales'),
                       ('width32_s1', 'width: C=32')]:
        r = delta_row(rid, label)
        if r:
            t6.append(r)
    save_table(t6, 'table6_component_ablation', cfg); tables['t6'] = t6
    # Table 7: loss terms
    t7 = [r for r in (delta_row(f'no_{k}_s1', f'w/o L_{k}') for k in cfg.LOSS_WEIGHTS) if r]
    save_table(t7, 'table7_loss_ablation', cfg); tables['t7'] = t7
    # SCWAM design
    ts = [r for r in (delta_row(rid, lab) for rid, lab in [('scwam_noShift_s1', 'no cyclic shift'),
                                                            ('scwam_swinMask_s1', 'Swin masked shift'),
                                                            ('scwam_RPB_s1', 'relative position bias instead of Meshgrid')]) if r]
    save_table(ts, 'table_scwam_design', cfg); tables['scwam'] = ts
    # Table 9 A: seeds
    t9 = []
    full_avgs = None
    for name in ('full', 'noSIM', 'noSRB', 'noSCWAM', 'noPrompt'):
        per_seed = [_scores(cfg, f'{name}_s{s}', B) for s in cfg.ABLATION_SEEDS]
        per_seed = [p for p in per_seed if p]
        if not per_seed:
            continue
        row = dict(variant=name, n_seeds=len(per_seed))
        for b in B:
            v = [p[b] for p in per_seed]
            row[f'{b}_mean'] = float(np.mean(v)); row[f'{b}_std'] = float(np.std(v, ddof=1)) if len(v) > 1 else None
        avg = [float(np.mean([p[b] for b in B])) for p in per_seed]
        row['avg_mean'] = float(np.mean(avg)); row['avg_std'] = float(np.std(avg, ddof=1)) if len(avg) > 1 else None
        if name == 'full':
            full_avgs = avg
        elif full_avgs and len(full_avgs) == len(avg) and len(avg) > 1:
            from scipy.stats import ttest_rel
            d = np.array(avg) - np.array(full_avgs)
            row['delta_avg_mean'] = float(d.mean()); row['delta_avg_std'] = float(d.std(ddof=1))
            row['paired_t_p'] = float(ttest_rel(avg, full_avgs).pvalue)
        # per-image paired Wilcoxon (seed 1) on each benchmark
        if name != 'full':
            for b in B:
                pairs = join_by_key(per_image(cfg, f'{name}_s1', b), per_image(cfg, 'full_s1', b))
                if len(pairs) >= 6:
                    t = paired_test([a['psnr'] for a, _ in pairs], [c['psnr'] for _, c in pairs], cfg.BOOTSTRAP_RESAMPLES, cfg.BOOTSTRAP_SEED)
                    row[f'{b}_wilcoxon_p'] = t['p_value']
        t9.append(row)
    save_table(t9, 'table9_seed_variance', cfg); tables['t9'] = t9
    # Table 9 B: interactions  I = d(A+B) - (dA + dB)  (negative = the pair overlaps less than additive)
    t9b = []
    for rid, a, b_ in [('noSIM+noSRB_s1', 'noSIM_s1', 'noSRB_s1'), ('noSIM+noSCWAM_s1', 'noSIM_s1', 'noSCWAM_s1'),
                       ('noSRB+noSCWAM_s1', 'noSRB_s1', 'noSCWAM_s1')]:
        r, ra, rb = delta_row(rid, rid), delta_row(a, a), delta_row(b_, b_)
        if r and ra and rb:
            t9b.append(dict(pair=rid.replace('_s1', ''), d_pair=r['avg_delta'], d_first=ra['avg_delta'], d_second=rb['avg_delta'],
                            additive_prediction=ra['avg_delta'] + rb['avg_delta'],
                            interaction=r['avg_delta'] - (ra['avg_delta'] + rb['avg_delta'])))
    r, ra, rf = delta_row('noSRB+noFFT_s1', 'noSRB+noFFT'), delta_row('noSRB_s1', 'noSRB'), delta_row('no_fft_s1', 'no_fft')
    if r and ra and rf:
        t9b.append(dict(pair='noSRB+noL_fft', d_pair=r['avg_delta'], d_first=ra['avg_delta'], d_second=rf['avg_delta'],
                        additive_prediction=ra['avg_delta'] + rf['avg_delta'],
                        interaction=r['avg_delta'] - (ra['avg_delta'] + rf['avg_delta'])))
    save_table(t9b, 'table9_interactions', cfg); tables['t9b'] = t9b
    # Table 12: loss-weight sensitivity
    t12 = []
    for k, w in cfg.LOSS_WEIGHTS.items():
        row = dict(term=k, default=w)
        for f in (0.5, 2.0):
            s = _scores(cfg, f'w_{k}_x{f:g}_s1', B)
            if s and ref:
                for b in B:
                    row[f'x{f:g}_d_{b}'] = s[b] - ref[b]
                row[f'x{f:g}_d_avg'] = float(np.mean([s[b] - ref[b] for b in B]))
        if len(row) > 2:
            t12.append(row)
    save_table(t12, 'table12_loss_sensitivity', cfg); tables['t12'] = t12
    # Table 8 B: extractors
    t8 = []
    for rid, label in [('full_s1', 'VGG-19 (paper)'), ('ext_resnet50_s1', 'ResNet-50 layer1-4'), ('ext_convnextT_s1', 'ConvNeXt-T stages 1-4')]:
        s = _scores(cfg, rid, B)
        if s:
            rec = read_json(os.path.join(cfg.run_dir(rid), 'run.json'), {})
            t8.append(dict(extractor=label, **{f'psnr_{b}': s[b] for b in B}, avg=float(np.mean(list(s.values()))),
                           trainable_M=rec.get('parameters_M', {}).get('trainable total'),
                           frozen_M=rec.get('parameters_M', {}).get('frozen extractor')))
    save_table(t8, 'table8_extractors', cfg); tables['t8'] = t8
    return tables


# ==========================================
# 10. main tables, Rain100H, zero-shot, Section 4.1
# ==========================================

def main_tables(cfg, run_id):
    rows = []
    for b in cfg.MAIN_BENCHMARKS + ['rain100h']:
        e = read_json(os.path.join(cfg.run_dir(run_id), 'eval', b, 'summary.json'))
        if e:
            rows.append(dict(benchmark=b, n=e['n_images'], psnr=e['psnr'], ssim=e['ssim'], lpips=e.get('lpips'), dists=e.get('dists'),
                             psnr_y=e.get('psnr_y'), ssim_y=e.get('ssim_y'),
                             input_psnr=e['input_psnr'], input_ssim=e['input_ssim'], gt_resized=e.get('n_gt_resized')))
    if rows:
        main = [r for r in rows if r['benchmark'] in cfg.ABLATION_BENCHMARKS]
        if len(main) == len(cfg.ABLATION_BENCHMARKS):
            rows.append(dict(benchmark='mean(GoPro, Rain100L, SIDD)', n=None, psnr=float(np.mean([r['psnr'] for r in main])),
                             ssim=float(np.mean([r['ssim'] for r in main]))))
    save_table(rows, 'table4_5_main_results', cfg, digits=dict(n=0, psnr=2, ssim=3, lpips=3, dists=3, psnr_y=2, ssim_y=3,
                                                              input_psnr=2, input_ssim=3, gt_resized=0))
    zs = []
    cats = cfg.train_categories()
    for b in ('live1_q10', 'lol', 'dpdd'):
        e = read_json(os.path.join(cfg.run_dir(run_id), 'eval', b, 'summary.json'))
        if e:
            fam = cfg.benchmarks()[b]['family']
            real = sorted(n for n, c in cats.items() if c['family'] == fam and not c.get('synth'))
            synth = sorted(n for n, c in cats.items() if c.get('synth') == 'composite' and fam in ('jpeg', 'lowlight', 'defocus'))
            exposure = ('real pairs: ' + ', '.join(real)) if real else ('synthetic only: ' + ', '.join(synth)) if synth else 'zero-shot'
            zs.append(dict(benchmark=b, n=e['n_images'], training_exposure=exposure, input_psnr=e['input_psnr'],
                           input_ssim=e['input_ssim'], psnr=e['psnr'], ssim=e['ssim']))
    save_table(zs, 'table14_zero_shot', cfg, digits=dict(n=0, input_psnr=2, input_ssim=3, psnr=2, ssim=3))
    return rows, zs


def section41_report(cfg):
    runs = []
    for rid, run in cfg.all_runs().items():
        if rid == cfg.full_alias():
            continue
        rec = read_json(os.path.join(cfg.run_dir(rid), 'run.json'))
        if rec and 'budget' in rec:
            runs.append(dict(run_id=rid, group=rec['group'], seed=rec['seed'], status=rec['status'],
                             epochs=rec['budget']['epochs'], iterations=rec['budget']['iterations_done'],
                             total_iters=rec['budget']['total_iters'], wall_clock_h=rec.get('wall_clock_hours'),
                             gpu=rec.get('environment', {}).get('gpu'), selected=rec.get('selection', {}).get('selected'),
                             best_val_psnr=rec.get('selection', {}).get('best_val_psnr')))
    save_table(runs, 'section41_runs', cfg, digits=dict(seed=0, epochs=0, iterations=0, total_iters=0, wall_clock_h=2, best_val_psnr=2))
    full = read_json(os.path.join(cfg.run_dir(cfg.report_run_id()), 'run.json'), {})
    reference = read_json(os.path.join(cfg.run_dir(cfg.FULL_RUN_ID), 'run.json'), {})
    man = read_json(os.path.join(cfg.manifest_dir, 'train_manifest.json'), {})
    rep = dict(full_run=full, reference_run_of_ablations=reference,
               inference_protocol=read_json(os.path.join(cfg.analysis_dir('inference_protocol'), 'selection.json'), {}),
               data_categories=man.get('categories'), leakage_audit=man.get('audit'), metric_protocol=PROTOCOL,
               total_gpu_hours_all_runs=float(np.nansum([r['wall_clock_h'] or 0 for r in runs])))
    write_json(rep, os.path.join(cfg.tables_dir, 'section41_reproducibility.json'))
    return rep

In [ ]:
# ===== main.py (identical to main.py in the repository, intra-repository imports removed) =====
# ==========================================================================
# KASIRNet - main(): full training -> full inference -> analyses -> ablations
# -> tables.  Stages are idempotent: re-running the notebook in a new Colab
# session resumes the current training run from its last checkpoint and skips
# everything that is already finished.
# ==========================================================================
import os
import gc
import numpy as np
import torch



WEIGHT_EXT = ('.pth', '.pt', '.ckpt', '.safetensors', '.bin', '.pkl')


def sync_weight_cache(cfg, direction):
    """Pretrained weights (torch hub cache: torchvision, pyiqa LPIPS/DISTS, competitor checkpoints) <->
    Google Drive (Config.WEIGHTS_CACHE), so that later sessions do not download them again."""
    import shutil
    hub, drive = torch.hub.get_dir(), cfg.WEIGHTS_CACHE
    if not drive:
        return 0
    src, dst = (drive, hub) if direction == 'restore' else (hub, drive)
    n = 0
    for dp, _, fs in os.walk(src):
        for f in fs:
            if not f.endswith(WEIGHT_EXT):
                continue
            a = os.path.join(dp, f)
            b = os.path.join(dst, os.path.relpath(a, src))
            if os.path.exists(b) and os.path.getsize(b) == os.path.getsize(a):
                continue
            try:
                os.makedirs(os.path.dirname(b), exist_ok=True)
                shutil.copyfile(a, b + '.tmp')
                os.replace(b + '.tmp', b)
                n += 1
            except OSError as e:
                print(f'[weights] could not copy {a}: {e}')
    return n


def prefetch_weights(cfg):
    """Downloads (or restores from Drive) every pretrained network the pipeline uses, before any long
    stage: VGG-19 (SIM, perceptual loss), ResNet-50 (identity loss, TODO-09B), ConvNeXt-T (TODO-09B),
    AlexNet + LPIPS v0.1 and VGG-16 + DISTS (metrics).  A missing network stops the run here."""
    import torchvision.models as tvm
    restored = sync_weight_cache(cfg, 'restore')
    jobs = {'VGG-19 (IMAGENET1K_V1)': lambda: tvm.vgg19(weights=tvm.VGG19_Weights.IMAGENET1K_V1),
            'ResNet-50 (IMAGENET1K_V1)': lambda: tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1),
            'ConvNeXt-T (IMAGENET1K_V1)': lambda: tvm.convnext_tiny(weights=tvm.ConvNeXt_Tiny_Weights.IMAGENET1K_V1),
            'LPIPS-Alex + DISTS (pyiqa)': lambda: MetricSuite('cpu', ('lpips', 'dists'))}
    failed = []
    for name, fn in jobs.items():
        for attempt in range(3):
            try:
                fn()
                break
            except Exception as e:
                if attempt == 2:
                    failed.append(f'{name}: {e}')
    stored = sync_weight_cache(cfg, 'store')
    print(f'[weights] pretrained networks ready ({restored} restored from Drive, {stored} newly cached in {cfg.WEIGHTS_CACHE})')
    if failed:
        raise RuntimeError('pretrained weights could not be downloaded:\n  ' + '\n  '.join(failed))


def setup_environment(cfg):
    try:
        from google.colab import drive  # noqa
        if not os.path.ismount(cfg.DRIVE_MOUNT):
            drive.mount(cfg.DRIVE_MOUNT)
    except ImportError:
        pass
    os.makedirs(cfg.OUTPUT_ROOT, exist_ok=True)
    os.makedirs(cfg.DATA_ROOT, exist_ok=True)
    write_json(cfg.to_dict(), os.path.join(cfg.OUTPUT_ROOT, 'config_snapshot.json'))


class Experiment:
    def __init__(self, cfg):
        self.cfg = cfg
        self.device = torch.device(cfg.DEVICE if torch.cuda.is_available() else 'cpu')
        self._roots, self._origin, self._manifest, self._evaluator, self._protocol = {}, {}, None, None, None
        self.cache = DatasetCache(cfg)
        self.stop = False

    # ------------------------------------------------------------ helpers
    def roots(self, keys, need_full=False):
        """Local roots of the data sources: Drive cache first, original download otherwise.
        need_full: a source restored from a 'used' cache is replaced by the complete source."""
        for k in sorted(set(keys)):
            partial = k in self._roots and self._origin.get(k) == 'used'
            if k in self._roots and not (need_full and partial):
                continue
            try:
                self._roots.update(resolve_sources(self.cfg, [k], self.cache, need_full))
                m = self.cache.info(k)
                from_cache = self._roots[k].startswith(self.cache.local)
                self._origin[k] = (m['mode'] if (m and from_cache) else 'origin')
            except Exception as e:
                print(f'[data] source {k} unavailable: {e}')
        return self._roots

    def bench_roots(self, benches):
        b = self.cfg.benchmarks()
        return self.roots([k for n in benches for k in spec_sources(b[n], b)])

    def manifest(self):
        """Frozen training manifest (built once from the complete sources, then reused)."""
        if self._manifest is None:
            cats = self.cfg.train_categories()
            b = self.cfg.benchmarks()
            keys = [k for s in cats.values() for k in spec_sources(s)]
            keys += [k for s in cats.values() for n in s.get('exclude_overlap_with', []) for k in spec_sources(b[n], b)]
            keys += [k for names in overlap_benchmarks(self.cfg).values() for n in names for k in spec_sources(b[n], b)]
            current = manifest_is_current(self.cfg)
            self._manifest = build_training_manifest(self.cfg, self.roots(keys, need_full=not current))
        return self._manifest

    def all_source_keys(self):
        b = self.cfg.benchmarks()
        keys = [k for s in self.cfg.train_categories().values() for k in spec_sources(s)]
        keys += [k for n in b for k in spec_sources(b[n], b)]
        return sorted(set(keys + ['jrdr', 'rain100l', 'legacy_raining']))

    def cache_benchmark_sources(self):
        """Sources read only by benchmarks are cached right after their download (their files are known
        without the training manifest), so a later failure in the audit cannot waste the download."""
        train_keys = {k for s in self.cfg.train_categories().values() for k in spec_sources(s)}
        files = {}
        for name in self.cfg.benchmarks():
            try:
                for k, r in benchmark_files(self.cfg, self._roots, name):
                    files.setdefault(k, set()).add(r)
            except (FileNotFoundError, KeyError):
                pass
        for k, root in sorted(self._roots.items()):
            if self._origin.get(k) == 'origin' and k not in train_keys and k in files:
                self.cache.store(k, root, files[k])

    def cache_sources(self):
        """Pack every source downloaded in this session into the Drive cache."""
        if self.cfg.DATA_CACHE_MODE == 'off':
            return
        used = manifest_used_files(self.cfg, self._roots, self.manifest())
        try:   # the Step-0 audit reads the complete RainTrainH folders
            d = resolve_path(self._roots, ('jrdr', 'JRDR/rain_data_train_Heavy'))
            for dp, _, fs in os.walk(d):
                used.setdefault('jrdr', set()).update(os.path.relpath(os.path.join(dp, f), self._roots['jrdr']) for f in fs)
        except (FileNotFoundError, KeyError):
            pass
        for k, root in sorted(self._roots.items()):
            if self._origin.get(k) == 'origin':
                self.cache.store(k, root, None if k == 'legacy_raining' else used.get(k))

    def evaluator(self):
        if self._evaluator is None:
            self._evaluator = Evaluator(self.cfg, self._roots, self.device)
        self._evaluator.roots = self._roots
        self._evaluator.protocol = self.protocol()
        return self._evaluator

    def protocol(self):
        """Inference protocol of every evaluation.  'auto': chosen once on the held-out validation pairs
        with the reference full model (KASIRNet_full, trained first) and then frozen for all runs."""
        cfg = self.cfg
        if cfg.INFERENCE_PROTOCOL != 'auto':
            return cfg.INFERENCE_PROTOCOL
        if self._protocol:
            return self._protocol
        rid = cfg.FULL_RUN_ID
        rec = read_json(os.path.join(cfg.run_dir(rid), 'run.json'), {})
        out_dir = cfg.analysis_dir('inference_protocol')
        key = [cfg.INFERENCE_CANDIDATES, list(cfg.INFERENCE_SELECT_MAX_SIZE), cfg.INFERENCE_SELECT_MIN_GAIN, rid,
               rec.get('selection', {}).get('selected')]
        sel = read_json(os.path.join(out_dir, 'selection.json'), {})
        if sel.get('key') == key:
            self._protocol = sel['selected']
            return self._protocol
        if rec.get('status') != 'complete':
            return 'full'          # nothing to select with yet (no evaluation happens before KASIRNet_full exists)
        model = Evaluator(cfg, self._roots, self.device).load_run_model(cfg.all_runs()[rid])
        sel = select_inference_protocol(cfg, model, self.manifest()['val'], self.device, out_dir)
        sel['key'] = key
        write_json(sel, os.path.join(out_dir, 'selection.json'))
        del model
        self.free()
        self._protocol = sel['selected']
        return self._protocol

    def complete(self, run_id):
        return read_json(os.path.join(self.cfg.run_dir(run_id), 'run.json'), {}).get('status') == 'complete'

    def model(self, run_id):
        return self.evaluator().load_run_model(self.cfg.all_runs()[run_id])

    @staticmethod
    def free():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def evaluate_many(self, run_id, benches, metric_names=None, save=False):
        ev = self.evaluator()
        model = None
        for b in benches:
            if ev.is_done(run_id, b):
                continue
            self.bench_roots([b])
            try:
                model = model or self.model(run_id)
                ev.evaluate(model, run_id, b, metric_names=metric_names,
                            save_images=save and self.cfg.SAVE_RESTORED_IMAGES and b in self.cfg.MAIN_BENCHMARKS and not b.startswith('snow'))
            except FileNotFoundError as e:
                print(f'[eval] {run_id} / {b} skipped: {e}')
        del model
        self.free()

    # ------------------------------------------------------------- stages
    def stage_audit(self):
        """Step 0: pretrained weights, all sources (Drive cache or download), frozen training manifest
        with the leakage audit, validation of every training category and benchmark, Step-0 facts, Drive cache."""
        cfg = self.cfg
        prefetch_weights(cfg)
        keys = self.all_source_keys()
        self.cache.inventory(keys)
        self.roots(keys, need_full=not manifest_is_current(cfg))
        self.cache_benchmark_sources()
        missing = [k for k in keys if k not in self._roots]
        if missing:
            raise RuntimeError(f'data sources not available in this session: {missing}. The reason is printed above '
                               f'(e.g. Google-Drive "quota exceeded": run again later, or place the archive in '
                               f'{cfg.DATA_CACHE_ROOT}/archives/). Everything downloaded so far is kept.')
        man = self.manifest()
        vsig = validation_signature(cfg)
        if read_json(os.path.join(cfg.manifest_dir, 'data_validation.sig.json'), {}).get('signature') != vsig:
            validate_data(cfg, self._roots, man)     # again whenever a benchmark or category definition changed
            write_json(dict(signature=vsig), os.path.join(cfg.manifest_dir, 'data_validation.sig.json'))
        if not os.path.exists(os.path.join(cfg.analysis_dir('step0_facts'), 'facts.json')):
            step0_audit(cfg, self._roots, self.device)
        self.cache_sources()

    def train(self, run):
        """Train one run unless another live session holds it.  Returns its status."""
        rid = run['run_id']
        if self.complete(rid):
            return 'complete'
        if not try_lock(self.cfg, rid):
            print(f'[{rid}] being trained by another session -> skipped here')
            return 'locked'
        try:
            rec = Trainer(self.cfg, run, self.manifest(), self.device).fit()
        finally:
            release_lock(self.cfg, rid)
            self.free()
        return rec.get('status')

    def stage_train_full(self):
        st = self.train(self.cfg.full_run())
        if st == 'paused':
            self.stop = True

    def stage_train_main(self):
        """Final model of Tables 4-5: warm restart of KASIRNet_full (Config.MAIN_*)."""
        cfg = self.cfg
        if cfg.MAIN_EPOCHS <= 0:
            return
        if cfg.MAIN_INIT_FROM and not self.complete(cfg.MAIN_INIT_FROM):
            print(f'[train_main] waits for {cfg.MAIN_INIT_FROM}'); return
        st = self.train(cfg.main_run())
        if st == 'paused':
            self.stop = True

    def stage_eval_full(self):
        rid = self.cfg.report_run_id()
        if not self.complete(rid):
            print(f'[eval_full] {rid} not trained yet'); return
        benches = self.cfg.MAIN_BENCHMARKS + [b for b in self.cfg.EXTRA_BENCHMARKS if not b.startswith('mix_')]
        self.evaluate_many(rid, benches, save=True)

    def stage_analyses_full(self):
        cfg, rid = self.cfg, self.cfg.report_run_id()
        if not self.complete(rid):
            print(f'[analyses_full] {rid} not trained yet'); return
        self.bench_roots(cfg.MAIN_BENCHMARKS + cfg.EXTRA_BENCHMARKS)
        ev = self.evaluator()
        steps = dict(
            cost=lambda: cost_report(cfg, cfg.all_runs()[rid], self.device),
            significance=lambda: significance_report(cfg, ev, rid),
            probe=lambda: prompt_probe_report(cfg, self._roots, ev, self.model(rid), rid),
            spectral=lambda: spectral_report(cfg, self._roots, self.model(rid), self.device),
            srb_consistency=lambda: srb_consistency_report(cfg, self._roots, self.model(rid), self.device),
            protocol_effect=lambda: protocol_effect_report(cfg, ev, self.model(rid), rid),
            figures=lambda: qualitative_figures(cfg, self._roots, ev, self.model(rid), rid),
            stability=lambda: feature_stability_report(cfg, self._roots, self.device),
            explain=lambda: self.explainability(rid),
            tables=lambda: main_tables(cfg, rid))
        done_path = os.path.join(cfg.analysis_dir('status'), f'{rid}.json')
        done = read_json(done_path, {})
        for name, fn in steps.items():
            if done.get(name):
                continue
            try:
                fn()
                done[name] = True
                write_json(done, done_path)
            except Exception as e:
                import traceback
                traceback.print_exc()
                print(f'[analyses_full] {name} failed: {e}')
            self.free()

    def explainability(self, rid):
        model = self.model(rid)
        inf = KASIRNetInferencer(model, self.cfg.analysis_dir('explainability'), self.evaluator().metrics,
                                 explain_max_size=512, enable_occlusion=self.cfg.EXPLAIN_OCCLUSION)
        for b in self.cfg.MAIN_BENCHMARKS:
            try:
                ds = BenchmarkDataset(self.cfg, self._roots, b, max_images=self.cfg.EXPLAIN_IMAGES_PER_BENCHMARK)
                inf.run_dataset_inference(b, [ds[i] for i in range(len(ds))])
            except Exception as e:
                print(f'[explain] {b} skipped: {e}')
        inf.finalize_global_reports()

    def stage_ablations(self):
        cfg = self.cfg
        plan = cfg.ablation_plan()
        full = read_json(os.path.join(cfg.run_dir(cfg.FULL_RUN_ID), 'run.json'), {})
        todo = [r for r in plan if not self.complete(r['run_id'])]
        if full.get('budget'):
            hist = []
            ck = os.path.join(cfg.run_dir(cfg.FULL_RUN_ID), 'history_train.csv')
            if os.path.exists(ck):
                import csv
                hist = [float(r['sec_per_iter']) for r in csv.DictReader(open(ck))]
            if hist:
                it = full['budget']['iters_per_epoch'] * cfg.ABLATION_EPOCHS
                print(f'[ablations] {len(todo)}/{len(plan)} runs left x {it} iterations x {np.median(hist):.3f} s/it '
                      f'= about {len(todo) * it * np.median(hist) / 3600:.0f} GPU-hours')
        self.bench_roots(cfg.ABLATION_BENCHMARKS)
        if cfg.full_alias() and self.complete(cfg.FULL_RUN_ID):   # the reference "full, seed 1" of every ablation table
            self.evaluate_many(cfg.FULL_RUN_ID, cfg.ABLATION_BENCHMARKS, metric_names=('psnr', 'ssim'))
        for run in plan:
            st = self.train(run)
            if st == 'paused':
                self.stop = True
                return
            if st == 'complete':
                self.evaluate_many(run['run_id'], cfg.ABLATION_BENCHMARKS, metric_names=('psnr', 'ssim'))

    def stage_analyses_ablation(self):
        cfg = self.cfg
        sev_benches = sorted({b for _, _, b, _ in SEVERITY_LEVELS})
        self.bench_roots(sev_benches)
        run_ids = [cfg.FULL_RUN_ID] + [r for r in cfg.SEVERITY_RUNS]
        for rid in run_ids:
            if self.complete(rid):
                self.evaluate_many(rid, sev_benches, metric_names=('psnr', 'ssim'))
        severity_report(cfg, [r for r in run_ids if self.complete(r)])
        ablation_tables(cfg)

    def stage_tables(self):
        main_tables(self.cfg, self.cfg.report_run_id())
        ablation_tables(self.cfg)
        section41_report(self.cfg)
        print(f'[tables] all tables in {self.cfg.tables_dir}')


def main(cfg=None):
    cfg = cfg or Config()
    setup_environment(cfg)
    exp = Experiment(cfg)
    for stage in cfg.STAGES:
        if exp.stop:
            print('[main] stopped: a training run paused at the session time limit -> run main() again to resume')
            break
        print(f'\n========== stage: {stage} ==========')
        getattr(exp, 'stage_' + stage)()
    return exp



In [ ]:
# ---- run everything (re-run this cell in a new session to resume)
cfg = Config()
# Optional overrides, e.g. to split the work across sessions:
# cfg.STAGES = ['audit', 'train_full', 'eval_full', 'analyses_full']
# cfg.RUN_ABLATION_GROUPS = ['TODO12A_seeds', 'TODO09B_extractor', 'TODO12B_interactions', 'TODO15_loss_sensitivity']
exp = main(cfg)
